# Подключение Спарк

In [48]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
pd.set_option('Display.max_columns', None)

In [2]:
import pandas as pd
import numpy as np
import re
from matplotlib import pyplot as plt
#from tqdm.notebook import tqdm
pd.set_option('Display.max_columns', None)

import sys
sys.path.append('../../ss_lal_military/src')
sys.path.append('../src/')
sys.path.append('../migrant/notebooks/utilities/')

In [3]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
def get_spark_session(name, level):
    """
    Get spark context
    :: name - set your app name
    :: level - set max resources level
    """
    python_path = sys.executable
    kernel = python_path.split('/')[-3]
    os.environ['SPARK_MAJOR_VERSION'] = '3'
    os.environ['SPARK_HOME'] = '/usr/sdp/current/spark3-client/'
    os.environ['PYSPARK_DRIVER_PYTHON'] = python_path
    os.environ['PYSPARK_PYTHON'] = python_path
    os.environ['LD_LIBRARY_PATH'] = '/opt/python/virtualenv/jupyter/lib'
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/')
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/lib/py4j_current')
 
    # Resources Level Profiles                           #  cpu --  ram -- desc
    if level == 1: lv = ['basic',2,10,2,10,2,2,10]       #   21 --  142 -- для базовых запросов (show create table tbl, show partitions tbl)
    if level == 2: lv = ['basic+CPU',2,10,2,10,2,2,20]   #   41 --  262 -- для простой аналитики (select * from limit 100, sum/count/avg)
    if level == 3: lv = ['middle',4,28,6,28,6,4,20]      #   81 --  742 -- для агрегатов за период 1-2мес (client_aggr_mnth, epk_campaign_daily)
    if level == 4: lv = ['middle+CPU',4,28,6,28,6,4,25]  #  101 --  912 -- для агрегатов за период >1-6мес  (client_aggr_mnth, epk_campaign_daily)
    if level == 5: lv = ['high',4,28,6,36,8,6,30]        #  121 -- 1100 -- для детальных таблиц с большими партициями (_sbol, _card, _eps)
    if level == 6: lv = ['high+CPU',4,18,5,36,8,6,40]    #  161 -- 1000 -- для детальных таблиц с мелкими партициями (feedbacks)
    if level == 7: lv = ['unfriendly',5,28,6,44,10,8,40] #  201 -- 1458 -- для запуска вечером/ночью или на пустом кластере (не рекомендуется)
    lvname = f'{level}.{lv[0]}({lv[1]*lv[7]+1},{lv[4]+lv[2]*lv[7]})'
    print(f'Kernel: {kernel}, Python_path: {python_path}, Resource_level: {lvname}')
    
    # Spark Config      
    from pyspark import SparkContext, SparkConf
    from pyspark.sql import SparkSession
  
    conf = SparkConf().setAppName(f'{name} \n ::{kernel}::{lvname}::')\
        .setMaster("yarn")\
        .set('spark.executor.cores',                     f'{lv[1]}')\
        .set('spark.executor.memory',                    f'{lv[2]}g')\
        .set('spark.executor.memoryOverhead',            f'{lv[3]}g')\
        .set('spark.driver.memory',                      f'{lv[4]}g')\
        .set('spark.driver.memoryOverhead',              f'{lv[5]}g')\
        .set('spark.driver.maxResultSize', '10g')\
        .set('spark.dynamicAllocation.initialExecutors', f'{lv[6]}')\
        .set('spark.dynamicAllocation.maxExecutors',     f'{lv[7]}')\
        .set('spark.dynamicAllocation.enabled', 'true')\
        .set('spark.dynamicAllocation.executorIdleTimeout', '120s')\
        .set('spark.dynamicAllocation.cachedExecutorIdleTimeout', '600s')\
        .set('spark.hive.mapred.supports.subdirectories', 'true')\
        .set('spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive', 'true')\
        .set('spark.shuffle.service.enabled', 'true')\
        .set('spark.port.maxRetries', '150')\
       .set('spark.sql.parquet.writeLegacyFormat', 'true')\
        .set('spark.kerberos.access.hadoopFileSystems','hdfs://arnsdpsbx:8020/')\
        .set('spark.sql.autoBroadcastJoinThreshold','20971520')
    
    spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
    return spark

try: spark
except NameError: print('Spark3 Starting')
else:
    print('Spark3 Restarting')
    spark.stop()
    
spark = get_spark_session('platon_agent_full_script_new_node', 4) # For example, MyPySpark
  
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pyspark.sql.types as T
  
sc = spark.sparkContext
sc.setLogLevel('OFF')  # or 'INFO' or 'WARN' or 'OFF'
spark


# Ставим даты для бесплатного и платного уровня

In [5]:
report_dt_free = '2026-04-30'
report_dt_paid = '2026-04-30'
date_now_free = report_dt_free
date_now_paid = report_dt_paid

# Витрины для скрипта

In [6]:
aum = 'prx_bpm_stocks_custom_cib_pkaptdul.dfa_aum_clients'
ostatki_na_schetax_yul = 'prx_ostatki_na_schetax_yul_custom_cib_p4d_passive_ul.ul_balance'
aggr = 'prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth'
feedback_table = 'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_card_transactions'
pos_p2p = 'prx_bpm_pos_operations_mcc_custom_rb_card.txn_mnth_union'
romashka_hist = 'prx_bpm_model_romashka_custom_rozn_coreml_ext_scores.dm_romashka_trnsf_model_scores_vsi1_hist'
pos = 'prx_bpm_pos_operations_mcc_custom_rb_card.ft_txn_det_union'
stick = 'prx_bpm_pos_operations_mcc_custom_rb_card.scd_card_union'
potential_benefit = 'prx_bpm_potentinal_benefit_custom_rozn_profits.ft_potential_client_benefit_monthly'
sber_prime = 'prx_bpm_sber_prime_custom_rozn_sberprime.ft_sberprime'
deposit = 'prx_bpm_balances_fl_custom_rozn_cod3d_balances.cod_balances'
coins = 'prx_bpm_coins_custom_rozn_cod3d_common_coins.coins_oper'
aquaring = 'prx_bpm_transactions_other_bank_custom_b2c_acquiring.t_fct_transaction_acquiring'
tags = 'prx_bpm_tags_custom_rozn_tag.ft_master_hshtg'

# Определение бесплатных клиентов

In [7]:
spark.sql(f'''
with nsi_ids as (
select distinct epk_id as epk, nsi_id 
from {tags}
where row_actual_to_dt = '9999-12-31'
AND is_deleted = 0
AND nsi_id IN ('41384', '41292', '41291', '39541', '39542', '39543',
'41616', '41617', '41618', '41619', '41620')
)

select epk, 
max(CASE
    WHEN nsi_id = '41384' THEN 6
    WHEN nsi_id = '41292' or nsi_id = '41620' then 5
    WHEN nsi_id = '41291' or nsi_id = '41619' then 4
    WHEN nsi_id = '39543' or nsi_id = '41618' then 3
    WHEN nsi_id = '39542' or nsi_id = '41617' then 2
    WHEN nsi_id = '39541' or nsi_id = '41616' then 1
ELSE 0 END) AS approval_level,
'FREE' as payment_status,
max(CASE
    WHEN nsi_id = '41384' THEN nsi_id
    WHEN nsi_id = '41292' or nsi_id = '41620' then nsi_id
    WHEN nsi_id = '41291' or nsi_id = '41619' then nsi_id
    WHEN nsi_id = '39543' or nsi_id = '41618' then nsi_id
    WHEN nsi_id = '39542' or nsi_id = '41617' then nsi_id
    WHEN nsi_id = '39541' or nsi_id = '41616' then nsi_id
ELSE '' END) AS approval_criteria,
'' as primary_approval_criterion
FROM nsi_ids
group by epk
''').write.saveAsTable('arnsdpsbx_team_ss.bpm_agent_py_level', mode='overwrite')

# Определение платных клиентов 

In [12]:
spark.sql(f'''with recommendation as (
        select
        distinct epk_id as epk,
        dep_acct_tot_bal_rub_amt,
        inv_tot_bal_rub_amt,
        insur_total_bal,
        crd_dc_pos_clear_amt,
        crd_cc_pos_clear_amt,
        0 as stocks
        from {aggr}
        where report_dt = '{date_now_paid}'
        and sd_age_yrs_comp_nv >= 18 and sd_age_yrs_comp_nv <= 69 and sd_dead_nflag = 0
        and cla_all_active_1m_nflag = 1 
),

aum as (
    select
    distinct cl_epk_id as epk,
    coalesce(sum(aum_total), 0) as aum_total
from {aum}
where part_dt between cast(date_format(to_date('{date_now_paid}'), 'yyyyMM01') as int) and cast(date_format(to_date('{date_now_paid}'), 'yyyyMMdd') as int)
group by cl_epk_id
),

rko as (
    select
    distinct ucprbid as epk,
    coalesce(sum(balance), 0) as z
from {ostatki_na_schetax_yul} 
where reportdate between trunc(to_date('{date_now_paid}'), 'MM') and to_date('{date_now_paid}')
group by ucprbid
),

aquaring as (
    select
    distinct epk_id as epk,
    coalesce(sum(transaction_ccy_amt), 0) as transaction_ccy_amt
from {aquaring}
where status_order_code = 'PAID'
and part_1_day between trunc(to_date('{date_now_paid}'), 'MM') and to_date('{date_now_paid}')
group by epk_id
),

scores as (
    select
    distinct epk_id as epk,
    score
    from arnsdpsbx_team_ss.bpm_agent_paid_v2_scores
),

avg_score_table as (
    select avg(score) as avg_score
from arnsdpsbx_team_ss.bpm_agent_paid_v2_scores
),

aggregated as (
    select
    r.epk,
    sum(coalesce(r.dep_acct_tot_bal_rub_amt, 0)) as dep,
    sum(coalesce(r.inv_tot_bal_rub_amt, 0)) as inv,
    sum(coalesce(r.insur_total_bal, 0)) as insur,
    sum(coalesce(r.crd_dc_pos_clear_amt, 0)) as pos_dc,
    sum(coalesce(r.crd_cc_pos_clear_amt, 0)) as pos_cc,
    max(coalesce(a.aum_total, 0)) as aum,
    max(coalesce(rko.z, 0)) as z,
    sum(coalesce(r.stocks, 0)) as stocks,
    max(coalesce(aq.transaction_ccy_amt, 0)) as aquaring_pos
from recommendation r
left join aum a on r.epk = a.epk
left join aquaring aq on r.epk = aq.epk
left join rko on r.epk = rko.epk
group by r.epk
),

calculated as (
    select
    epk,
    dep, inv, insur, pos_dc, pos_cc, aum, stocks, z, aquaring_pos,
    (dep + inv + insur + aum) as x,
(pos_dc + pos_cc + aquaring_pos) as y
from aggregated
),

final_recommendations as (
select
c.epk,
case
    when s.score >= (select avg_score from avg_score_table)
        and (c.x >= 2000000 or c.y >= 150000 or c.z >= 2000000) 
        and (c.x <= 3000000 or c.y <= 200000 or c.z <= 3000000) then 3
    when s.score >= (select avg_score from avg_score_table)
        and (c.x >= 1000000 or c.y >= 100000 or c.z >= 1000000) 
        and (c.x <= 2000000 or c.y <= 150000 or c.z <= 2000000) then 2
    when s.score >= (select avg_score from avg_score_table)
        and (c.x >= 500000 or c.y >= 40000 or c.z >= 500000) 
        and (c.x <= 1000000 or c.y <= 100000 or c.z <= 1000000) then 1
else 0
end as recommendation_level
from calculated c
left join scores s on c.epk = s.epk
),

result as (
    select
    distinct epk,
    recommendation_level as approval_level,
    case when recommendation_level >= 1 then 'PAID' else '' end as payment_status,
    '' as approval_criteria,
    '' as primary_approval_criterion
    from final_recommendations
    where recommendation_level >= 1
),

paid as (
select
    distinct r.*
from result r
where not exists (
    select 1
    from arnsdpsbx_team_ss.bpm_agent_py_level b
    where b.epk = r.epk)
),

free as (
select * from arnsdpsbx_team_ss.bpm_agent_py_level
)

select * from paid
union all
select * from free
''').write.mode('overwrite').saveAsTable('arnsdpsbx_team_ss.bpm_agent_py_levels_all')

In [13]:
spark.sql('''
select payment_status, count(epk), count(distinct epk) from arnsdpsbx_team_ss.bpm_agent_py_levels_all
group by payment_status
''').show()

+--------------+----------+-------------------+
|payment_status|count(epk)|count(DISTINCT epk)|
+--------------+----------+-------------------+
|          FREE|   5580874|            5580874|
|          PAID|   8517510|            8517510|
+--------------+----------+-------------------+



# Потенциальная выгода клиента

In [14]:
spark.sql(f'''
with poten_benefit as (select epk, 
round(sum(coalesce(p2p_transfers_without_premier_level_1,0) + coalesce(sound_without_premier_level_1,0) 
+ coalesce(okko_without_premier_level_1,0) + coalesce(sbercard_service_without_premier_1,0)  
+ coalesce(deposit_without_premier_level_1_without_bestpremier,0) + coalesce(deposit_without_premier_level_1_with_vklad_monthly,0) 
+ coalesce(savings_account_without_premier_level_1_with_save,0) + coalesce(savings_account_without_premier_level_1_without_save,0))) as sum_benefit_rubles_lvl1,

round(sum(coalesce(spasibo_without_sber_premier_level_1,0) + coalesce(kuper_without_premier_level_1, 0) + coalesce(samokat_without_premier_level_1, 0))) as sum_benefit_bonuses_lvl1,

round(sum(coalesce(p2p_transfers_without_premier_level_2,0) + coalesce(sound_without_premier_level_2,0) 
+ coalesce(okko_without_premier_level_2,0) + coalesce(sbercard_service_without_premier_2,0) 
+ coalesce(deposit_without_premier_level_2_without_bestpremier,0) + coalesce(deposit_without_premier_level_2_with_vklad_monthly,0) 
+ coalesce(savings_account_without_premier_level_2_with_save,0) + coalesce(savings_account_without_premier_level_2_without_save,0) 
+ coalesce(business_lounge_without_premier_level_2,0))) as sum_benefit_rubles_lvl2,

round(sum(coalesce(spasibo_without_sber_premier_level_2,0) + coalesce(kuper_without_premier_level_2,0) + coalesce(samokat_without_premier_level_2,0))) as sum_benefit_bonuses_lvl2,

round(sum(coalesce(p2p_transfers_without_premier_level_3,0) + coalesce(sound_without_premier_level_3,0) 
+ coalesce(okko_without_premier_level_3,0) + coalesce(sbercard_service_without_premier_3,0) 
+ coalesce(deposit_without_premier_level_3_without_bestpremier,0) + coalesce(deposit_without_premier_level_3_with_vklad_monthly,0) 
+ coalesce(savings_account_without_premier_level_3_with_save,0) + coalesce(savings_account_without_premier_level_3_without_save,0) 
+ coalesce(business_lounge_without_premier_level_3,0))) as sum_benefit_rubles_lvl3,

round(sum(coalesce(spasibo_without_sber_premier_level_3,0) + coalesce(kuper_without_premier_level_3,0) + coalesce(samokat_without_premier_level_3,0))) as sum_benefit_bonuses_lvl3,

round(sum(coalesce(p2p_transfers_without_sber_first_level_4,0) + coalesce(sound_without_sber_first_level_4_profit,0)
+ coalesce(okko_without_sber_first_level_4_profit,0) + coalesce(sbercard_service_without_sb1_4,0) + coalesce(business_lounge_without_sber_first_level_4,0) 
+ coalesce(deposit_without_sber_first_level_4_without_bestpremier,0) + coalesce(deposit_without_sber_first_level_4_with_vklad_monthly,0)
+ coalesce(savings_account_without_sber_first_level_4_with_save,0) + coalesce(savings_account_without_sber_first_level_4_without_save,0))) as sum_benefit_rubles_lvl4,

round(sum(coalesce(spasibo_without_sber_first_level_4,0) + coalesce(kuper_without_sber_first_level_4,0) + coalesce(samokat_without_sber_first_level_4,0))) as sum_benefit_bonuses_lvl4,

round(sum(coalesce(p2p_transfers_without_sber_first_level_5,0) + coalesce(sound_without_sber_first_level_5_profit,0)
+ coalesce(okko_without_sber_first_level_5_profit,0) + coalesce(sbercard_service_without_sb1_5,0) + coalesce(business_lounge_without_sber_first_level_5,0)
+ coalesce(deposit_without_sber_first_level_5_without_bestpremier,0) + coalesce(deposit_without_sber_first_level_5_with_vklad_monthly,0)
+ coalesce(savings_account_without_sber_first_level_5_with_save,0) + coalesce(savings_account_without_sber_first_level_5_without_save,0))) as sum_benefit_rubles_lvl5,

round(sum(coalesce(spasibo_without_sber_first_level_5,0) + coalesce(kuper_without_sber_first_level_5,0) + coalesce(samokat_without_sber_first_level_5,0))) as sum_benefit_bonuses_lvl5,


round(sum(coalesce(p2p_transfers_without_sber_first_level_5,0) + coalesce(sound_without_sber_first_level_5_profit,0)
+ coalesce(okko_without_sber_first_level_5_profit,0) + coalesce(sbercard_service_without_sb1_5,0) + coalesce(business_lounge_without_sber_first_level_5,0)
+ coalesce(deposit_without_sber_first_level_5_without_bestpremier,0) + coalesce(deposit_without_sber_first_level_5_with_vklad_monthly,0)
+ coalesce(savings_account_without_sber_first_level_5_with_save,0) + coalesce(savings_account_without_sber_first_level_5_without_save,0))) as sum_benefit_rubles_lvl6,

round(sum(coalesce(spasibo_without_sber_first_level_5,0) + coalesce(kuper_without_sber_first_level_5,0) + coalesce(samokat_without_sber_first_level_5,0))) as sum_benefit_bonuses_lvl6

from arnsdpsbx_team_ss.bpm_agent_py_levels_all a
left join {potential_benefit} b on a.epk = b.epk_id
where report_dt = '{date_now_free}' and death_flag = 0
group by epk
)

select * from arnsdpsbx_team_ss.bpm_agent_py_levels_all
left join poten_benefit using(epk)
''').write.saveAsTable(f'arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit', mode='overwrite')

# Core продукты

In [15]:
p2p_transfers=spark.sql(f"""
with t1 as (
select t.epk, 
sum(case when trx_type in ('4060' , '4062','4061','4050','4052','4051','4071') then trx_rur_amt else null end) as transfer_p2p_com
from {pos_p2p} tt
right join arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit t
on t.epk=tt.epk_id
where month_part>=trunc(add_months('{date_now_free}', -1), 'MM') --пример для расчета для мая 2025
and month_part < trunc('{date_now_free}', 'mm') --пример для расчета для мая 2025
group by t.epk)

select distinct epk, 
case when transfer_p2p_com > 0 then 1 else 0 end as p2p_transfer_flag
from t1
""")

p2p_transfers.createOrReplaceTempView("p2p_transfers_flags")

In [16]:
pays_comissions = spark.sql(f'''
with t1 as (
select main.epk, 
case when other.trans_type in('1000025' , '3001' , '3030' , '3800' , '3801') then amt else 0 end as year_com_rub_trans, -- обслуживание карты
case when other.trans_type in('320', '3200','1000026') then amt else 0 end as mob_bank_rub_trans --мобильный банк(уведомления)
from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit main
left join {pos} other on main.epk = other.epk_id
where day_part between last_day(date('{date_now_free}') - interval 1 month) and {date_now_free}
),

t2 as (
select main.epk,
sum(case when trx_type in ('4000','4001','4010','4011','4020','4021','4100','4110','4200','4210','4500') then trx_rur_amt else null end) as com_cash -- комиссия за снятие наличных
from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit main
left join {pos_p2p} other on main.epk = other.epk_id
where month_part between last_day(date('{date_now_free}') - interval 1 month) and {date_now_free}
group by main.epk
)

select distinct epk, year_com_rub_trans, mob_bank_rub_trans, com_cash
from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit
left join t1 using(epk)
left join t2 using(epk)
''')

pays_comissions.createOrReplaceTempView("pays_comissions_and_pushes")

In [17]:
prime=spark.sql(f"""
with t as(
select distinct epk, 
case when packetcategory in ('SberPrime','SberPrime2','SberPrimeSbol','SberPrimeVmeste') then 1 else 0 end as prime,
case when packetcategory in ('SberPrimePlus','SberPrimePlus2','PrimePlusGroup','SberPrimePlusVmeste') then 1 else 0 end as primepl,
case when (lower(packetname) like ('%старт%')) or (lower(packetname) like ('%start%')) then '1' else '0' end as prime_start
from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit a 
left join {sber_prime} b on a.epk = b.epk_id
where state = 'ACTIVE'),

pr as(
select epk, max (prime) as prime, max (primepl) as primepl
from t
where prime_start = '0'
group by epk)

select distinct e.epk, 
pr.prime, pr.primepl
from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit e
left join pr on e.epk=pr.epk
""")

prime.createOrReplaceTempView("prime_flags")

In [18]:
deposits = spark.sql(f'''
select distinct epk, 
            case when lower(qsname) like '%премьер%' and cast(closeday as date) > last_day(date('{date_now_free}') + interval 2 month) 
                then 1 else 0 end as only_deposit_premier,
            case when lower(qsname) like '%лидер%' and cast(closeday as date) > last_day(date('{date_now_free}') + interval 2 month) 
                then 1 else 0 end as only_deposit_sb1,
            case when lower(qsname) like '%private%' and cast(closeday as date) > last_day(date('{date_now_free}') + interval 2 month) 
                then 1 else 0 end as only_deposit_pb
from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit a
left join {deposit} b on a.epk = b.epk_id
where cast(report_date as date) = '{date_now_free}'
    and state != 2
    and deposit_type in ('Срочные счета','Новые НС', 'Остальные НС', 'Текущие счета')
    and qsname in ('Лучший % Премьер', 'Лучший % Лидер +',
       'Лучший % Лидер','СберВклад Лидер',
       'Лучший % Лидер Зарплатный', 'Управляй Лидер +',
       'СберВклад Прайм Лидер', 'Ключевой Лидер',
       'СберВклад Лидер +',
       'Управляй Лидер', 
       'Лидер Управляй', 
       'СберВклад Прайм Премьер',
       'Лидер Пополняй', 
       'Лидер Управляй +', 'Лучший % Премьер Зарплатный',
       'Лидер Сохраняй', 
       'Ключевой Премьер',
       'Ежедневный % Премьер', 
       'СберВклад Премьер',
       'Private Banking Сохраняй', 
       'Управляй Лидер Зарплатный')
''')

deposits.createOrReplaceTempView("deposits_flags")

In [19]:
savings = spark.sql(f'''
select distinct epk, case when lower(qsname) like '%премьер%' and cast(closeday as date) > last_day(date('{date_now_free}') + interval 2 month)
                then 1 else 0 end as only_savings_premier,
            case when lower(qsname) like '%лидер%' and cast(closeday as date) > last_day(date('{date_now_free}') + interval 2 month)
                then 1 else 0 end as only_savings_sb1,
            case when lower(qsname) like '%private%' and cast(closeday as date) > last_day(date('{date_now_free}') + interval 2 month) 
                then 1 else 0 end as only_savings_pb
from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit a
left join {deposit} b on a.epk = b.epk_id
where cast(report_date as date) = '{date_now_free}'
    and state != 2
    and deposit_type in ('Срочные счета','Новые НС', 'Остальные НС', 'Текущие счета')
    and qsname in ( 
       'Накопительный счет Лидер +', 
       'Private накопительный счет', 'Накопительный счет Премьер',
       'Накопительный счет Лидер')
''')

savings.createOrReplaceTempView("savings_flags")

In [20]:
coin = spark.sql(f'''
select distinct epk, case when lower(scenario_name) like '%покупка%' then 1 else 0 end as coins_flag
from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit a
left join {coins} b on a.epk = b.epkid
where epkid is not null and operday between last_day(date('{date_now_free}') - interval 18 month) and {date_now_free}
''')

coin.createOrReplaceTempView("coin_flags")

In [22]:
spark.sql(f'''
with flags_aggr as (
    select distinct a.epk as epk, crd_open_mir_nflag as card_flag, crd_cc_open_mir_nflag as credit_card_flag,
            prd_pl_active_nflag as potreb_credit_flag, srv_thanks_nflag as spasibo_flag,
            p2p_transfer_flag, primepl as prime_plus_flag, prime as prime_flag,
            year_com_rub_trans, mob_bank_rub_trans, com_cash,
            only_savings_premier, only_savings_sb1, only_savings_pb,
            only_deposit_premier, only_deposit_sb1, only_deposit_pb,
            case when dep_acct_ma_bal_rub_amt > x then 1 else 0 end as oms_flag,
            crd_trx_spnd_pos_avg_3m_amt, coins_flag
        from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit a
        left join {aggr} b on a.epk = b.epk_id
        left join p2p_transfers_flags c using(epk)
        left join prime_flags using(epk)
        left join pays_comissions_and_pushes using(epk)
        left join deposits_flags using(epk)
        left join savings_flags using(epk)
        left join coin_flags using(epk)
        where report_dt = '{date_now_free}' and cla_full_active_nflag = 1 and sd_dead_nflag = 0
),
 
sticker_and_metal as (
    select epk,
    max(case when card_design like '%SBPAYSTIK%' then 1 else 0 end) as sticker_flag,
    max(case when card_design like '%SBERMETAL%' then 1 else 0 end) as metalCard_flag
    from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit a
    left join {stick} b on a.epk = b.epk_id
    where row_actual_to_dt ='9999-12-31' and (card_design like '%SBPAYSTIK%' or card_design = 'SBSTDSTIKNO' or card_design like '%SBERMETAL%')
    group by epk
),
 
balance_for_savings as (
select distinct epk, avg(dep_acct_dep_ca_bal_rub_amt) as avg_balance_3m, sum(crd_dc_m2m_other_bank_amt) as sum_m2m_to_others
from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit a
left join {aggr} b on a.epk = b.epk_id
where report_dt between last_day(date('{date_now_free}') - interval 2 month) and '{date_now_free}'
group by epk
),
 
nsi as (
select epk, max(case when nsi_id = '2770' then 1 else 0 end) as subscriptionIp_flag_2770,
            max(case when nsi_id = '41227' then 1 else 0 end) as subscriptionIp_flag_41227 
from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit a
left join {tags} b on a.epk = b.epk_id
where row_actual_to_dt = '9999-12-31'  and is_deleted = 0
and nsi_id in ('2770', '41227')
group by epk
),
 
all_flags as (
    select distinct base.epk as epk, max(card_flag) as card_flag, max(credit_card_flag) as credit_card_flag,
            max(potreb_credit_flag) as potreb_credit_flag, max(spasibo_flag) as spasibo_flag,
            max(p2p_transfer_flag) as p2p_transfer_flag, max(prime_plus_flag) as prime_plus_flag, max(prime_flag) as prime_flag,
            max(year_com_rub_trans) as year_com_rub_trans, max(mob_bank_rub_trans) as mob_bank_rub_trans, max(com_cash) as com_cash,
            max(only_savings_premier) as only_savings_premier, max(only_savings_sb1) as only_savings_sb1,
            max(only_savings_pb) as only_savings_pb,
            max(only_deposit_premier) as only_deposit_premier,
            max(only_deposit_sb1) as only_deposit_sb1, max(only_deposit_pb) as only_deposit_pb,
            max(oms_flag) as oms_flag, max(crd_trx_spnd_pos_avg_3m_amt) as crd_trx_spnd_pos_avg_3m_amt, max(coins_flag) as coins_flag,
            max(subscriptionIp_flag_2770) as subscriptionIp_flag_2770,
            max(subscriptionIp_flag_41227) as subscriptionIp_flag_41227,
            max(avg_balance_3m) as avg_balance_3m, max(sum_m2m_to_others) as sum_m2m_to_others,
            max(sticker_flag) as sticker_flag, max(metalCard_flag) as metalCard_flag
    from flags_aggr base
    left join nsi using(epk)
    left join sticker_and_metal using(epk)
    left join balance_for_savings using(epk)
    group by base.epk
),
 
 
 
initial_data as (
    select distinct a.epk as epk, max(approval_level) as approval_level,
        max(coalesce(card_flag, 0)) as card_flag, max(coalesce(credit_card_flag, 0)) as credit_card_flag,
        max(coalesce(potreb_credit_flag, 0)) as potreb_credit_flag, max(coalesce(spasibo_flag, 0)) as spasibo_flag,
        max(coalesce(p2p_transfer_flag, 0)) as p2p_transfer_flag, max(coalesce(prime_plus_flag, 0)) as prime_plus_flag, max(coalesce(prime_flag, 0)) as prime_flag,
        max(coalesce(year_com_rub_trans, 0)) as year_com_rub_trans, max(coalesce(mob_bank_rub_trans, 0)) as mob_bank_rub_trans, max(coalesce(com_cash, 0)) as com_cash,
        max(coalesce(only_savings_premier, 0)) as only_savings_premier,
        max(coalesce(only_savings_sb1, 0)) as only_savings_sb1, max(coalesce(only_savings_pb, 0)) as only_savings_pb,
        max(coalesce(only_deposit_premier, 0)) as only_deposit_premier,
        max(coalesce(only_deposit_sb1, 0)) as only_deposit_sb1, max(coalesce(only_deposit_pb, 0)) as only_deposit_pb,
        max(coalesce(oms_flag, 0)) as oms_flag, max(coalesce(sticker_flag, 0)) as sticker_flag, max(coalesce(avg_balance_3m, 0)) as avg_balance_3m,
        max(coalesce(sum_m2m_to_others, 0)) as sum_m2m_to_others, max(coalesce(metalCard_flag, 0)) as metalCard_flag, max(coalesce(coins_flag, 0)) as coins_flag,
        max(coalesce(crd_trx_spnd_pos_avg_3m_amt, 0)) as crd_trx_spnd_pos_avg_3m_amt, max(coalesce(subscriptionIp_flag_2770, 0)) as subscriptionIp_flag_2770,
        max(coalesce(subscriptionIp_flag_41227, 0)) as subscriptionIp_flag_41227,
       
        max(coalesce(p2p_transfers_without_premier_level_1, 0)) as p2p_transfers_without_premier_level_1,
        max(coalesce(spasibo_without_sber_premier_level_1, 0)) as spasibo_without_sber_premier_level_1,
        max(coalesce(sbercard_service_without_premier_1, 0)) as sbercard_service_without_premier_1,
        max(coalesce(deposit_without_premier_level_1_without_bestpremier, 0)) as deposit_without_premier_level_1_without_bestpremier,
        max(coalesce(deposit_without_premier_level_1_with_vklad_monthly, 0)) as deposit_without_premier_level_1_with_vklad_monthly,
        max(coalesce(savings_account_without_premier_level_1_with_save, 0)) as savings_account_without_premier_level_1_with_save,
        max(coalesce(savings_account_without_premier_level_1_without_save, 0)) as savings_account_without_premier_level_1_without_save,
       
        max(coalesce(p2p_transfers_with_premier_level_1, 0)) as p2p_transfers_with_premier_level_1,
        max(coalesce(deposit_with_premier_level_1_without_bestpremier, 0)) as deposit_with_premier_level_1_without_bestpremier,
        max(coalesce(deposit_with_premier_level_1_with_vklad_monthly, 0)) as deposit_with_premier_level_1_with_vklad_monthly,
        max(coalesce(savings_account_with_premier_level_1_with_save, 0)) as savings_account_with_premier_level_1_with_save,
        max(coalesce(savings_account_with_premier_level_1_without_save, 0)) as savings_account_with_premier_level_1_without_save,
       
        max(coalesce(p2p_transfers_without_premier_level_2, 0)) as p2p_transfers_without_premier_level_2,
        max(coalesce(spasibo_without_sber_premier_level_2, 0)) as spasibo_without_sber_premier_level_2,
        max(coalesce(sbercard_service_without_premier_2, 0)) as sbercard_service_without_premier_2,
        max(coalesce(deposit_without_premier_level_2_without_bestpremier, 0)) as deposit_without_premier_level_2_without_bestpremier,
        max(coalesce(deposit_without_premier_level_2_with_vklad_monthly, 0)) as deposit_without_premier_level_2_with_vklad_monthly,
        max(coalesce(savings_account_without_premier_level_2_with_save, 0)) as savings_account_without_premier_level_2_with_save,
        max(coalesce(savings_account_without_premier_level_2_without_save, 0)) as savings_account_without_premier_level_2_without_save,
       
        max(coalesce(p2p_transfers_with_premier_level_2, 0)) as p2p_transfers_with_premier_level_2,
        max(coalesce(deposit_with_premier_level_2_without_bestpremier, 0)) as deposit_with_premier_level_2_without_bestpremier,
        max(coalesce(deposit_with_premier_level_2_with_vklad_monthly, 0)) as deposit_with_premier_level_2_with_vklad_monthly,
        max(coalesce(savings_account_with_premier_level_2_with_save, 0)) as savings_account_with_premier_level_2_with_save,
        max(coalesce(savings_account_with_premier_level_2_without_save, 0)) as savings_account_with_premier_level_2_without_save,
       
        max(coalesce(p2p_transfers_without_premier_level_3, 0)) as p2p_transfers_without_premier_level_3,
        max(coalesce(spasibo_without_sber_premier_level_3, 0)) as spasibo_without_sber_premier_level_3,
        max(coalesce(sbercard_service_without_premier_3, 0)) as sbercard_service_without_premier_3,
        max(coalesce(deposit_without_premier_level_3_without_bestpremier, 0)) as deposit_without_premier_level_3_without_bestpremier,
        max(coalesce(deposit_without_premier_level_3_with_vklad_monthly, 0)) as deposit_without_premier_level_3_with_vklad_monthly,
        max(coalesce(savings_account_without_premier_level_3_with_save, 0)) as savings_account_without_premier_level_3_with_save,
        max(coalesce(savings_account_without_premier_level_3_without_save, 0)) as savings_account_without_premier_level_3_without_save,
       
        max(coalesce(p2p_transfers_with_premier_level_3, 0)) as p2p_transfers_with_premier_level_3,
        max(coalesce(deposit_with_premier_level_3_without_bestpremier, 0)) as deposit_with_premier_level_3_without_bestpremier,
        max(coalesce(deposit_with_premier_level_3_with_vklad_monthly, 0)) as deposit_with_premier_level_3_with_vklad_monthly,
        max(coalesce(savings_account_with_premier_level_3_with_save, 0)) as savings_account_with_premier_level_3_with_save,
        max(coalesce(savings_account_with_premier_level_3_without_save, 0)) as savings_account_with_premier_level_3_without_save,
       
        max(coalesce(p2p_transfers_without_sber_first_level_4, 0)) as p2p_transfers_without_sber_first_level_4,
        max(coalesce(spasibo_without_sber_first_level_4, 0)) as spasibo_without_sber_first_level_4,
        max(coalesce(sbercard_service_without_sb1_4, 0)) as sbercard_service_without_sb1_4,
        max(coalesce(deposit_without_sber_first_level_4_without_bestpremier, 0)) as deposit_without_sber_first_level_4_without_bestpremier,
        max(coalesce(deposit_without_sber_first_level_4_with_vklad_monthly, 0)) as deposit_without_sber_first_level_4_with_vklad_monthly,
        max(coalesce(savings_account_without_sber_first_level_4_with_save, 0)) as savings_account_without_sber_first_level_4_with_save,
        max(coalesce(savings_account_without_sber_first_level_4_without_save, 0)) as savings_account_without_sber_first_level_4_without_save,
       
        max(coalesce(p2p_transfers_with_sber_first_level_4, 0)) as p2p_transfers_with_sber_first_level_4,
        max(coalesce(deposit_with_sber_first_level_4_without_bestpremier, 0)) as deposit_with_sber_first_level_4_without_bestpremier,
        max(coalesce(deposit_with_sber_first_level_4_with_vklad_monthly, 0)) as deposit_with_sber_first_level_4_with_vklad_monthly,
        max(coalesce(savings_account_with_sber_first_level_4_with_save, 0)) as savings_account_with_sber_first_level_4_with_save,
        max(coalesce(savings_account_with_sber_first_level_4_without_save, 0)) as savings_account_with_sber_first_level_4_without_save,
       
        max(coalesce(p2p_transfers_without_sber_first_level_5, 0)) as p2p_transfers_without_sber_first_level_5,
        max(coalesce(spasibo_without_sber_first_level_5, 0)) as spasibo_without_sber_first_level_5,
        max(coalesce(sbercard_service_without_sb1_5, 0)) as sbercard_service_without_sb1_5,
        max(coalesce(deposit_without_sber_first_level_5_without_bestpremier, 0)) as deposit_without_sber_first_level_5_without_bestpremier,
        max(coalesce(deposit_without_sber_first_level_5_with_vklad_monthly, 0)) as deposit_without_sber_first_level_5_with_vklad_monthly,
        max(coalesce(savings_account_without_sber_first_level_5_with_save, 0)) as savings_account_without_sber_first_level_5_with_save,
        max(coalesce(savings_account_without_sber_first_level_5_without_save, 0)) as savings_account_without_sber_first_level_5_without_save,
        
        max(coalesce(p2p_transfers_with_sber_first_level_5, 0)) as p2p_transfers_with_sber_first_level_5,
        max(coalesce(deposit_with_sber_first_level_5_without_bestpremier, 0)) as deposit_with_sber_first_level_5_without_bestpremier,
        max(coalesce(deposit_with_sber_first_level_5_with_vklad_monthly, 0)) as deposit_with_sber_first_level_5_with_vklad_monthly,
        max(coalesce(savings_account_with_sber_first_level_5_with_save, 0)) as savings_account_with_sber_first_level_5_with_save,
        max(coalesce(savings_account_with_sber_first_level_5_without_save, 0)) as savings_account_with_sber_first_level_5_without_save,

        # NPV replacement values, since I am under NDA
        1 as npv_sbercard_all,
        2 as npv_deposit_premier,
        3 as npv_deposit_sb1,
        4 as npv_deposit_best_premier,
        5 as npv_deposit_best_sb1,
        6 as npv_savings_account_premier,
        7 as npv_savings_account_sb1,
        8 as npv_credit_card_sb1,
        9 as npv_oms_all,
        10 as npv_potreb_credit_sb1
    from all_flags a
    left join arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit using(epk)
    left join {potential_benefit} d on a.epk = d.epk_id
    where report_dt = '{date_now_free}' and death_flag = 0
    group by a.epk
),

# The exact values have been replaced since I am under NDA.
scoring_data as (
select distinct epk, sticker_flag, coins_flag, subscriptionIp_flag_2770, subscriptionIp_flag_41227,
case when card_flag = 1 and (prime_flag = 0 or prime_plus_flag = 0 or com_cash = 1 or mob_bank_rub_trans = 1) and sbercard_service_without_premier_1 > 0
        then sbercard_service_without_premier_1
    when card_flag = 0 and (year_com_rub_trans = 1 or mob_bank_rub_trans = 1 or com_cash = 1)
        then round((0.4 * sbercard_service_without_premier_1) + (0.6 * npv_sbercard_all), 4)
    else 1 end as score_sbercard_premier_1,
   
case when metalCard_flag = 1 and (mob_bank_rub_trans = 1 or com_cash = 1) and sbercard_service_without_premier_1 > 0 then sbercard_service_without_premier_1
    when metalCard_flag = 0 and (year_com_rub_trans = 1 or mob_bank_rub_trans = 1 or com_cash = 1)
        then round((0.4 * sbercard_service_without_premier_1) + (0.6 * npv_sbercard_all), 4)
    else 1 end as score_metalcard_premier_1,
       
case when only_deposit_premier = 0
        and coalesce(deposit_without_premier_level_1_without_bestpremier, 0) + coalesce(deposit_without_premier_level_1_with_vklad_monthly, 0) > 0
        then coalesce(deposit_without_premier_level_1_without_bestpremier, 0) + coalesce(deposit_without_premier_level_1_with_vklad_monthly, 0)
    when only_deposit_premier = 0 and only_deposit_sb1 = 0 and only_deposit_pb = 0
        and (avg_balance_3m > x or sum_m2m_to_others > x)
        then round((0.4 * (coalesce(deposit_without_premier_level_1_without_bestpremier, 0) + coalesce(deposit_without_premier_level_1_with_vklad_monthly, 0))) + (0.6 * npv_deposit_premier), 4)
        else 1 end as score_deposit_best_premier_1,
       
case when only_savings_premier = 0
        and coalesce(savings_account_without_premier_level_1_with_save, 0) + coalesce(savings_account_without_premier_level_1_without_save, 0) > 0
        then coalesce(savings_account_without_premier_level_1_with_save, 0) + coalesce(savings_account_without_premier_level_1_without_save, 0)
    when only_savings_premier = 0 and only_savings_sb1 = 0 and only_savings_pb = 0 and avg_balance_3m > 50000
        then round((0.4 * (coalesce(savings_account_without_premier_level_1_with_save, 0) + coalesce(savings_account_without_premier_level_1_without_save, 0))) + (0.6 * npv_savings_account_premier), 4)
        else 1 end as score_savings_account_with_save_premier_1,
       
case when oms_flag = 1 then round(0.6 * npv_oms_all, 4)
    when oms_flag = 0 and coins_flag = 1 then round(0.5 * npv_oms_all, 4)
        else 1 end as score_oms_premier_1,
       
case when spasibo_flag = 1 and (prime_flag = 1 or prime_plus_flag = 1) and spasibo_without_sber_premier_level_1 > 0 then spasibo_without_sber_premier_level_1
    when spasibo_flag = 0 and crd_trx_spnd_pos_avg_3m_amt > x then round(0.5 * spasibo_without_sber_premier_level_1, 4)
        else 1 end as score_spasibo_premier_1,
       
case when p2p_transfer_flag = 1 and (prime_flag = 1 or prime_plus_flag = 1) and p2p_transfers_without_premier_level_1 > 0 then p2p_transfers_without_premier_level_1
    when p2p_transfer_flag = 0 then round(0.5 * p2p_transfers_without_premier_level_1, 4)
        else 1 end as score_p2p_transfers_premier_1,
       

 
 
 
case when card_flag = 1 and (prime_flag = 0 or prime_plus_flag = 0 or com_cash = 1 or mob_bank_rub_trans = 1) and sbercard_service_without_premier_2 > 0
        then sbercard_service_without_premier_2
    when card_flag = 0 and (year_com_rub_trans = 1 or mob_bank_rub_trans = 1 or com_cash = 1)
        then round((0.4 * sbercard_service_without_premier_2) + (0.6 * npv_sbercard_all), 4)
    else 1 end as score_sbercard_premier_2,
   
case when metalCard_flag = 1 and (mob_bank_rub_trans = 1 or com_cash = 1) and sbercard_service_without_premier_2 > 0 then sbercard_service_without_premier_2
    when metalCard_flag = 0 and (year_com_rub_trans = 1 or mob_bank_rub_trans = 1 or com_cash = 1)
        then round((0.4 * sbercard_service_without_premier_2) + (0.6 * npv_sbercard_all), 4)
    else 1 end as score_metalcard_premier_2,
       
case when only_deposit_premier = 0
        and coalesce(deposit_without_premier_level_2_without_bestpremier, 0) + coalesce(deposit_without_premier_level_2_with_vklad_monthly, 0) > 0
        then coalesce(deposit_without_premier_level_2_without_bestpremier, 0) + coalesce(deposit_without_premier_level_2_with_vklad_monthly, 0)
    when only_deposit_premier = 0
        and (avg_balance_3m > x or sum_m2m_to_others > x)
        then round((0.4 * (coalesce(deposit_without_premier_level_2_without_bestpremier, 0) + coalesce(deposit_without_premier_level_2_with_vklad_monthly, 0))) + (0.6 * npv_deposit_premier), 4)
        else 1 end as score_deposit_best_premier_2,
       
case when only_savings_premier = 0
        and coalesce(savings_account_without_premier_level_2_with_save, 0) + coalesce(savings_account_without_premier_level_2_without_save, 0) > 0
        then coalesce(savings_account_without_premier_level_2_with_save, 0) + coalesce(savings_account_without_premier_level_2_without_save, 0)
    when only_savings_premier = 0 and avg_balance_3m > x
        then round((0.4 * (coalesce(savings_account_without_premier_level_2_with_save, 0) + coalesce(savings_account_without_premier_level_2_without_save, 0))) + (0.6 * npv_savings_account_premier), 4)
        else 1 end as score_savings_account_with_save_premier_2,
       
case when oms_flag = 1 then round(0.5 * npv_oms_all, 4)
    when oms_flag = 0 and coins_flag = 1 then round(0.6 * npv_oms_all, 4)
        else 1 end as score_oms_premier_2,
       
case when spasibo_flag = 1 and (prime_flag = 1 or prime_plus_flag = 1) and spasibo_without_sber_premier_level_2 > 0 then spasibo_without_sber_premier_level_2
    when spasibo_flag = 0 and crd_trx_spnd_pos_avg_3m_amt > x then round(0.5 * spasibo_without_sber_premier_level_2, 4)
        else 1 end as score_spasibo_premier_2,
       
case when p2p_transfer_flag = 1 and (prime_flag = 1 or prime_plus_flag = 1) and p2p_transfers_without_premier_level_2 > 0 then p2p_transfers_without_premier_level_2
    when p2p_transfer_flag = 0 then round(0.5 * p2p_transfers_without_premier_level_2, 4)
        else 1 end as score_p2p_transfers_premier_2,
 

   
 
 
case when card_flag = 1 and (prime_flag = 0 or prime_plus_flag = 0 or com_cash = 1 or mob_bank_rub_trans = 1) and sbercard_service_without_premier_3 > 0
        then sbercard_service_without_premier_3
    when card_flag = 0 and (year_com_rub_trans = 1 or mob_bank_rub_trans = 1 or com_cash = 1)
        then round((0.4 * sbercard_service_without_premier_2) + (0.6 * npv_sbercard_all), 4)
    else 1 end as score_sbercard_premier_3,
   
case when metalCard_flag = 1 and (mob_bank_rub_trans = 1 or com_cash = 1) and sbercard_service_without_premier_3 > 0 then sbercard_service_without_premier_3
    when metalCard_flag = 0 and (year_com_rub_trans = 1 or mob_bank_rub_trans = 1 or com_cash = 1)
        then round((0.4 * sbercard_service_without_premier_3) + (0.6 * npv_sbercard_all), 4)
    else 1 end as score_metalcard_premier_3,
       
case when only_deposit_premier = 0
        and coalesce(deposit_without_premier_level_3_without_bestpremier, 0) + coalesce(deposit_without_premier_level_3_with_vklad_monthly, 0) > 0
        then coalesce(deposit_without_premier_level_3_without_bestpremier, 0) + coalesce(deposit_without_premier_level_3_with_vklad_monthly, 0)
    when only_deposit_premier = 0
        and (avg_balance_3m > x or sum_m2m_to_others > x)
        then round((0.4 * (coalesce(deposit_without_premier_level_3_without_bestpremier, 0) + coalesce(deposit_without_premier_level_3_with_vklad_monthly, 0))) + (0.6 * npv_deposit_premier), 4)
    else 1 end as score_deposit_best_premier_3,
       
case when only_savings_premier = 0
        and coalesce(savings_account_without_premier_level_3_with_save, 0) + coalesce(savings_account_without_premier_level_3_without_save, 0) > 0
        then coalesce(savings_account_without_premier_level_3_with_save, 0) + coalesce(savings_account_without_premier_level_3_without_save, 0)
    when only_savings_premier = 0 and avg_balance_3m > x
        then round((0.4 * (coalesce(savings_account_without_premier_level_3_with_save, 0) + coalesce(savings_account_without_premier_level_3_without_save, 0))) + (0.6 * npv_savings_account_premier), 4)
    else 1 end as score_savings_account_with_save_premier_3,
       
case when oms_flag = 1 then round(0.5 * npv_oms_all, 4)
    when oms_flag = 0 and coins_flag = 1 then round(0.6 * npv_oms_all, 4)
        else 1 end as score_oms_premier_3,
       
case when spasibo_flag = 1 and (prime_flag = 1 or prime_plus_flag = 1) and spasibo_without_sber_premier_level_3 > 0 then spasibo_without_sber_premier_level_3
    when spasibo_flag = 0 and crd_trx_spnd_pos_avg_3m_amt > x then round(0.5 * spasibo_without_sber_premier_level_3, 4)
        else 1 end as score_spasibo_premier_3,
       
case when p2p_transfer_flag = 1 and (prime_flag = 1 or prime_plus_flag = 1) and p2p_transfers_without_premier_level_3 > 0 then p2p_transfers_without_premier_level_3
    when p2p_transfer_flag = 0 then round(0.5 * p2p_transfers_without_premier_level_3, 4)
       else 1 end as score_p2p_transfers_premier_3,
       

       
        
        
        
case when card_flag = 1 and (prime_flag = 0 or prime_plus_flag = 0 or com_cash = 1 or mob_bank_rub_trans = 1) and sbercard_service_without_sb1_4 > 0
        then sbercard_service_without_sb1_4
    when card_flag = 0 and (year_com_rub_trans = 1 or mob_bank_rub_trans = 1 or com_cash = 1)
        then round((0.4 * sbercard_service_without_sb1_4) + (0.6 * npv_sbercard_all), 4)
    else 1 end as score_sbercard_sb1_1,
   
case when metalCard_flag = 1 and (mob_bank_rub_trans = 1 or com_cash = 1) and sbercard_service_without_sb1_4 > 0 then sbercard_service_without_sb1_4
    when metalCard_flag = 0 and (year_com_rub_trans = 1 or mob_bank_rub_trans = 1 or com_cash = 1)
        then round((0.4 * sbercard_service_without_sb1_4) + (0.6 * npv_sbercard_all), 4)
    else 1 end as score_metalcard_sb1_1,
       
case when only_deposit_sb1 = 0 and coalesce(deposit_without_sber_first_level_4_without_bestpremier, 0) + coalesce(deposit_without_sber_first_level_4_with_vklad_monthly, 0) > 0
        then coalesce(deposit_without_sber_first_level_4_without_bestpremier, 0) + coalesce(deposit_without_sber_first_level_4_with_vklad_monthly, 0)
    when only_deposit_sb1 = 0
        and (avg_balance_3m > x or sum_m2m_to_others > x)
        then round((0.4 * (coalesce(deposit_without_sber_first_level_4_without_bestpremier, 0) + coalesce(deposit_without_sber_first_level_4_with_vklad_monthly, 0))) + (0.6 * npv_deposit_sb1), 4)
    else 1 end as score_deposit_best_sb1_1,
       
case when only_savings_sb1 = 0 and coalesce(savings_account_without_sber_first_level_4_with_save, 0) + coalesce(savings_account_without_sber_first_level_4_without_save, 0) > 0
        then coalesce(savings_account_without_sber_first_level_4_with_save, 0) + coalesce(savings_account_without_sber_first_level_4_without_save, 0)
    when only_savings_premier = 0 and avg_balance_3m > x
        then round((0.4 * (coalesce(savings_account_without_sber_first_level_4_with_save, 0) + coalesce(savings_account_without_sber_first_level_4_without_save, 0))) + (0.6 * npv_savings_account_sb1), 4)
    else 1 end as score_savings_account_with_save_sb1_1,
       
case when credit_card_flag = 1 then round(0.1 * npv_credit_card_sb1)
    when credit_card_flag = 0 then round(0.2 * npv_credit_card_sb1)
        else 1 end as score_credit_card_sb1_1,
       
case when oms_flag = 1 then round(0.5 * npv_oms_all, 4)
    when oms_flag = 0 and coins_flag = 1 then round(0.6 * npv_oms_all, 4)
        else 1 end as score_oms_sb1_1,
       
case when potreb_credit_flag = 1 then round(0.1 * npv_potreb_credit_sb1, 4)
    when potreb_credit_flag = 0 then round(0.2 * npv_potreb_credit_sb1, 4)
        else 1 end as score_potreb_credit_sb1_1,
       
case when spasibo_flag = 1 and (prime_flag = 1 or prime_plus_flag = 1) and spasibo_without_sber_first_level_4 > 0 then spasibo_without_sber_first_level_4
    when spasibo_flag = 0 and crd_trx_spnd_pos_avg_3m_amt > x then round(0.5 * spasibo_without_sber_first_level_4, 4)
        else 1 end as score_spasibo_sb1_1,
       
case when p2p_transfer_flag = 1 and (prime_flag = 1 or prime_plus_flag = 1) and p2p_transfers_without_sber_first_level_4 > 0 then p2p_transfers_without_sber_first_level_4
    when p2p_transfer_flag = 0 then round(0.5 * p2p_transfers_without_sber_first_level_4, 4)
        else 1 end as score_p2p_transfers_sb1_1,
       

       
        
        
case when card_flag = 1 and (prime_flag = 0 or prime_plus_flag = 0 or com_cash = 1 or mob_bank_rub_trans = 1) and sbercard_service_without_sb1_5 > 0
        then sbercard_service_without_sb1_5
    when card_flag = 0 and (year_com_rub_trans = 1 or mob_bank_rub_trans = 1 or com_cash = 1)
        then round((0.4 * sbercard_service_without_sb1_5) + (0.6 * npv_sbercard_all), 4)
    else 1 end as score_sbercard_sb1_2,
   
case when metalCard_flag = 1 and (mob_bank_rub_trans = 1 or com_cash = 1) and sbercard_service_without_sb1_5 > 0 then sbercard_service_without_sb1_5
    when metalCard_flag = 0 and (year_com_rub_trans = 1 or mob_bank_rub_trans = 1 or com_cash = 1)
        then round((0.4 * sbercard_service_without_sb1_5) + (0.6 * npv_sbercard_all), 4)
    else 1 end as score_metalcard_sb1_2,
       
case when only_deposit_sb1 = 0 and coalesce(deposit_without_sber_first_level_5_without_bestpremier, 0) + coalesce(deposit_without_sber_first_level_5_with_vklad_monthly, 0) > 0
        then coalesce(deposit_without_sber_first_level_5_without_bestpremier, 0) + coalesce(deposit_without_sber_first_level_5_with_vklad_monthly, 0)
    when only_deposit_sb1 = 0
        and (avg_balance_3m > x or sum_m2m_to_others > x)
        then round((0.4 * (coalesce(deposit_without_sber_first_level_5_without_bestpremier, 0) + coalesce(deposit_without_sber_first_level_5_with_vklad_monthly, 0))) + (0.6 * npv_deposit_sb1), 4)
    else 1 end as score_deposit_best_sb1_2,
       
case when only_savings_sb1 = 0 and coalesce(savings_account_without_sber_first_level_5_with_save, 0) + coalesce(savings_account_without_sber_first_level_5_without_save, 0) > 0
        then coalesce(savings_account_without_sber_first_level_5_with_save, 0) + coalesce(savings_account_without_sber_first_level_5_without_save, 0)
    when only_savings_premier = 0 and avg_balance_3m > x
        then round((0.4 * (coalesce(savings_account_without_sber_first_level_5_with_save, 0) + coalesce(savings_account_without_sber_first_level_5_without_save, 0))) + (0.6 * npv_savings_account_sb1), 4)
    else 1 end as score_savings_account_with_save_sb1_2,
       
case when credit_card_flag = 1 then round(0.1 * npv_credit_card_sb1)
    when credit_card_flag = 0 then round(0.2 * npv_credit_card_sb1)
        else 1 end as score_credit_card_sb1_2,
       
case when oms_flag = 1 then round(0.5 * npv_oms_all, 4)
    when oms_flag = 0 and coins_flag = 1 then round(0.6 * npv_oms_all, 4)
        else 1 end as score_oms_sb1_2,
       
case when potreb_credit_flag = 1 then round(0.1 * npv_potreb_credit_sb1, 4)
    when potreb_credit_flag = 0 then round(0.2 * npv_potreb_credit_sb1, 4)
        else 1 end as score_potreb_credit_sb1_2,
       
case when spasibo_flag = 1 and (prime_flag = 1 or prime_plus_flag = 1) and spasibo_without_sber_first_level_5 > 0 then spasibo_without_sber_first_level_5
    when spasibo_flag = 0 and crd_trx_spnd_pos_avg_3m_amt > x then round(0.5 * spasibo_without_sber_first_level_5, 4)
        else 1 end as score_spasibo_sb1_2,
       
case when p2p_transfer_flag = 1 and (prime_flag = 1 or prime_plus_flag = 1) and p2p_transfers_without_sber_first_level_5 > 0 then p2p_transfers_without_sber_first_level_5
    when p2p_transfer_flag = 0 then round(0.5 * p2p_transfers_without_sber_first_level_5, 4)
        else 1 end as score_p2p_transfers_sb1_2
       
   
from initial_data
),
 
 
final as (
    # I can't show you how the Offer Type is calculated because I'm under an NDA.
)
 
select distinct * from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit
left join final using(epk)
''').write.saveAsTable(f'arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_product', mode='overwrite')

In [23]:
spark.sql('''
select count(epk), count(distinct epk) from arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_product
''').show()

+----------+-------------------+
|count(epk)|count(DISTINCT epk)|
+----------+-------------------+
|  14098384|           14098384|
+----------+-------------------+



# Lifestyle продукты

In [24]:
lifestyleGeneral_advantages = spark.sql(f"""
WITH transaction_cat AS (
    SELECT
        a.epk,
        b.amt,
        b.day_part,
        cast(b.mcc_code as string) as mcc_code,

        -- 1) БАЗОВАЯ категоризация ТОЛЬКО по тратам (MCC)
        CASE
            WHEN cast(b.mcc_code as string) IN (
                '79996','5947','7999','7991','4112',
                '4722','4789','7922','9399','5812',
                '89999','7299','5944'
            ) THEN 'excursion'

            WHEN cast(b.mcc_code as string) IN (
                '7298','7230','5812','8099','8050',
                '8011','5995','5814','5977','8041',
                '5976','7399','5211','5331'
            ) THEN 'spa'

            WHEN cast(b.mcc_code as string) = '7941' THEN 'training'
            WHEN cast(b.mcc_code as string) = '3991' THEN 'paidTickets'
            WHEN cast(b.mcc_code as string) IN ('7922','3990','7299','5812') THEN 'concerts'

            WHEN cast(b.mcc_code as string) IN (
                '7922','7991','3990','9399',
                '7999','7991','5945','5462',
                '5441','7996','8299','5947',
                '5971','8220','7399','4722'
            ) THEN 'exhibitions'

            ELSE ''
        END AS spend_category,

        -- 2) Флаги regular ТОЛЬКО по тратам (чтобы теги не попадали в regular)
        CASE
            WHEN
                (
                    CASE
                        WHEN cast(b.mcc_code as string) IN (
                            '79996','5947','7999','7991','4112',
                            '4722','4789','7922','9399','5812',
                            '89999','7299','5944'
                        ) THEN 'excursion'
                        WHEN cast(b.mcc_code as string) IN (
                            '7298','7230','5812','8099','8050',
                            '8011','5995','5814','5977','8041',
                            '5976','7399','5211','5331'
                        ) THEN 'spa'
                        WHEN cast(b.mcc_code as string) = '7941' THEN 'training'
                        WHEN cast(b.mcc_code as string) = '3991' THEN 'paidTickets'
                        WHEN cast(b.mcc_code as string) IN ('7922','3990','7299','5812') THEN 'concerts'
                        WHEN cast(b.mcc_code as string) IN (
                            '7922','7991','3990','9399',
                            '7999','7991','5945','5462',
                            '5441','7996','8299','5947',
                            '5971','8220','7399','4722'
                        ) THEN 'exhibitions'
                        ELSE ''
                    END
                ) <> ''
                AND b.amt > 0
                AND b.day_part BETWEEN trunc(add_months(to_date('{date_now_free}'), -2), 'MM')
                                   AND last_day(date('{date_now_free}') - interval 2 month)
            THEN 1 ELSE 0 END AS flag_buy_3month_ago,

        CASE
            WHEN
                (
                    CASE
                        WHEN cast(b.mcc_code as string) IN (
                            '79996','5947','7999','7991','4112',
                            '4722','4789','7922','9399','5812',
                            '89999','7299','5944'
                        ) THEN 'excursion'
                        WHEN cast(b.mcc_code as string) IN (
                            '7298','7230','5812','8099','8050',
                            '8011','5995','5814','5977','8041',
                            '5976','7399','5211','5331'
                        ) THEN 'spa'
                        WHEN cast(b.mcc_code as string) = '7941' THEN 'training'
                        WHEN cast(b.mcc_code as string) = '3991' THEN 'paidTickets'
                        WHEN cast(b.mcc_code as string) IN ('7922','3990','7299','5812') THEN 'concerts'
                        WHEN cast(b.mcc_code as string) IN (
                            '7922','7991','3990','9399',
                            '7999','7991','5945','5462',
                            '5441','7996','8299','5947',
                            '5971','8220','7399','4722'
                        ) THEN 'exhibitions'
                        ELSE ''
                    END
                ) <> ''
                AND b.amt > 0
                AND b.day_part BETWEEN trunc(add_months(to_date('{date_now_free}'), -1), 'MM')
                                   AND last_day(date('{date_now_free}') - interval 1 month)
            THEN 1 ELSE 0 END AS flag_buy_2month_ago,

        CASE
            WHEN
                (
                    CASE
                        WHEN cast(b.mcc_code as string) IN (
                            '79996','5947','7999','7991','4112',
                            '4722','4789','7922','9399','5812',
                            '89999','7299','5944'
                        ) THEN 'excursion'
                        WHEN cast(b.mcc_code as string) IN (
                            '7298','7230','5812','8099','8050',
                            '8011','5995','5814','5977','8041',
                            '5976','7399','5211','5331'
                        ) THEN 'spa'
                        WHEN cast(b.mcc_code as string) = '7941' THEN 'training'
                        WHEN cast(b.mcc_code as string) = '3991' THEN 'paidTickets'
                        WHEN cast(b.mcc_code as string) IN ('7922','3990','7299','5812') THEN 'concerts'
                        WHEN cast(b.mcc_code as string) IN (
                            '7922','7991','3990','9399',
                            '7999','7991','5945','5462',
                            '5441','7996','8299','5947',
                            '5971','8220','7399','4722'
                        ) THEN 'exhibitions'
                        ELSE ''
                    END
                ) <> ''
                AND b.amt > 0
                AND b.day_part BETWEEN trunc(to_date('{date_now_free}'), 'MM') AND '{date_now_free}'
            THEN 1 ELSE 0 END AS flag_buy_1month_ago

    FROM arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_product a
    LEFT JOIN {pos} b
        ON a.epk = b.epk_id
    WHERE b.mcc_code IS NOT NULL
      AND b.ecom_fl = 0
      AND b.is_transaction = 1
      AND b.day_part BETWEEN trunc(add_months(to_date('{date_now_free}'), -2), 'MM') AND '{date_now_free}'
),

-- оставляем только транзакции, которые попали в spend_category (это важно!)
tx_filtered AS (
    SELECT *
    FROM transaction_cat
    WHERE spend_category <> ''
),

products_stats AS (
    SELECT
        epk,
        spend_category AS category,
        count(amt) AS transaction_count,
        sum(amt)   AS total_amount
    FROM tx_filtered
    GROUP BY epk, spend_category
),

pivoted_stats AS (
    SELECT
        epk,

        sum(CASE WHEN category='autoRent'     THEN transaction_count ELSE 0 END) AS autoRent_transaction_count,
        sum(CASE WHEN category='autoRent'     THEN total_amount      ELSE 0 END) AS autoRent_total_amount,

        sum(CASE WHEN category='excursion'    THEN transaction_count ELSE 0 END) AS excursion_transaction_count,
        sum(CASE WHEN category='excursion'    THEN total_amount      ELSE 0 END) AS excursion_total_amount,

        sum(CASE WHEN category='tourOrHotel'  THEN transaction_count ELSE 0 END) AS tourOrHotel_transaction_count,
        sum(CASE WHEN category='tourOrHotel'  THEN total_amount      ELSE 0 END) AS tourOrHotel_total_amount,

        sum(CASE WHEN category='eventForeign' THEN transaction_count ELSE 0 END) AS eventForeign_transaction_count,
        sum(CASE WHEN category='eventForeign' THEN total_amount      ELSE 0 END) AS eventForeign_total_amount,

        sum(CASE WHEN category='spa'          THEN transaction_count ELSE 0 END) AS spa_transaction_count,
        sum(CASE WHEN category='spa'          THEN total_amount      ELSE 0 END) AS spa_total_amount,

        sum(CASE WHEN category='event'        THEN transaction_count ELSE 0 END) AS event_transaction_count,
        sum(CASE WHEN category='event'        THEN total_amount      ELSE 0 END) AS event_total_amount,

        sum(CASE WHEN category='training'     THEN transaction_count ELSE 0 END) AS training_transaction_count,
        sum(CASE WHEN category='training'     THEN total_amount      ELSE 0 END) AS training_total_amount,

        sum(CASE WHEN category='lecture'      THEN transaction_count ELSE 0 END) AS lecture_transaction_count,
        sum(CASE WHEN category='lecture'      THEN total_amount      ELSE 0 END) AS lecture_total_amount,

        sum(CASE WHEN category='paidTickets'  THEN transaction_count ELSE 0 END) AS paidTickets_transaction_count,
        sum(CASE WHEN category='paidTickets'  THEN total_amount      ELSE 0 END) AS paidTickets_total_amount,

        sum(CASE WHEN category='concerts'     THEN transaction_count ELSE 0 END) AS concerts_transaction_count,
        sum(CASE WHEN category='concerts'     THEN total_amount      ELSE 0 END) AS concerts_total_amount,

        sum(CASE WHEN category='exhibitions'  THEN transaction_count ELSE 0 END) AS exhibitions_transaction_count,
        sum(CASE WHEN category='exhibitions'  THEN total_amount      ELSE 0 END) AS exhibitions_total_amount,

        -- доли только по размеченным тратам
        sum(transaction_count) AS all_transaction_count,
        sum(total_amount)      AS all_total_amount

    FROM products_stats
    GROUP BY epk
),

flags_agg AS (
    SELECT
        epk,
        max(flag_buy_3month_ago) AS flag_buy_3month_ago,
        max(flag_buy_2month_ago) AS flag_buy_2month_ago,
        max(flag_buy_1month_ago) AS flag_buy_1month_ago
    FROM tx_filtered
    GROUP BY epk
),

score_names AS (
    SELECT
        p.epk,

        coalesce(p.autoRent_transaction_count     / nullif(p.all_transaction_count,0), 0) AS autoRent_cnt_share,
        coalesce(p.autoRent_total_amount          / nullif(p.all_total_amount,0), 0)     AS autoRent_amt_share,

        coalesce(p.excursion_transaction_count    / nullif(p.all_transaction_count,0), 0) AS excursion_cnt_share,
        coalesce(p.excursion_total_amount         / nullif(p.all_total_amount,0), 0)      AS excursion_amt_share,

        coalesce(p.tourOrHotel_transaction_count  / nullif(p.all_transaction_count,0), 0) AS tourOrHotel_cnt_share,
        coalesce(p.tourOrHotel_total_amount       / nullif(p.all_total_amount,0), 0)      AS tourOrHotel_amt_share,

        coalesce(p.eventForeign_transaction_count / nullif(p.all_transaction_count,0), 0) AS eventForeign_cnt_share,
        coalesce(p.eventForeign_total_amount      / nullif(p.all_total_amount,0), 0)      AS eventForeign_amt_share,

        coalesce(p.spa_transaction_count          / nullif(p.all_transaction_count,0), 0) AS spa_cnt_share,
        coalesce(p.spa_total_amount               / nullif(p.all_total_amount,0), 0)      AS spa_amt_share,

        coalesce(p.event_transaction_count        / nullif(p.all_transaction_count,0), 0) AS event_cnt_share,
        coalesce(p.event_total_amount             / nullif(p.all_total_amount,0), 0)      AS event_amt_share,

        coalesce(p.training_transaction_count     / nullif(p.all_transaction_count,0), 0) AS training_cnt_share,
        coalesce(p.training_total_amount          / nullif(p.all_total_amount,0), 0)      AS training_amt_share,

        coalesce(p.lecture_transaction_count      / nullif(p.all_transaction_count,0), 0) AS lecture_cnt_share,
        coalesce(p.lecture_total_amount           / nullif(p.all_total_amount,0), 0)      AS lecture_amt_share,

        coalesce(p.paidTickets_transaction_count  / nullif(p.all_transaction_count,0), 0) AS paidTickets_cnt_share,
        coalesce(p.paidTickets_total_amount       / nullif(p.all_total_amount,0), 0)      AS paidTickets_amt_share,

        coalesce(p.concerts_transaction_count     / nullif(p.all_transaction_count,0), 0) AS concerts_cnt_share,
        coalesce(p.concerts_total_amount          / nullif(p.all_total_amount,0), 0)      AS concerts_amt_share,

        coalesce(p.exhibitions_transaction_count  / nullif(p.all_transaction_count,0), 0) AS exhibitions_cnt_share,
        coalesce(p.exhibitions_total_amount       / nullif(p.all_total_amount,0), 0)      AS exhibitions_amt_share,

        CASE
            WHEN (f.flag_buy_3month_ago=1 AND f.flag_buy_2month_ago=1)
              OR (f.flag_buy_3month_ago=1 AND f.flag_buy_1month_ago=1)
              OR (f.flag_buy_2month_ago=1 AND f.flag_buy_1month_ago=1)
            THEN 1 ELSE 0 END AS regular_flag

    FROM pivoted_stats p
    LEFT JOIN flags_agg f USING(epk)
),

scores AS (
    SELECT
        epk,
        autoRent_amt_share     *0.5 + autoRent_cnt_share     *0.3 + regular_flag*0.2 AS autoRent_score,
        excursion_amt_share    *0.5 + excursion_cnt_share    *0.3 + regular_flag*0.2 AS excursion_score,
        tourOrHotel_amt_share  *0.5 + tourOrHotel_cnt_share  *0.3 + regular_flag*0.2 AS tourOrHotel_score,
        eventForeign_amt_share *0.5 + eventForeign_cnt_share *0.3 + regular_flag*0.2 AS eventForeign_score,
        spa_amt_share          *0.5 + spa_cnt_share          *0.3 + regular_flag*0.2 AS spa_score,
        event_amt_share        *0.5 + event_cnt_share        *0.3 + regular_flag*0.2 AS event_score,
        training_amt_share     *0.5 + training_cnt_share     *0.3 + regular_flag*0.2 AS training_score,
        lecture_amt_share      *0.5 + lecture_cnt_share      *0.3 + regular_flag*0.2 AS lecture_score,
        paidTickets_amt_share  *0.5 + paidTickets_cnt_share  *0.3 + regular_flag*0.2 AS paidTickets_score,
        concerts_amt_share     *0.5 + concerts_cnt_share     *0.3 + regular_flag*0.2 AS concerts_score,
        exhibitions_amt_share  *0.5 + exhibitions_cnt_share  *0.3 + regular_flag*0.2 AS exhibitions_score
    FROM score_names
),

base_scored AS (
    SELECT
        epk,
        stack(11,
            'autoRent',     autoRent_score,
            'excursion',    excursion_score,
            'tourOrHotel',  tourOrHotel_score,
            'eventForeign', eventForeign_score,
            'spa',          spa_score,
            'event',        event_score,
            'training',     training_score,
            'lecture',      lecture_score,
            'paidTickets',  paidTickets_score,
            'concerts',     concerts_score,
            'exhibitions',  exhibitions_score
        ) as (category, base_score)
    FROM scores
),

ranked_base AS (
    SELECT
        epk,
        category,
        base_score,
        row_number() over (partition by epk order by base_score desc, category) as rn
    FROM base_scored
),

top2 AS (
    SELECT
        epk,
        max(CASE WHEN rn=1 THEN base_score END) AS top1_base_score,
        max(CASE WHEN rn=2 THEN base_score END) AS top2_base_score
    FROM ranked_base
    WHERE rn <= 2
    GROUP BY epk
),

need_tags AS (
    SELECT
        epk,
        CASE
            WHEN top1_base_score IS NULL THEN 1
            WHEN top1_base_score = 0 THEN 1
            WHEN top2_base_score IS NULL THEN 0
            WHEN ( (top1_base_score - top2_base_score) * 100.0 / nullif(top1_base_score,0) ) < 10 THEN 1
            ELSE 0
        END AS need_tags_flag
    FROM top2
),

-- ТЕГИ: берём только nsi_id -> category_name
client_tags AS (
    SELECT
        epk_id as epk,
        cast(nsi_id as string) as tag_id,
        CASE
            WHEN cast(nsi_id as string) = '22631' THEN 'autoRent'
            WHEN cast(nsi_id as string) IN ('10582','10583') THEN 'tourOrHotel'
            WHEN cast(nsi_id as string) = '39602' THEN 'eventForeign'
            WHEN cast(nsi_id as string) IN ('10138','33532','10175','10144','20082','20116') THEN 'event'
            WHEN cast(nsi_id as string) = '33539' THEN 'lecture'
            WHEN cast(nsi_id as string) IN (
                '10067','10095','10564','21929','21928',
                '33936','21936','21930','21937','21934'
            ) THEN 'exhibitions'
            ELSE 'all'
        END AS category_name
    FROM arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_product a
    left join {tags} b on a.epk = b.epk_id 
    WHERE row_actual_to_dt = '9999-12-31'
      AND is_deleted = 0
),

scored_with_tags AS (
    SELECT
        b.epk,
        b.category,
        b.base_score,
        n.need_tags_flag,
        t2.top1_base_score,

        CASE
            WHEN n.need_tags_flag = 1 THEN
                least(
                    count(distinct ct.tag_id) * 0.05,
                    0.30 * CASE
                        WHEN b.base_score > 0 THEN b.base_score
                        WHEN coalesce(t2.top1_base_score,0) > 0 THEN t2.top1_base_score
                        ELSE 1.0
                    END
                )
            ELSE 0
        END AS tag_bonus

    FROM base_scored b
    LEFT JOIN need_tags n
        ON b.epk = n.epk
    LEFT JOIN top2 t2
        ON b.epk = t2.epk
    LEFT JOIN client_tags ct
        ON b.epk = ct.epk
       AND b.category = ct.category_name
    GROUP BY
        b.epk, b.category, b.base_score, n.need_tags_flag, t2.top1_base_score
),

final_scored AS (
    SELECT
        epk,
        category,
        base_score,
        (base_score + tag_bonus) AS final_score
    FROM scored_with_tags
),

ranked_final AS (
    SELECT
        epk,
        category,
        final_score,
        row_number() over (partition by epk order by final_score desc, category) as rn
    FROM final_scored
),

result AS (
    SELECT
        epk,
        max(CASE WHEN rn=1 THEN category END) AS top_1_lifestyleGeneral_advantages
    FROM ranked_final
    WHERE rn=1
    GROUP BY epk
)

SELECT distinct * FROM arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_product a
left join result using(epk)
""")

lifestyleGeneral_advantages.createOrReplaceTempView("lifestyleGeneral_advantages")

In [25]:
# I can't show you how the Offer Type is calculated because I'm under an NDA.
spark.sql(f'''
with businessLounges_advan as (
select distinct epk, max(case when mcc_code in ('45882', '4722', '5814', '72999', '73999') 
        then 'x' else '' end) as businessLounges_advantages,
        max(case when mcc_code in ('4121', '3990', '4789', '4131', '5999', '4816', '73992', '7299', '4111') 
        then 'x' else '' end) as taxiRestaurantRefund_offerType
from arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_product a
left join {pos} b on a.epk = b.epk_id
where mcc_code is not null
and ecom_fl = 0
and is_transaction = 1
and day_part between last_day(date('{date_now_free}') - interval 1 month) and date('{date_now_free}')
group by epk
),

new_products as (
# I can not show this part where i create offer type, because i am under NDA
)

select distinct * from arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_product
left join new_products using(epk)
''').write.saveAsTable('arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products', mode='overwrite')

# Экосистема продукты

In [26]:
model_scores = spark.read.parquet('hdfs://hdfsgw/arnsdpmlexp__bpm_model_romashka-CUSTOM_ROZN_COREML_EXT_SCORES-DM_ROMASHKA_TRNSF_MODEL_SCORES_DESCRIPTION/data/custom/rozn/coreml_ext_scores/pa/dm_romashka_trnsf_model_scores_description')
model_scores.createOrReplaceTempView('need_id')

In [27]:
spark.sql(f'''
                with need_id_and_situation as (
                    select need_ID, life_situation, model_id, need
                    from need_id
                    where model_name like '%d14' and need_ID in ('10.18.5', '10.18.6', '1.51.3', '1.51.4', '6.45.10', '1.9.1', '1.9.2', '10.19.7',
                        '10.19.10', '8.8.4', '8.8.5', '8.16.15', '8.16.1', '8.59.6', '8.59.5',
                        '7.2.1', '7.2.2', '7.12.1', '7.12.4', '7.50.1', '7.50.3',
                        '10.18.2', '6.45.11', '6.44.1', '6.44.2', '1.17.2', '1.17.6',
                        '2.25.3', '2.25.4', '2.30.2', '2.30.3', '2.1.5', '2.30.5', '1.6.1', '3.4.4', '3.15.5',
                        '1.54.4', '8.56.3', '8.62.1', '5.37.2', '5.38.4',
                        '8.8.7', '4.31.1', '3.26.4', '3.5.1',
                        '8.8.2', '2.22.4', '2.27.1', '5.40.2', '6.42.1')
                ),
                
                epk_and_situation as (
                    select epk_id as epk, need_ID, life_situation, need, raw_score
                    from need_id_and_situation
                    inner join {romashka_hist} using(model_id)
                    where scoring_dt between date('{date_now_free}' - interval 2 weeks) and '{date_now_free}'
                ),
                
                product_mapping AS (
                  SELECT
                    epk,
                    need_ID,
                    life_situation,
                    need,
                    raw_score,
                    approval_level,
                    CASE
                      WHEN need_ID IN ('10.18.5', '10.18.6', '1.51.3', '1.51.4', '6.45.10', '1.9.1', '1.9.2', '10.19.7',
                                      '10.19.10', '8.8.4', '8.8.5', '8.16.15', '8.16.1', '8.59.6', '8.59.5',
                                      '7.2.1', '7.2.2', '7.12.1', '7.12.4', '7.50.1', '7.50.3', '10.18.2',
                                      '6.45.10', '6.45.11', '6.44.1', '6.44.2') THEN 'cooper'
                      WHEN need_ID IN ('10.18.5', '10.18.6', '10.18.2', '1.17.2', '1.17.6', '10.17.6', '1.17.2', '1.17.6') THEN 'sberHealth'
                      WHEN need_ID IN ('2.25.3', '2.25.4') THEN 'citydrive'
                      WHEN need_ID IN ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5', '1.6.1', '3.4.4', '3.15.5') THEN 'sberPravo'
                      WHEN need_ID IN ('1.51.3', '1.54.4', '6.45.10', '6.44.1', '6.44.2', '10.19.7', '10.19.10',
                                      '8.8.4', '8.8.5', '8.56.3', '8.62.1', '5.37.2', '5.38.4', '7.12.1', '7.12.4',
                                      '7.50.1', '7.50.3', '7.2.1') THEN 'samokat'
                      WHEN need_ID IN ('1.51.3', '8.8.4', '8.8.5', '5.37.2', '5.38.4', '8.56.3') THEN 'okko'
                      WHEN need_ID IN ('1.51.3', '8.8.4', '8.8.5') THEN 'zvuk'
                      WHEN need_ID IN ('1.17.2', '1.17.6', '3.4.4', '4.31.1', '3.26.4', '3.15.5', '3.5.1') THEN 'zls'
                      WHEN need_ID IN ('8.8.4', '8.8.7') THEN 'vzr'
                      WHEN need_ID IN ('1.17.6', '1.17.2', '8.8.4', '8.8.5') THEN 'mriya'
                      WHEN need_ID IN ('8.8.4', '8.8.5', '3.26.4') THEN 'sberMobile'
                      WHEN need_ID IN ('8.8.4', '8.8.5') THEN 'otello'
                      WHEN need_ID IN ('8.56.3') THEN 'afisha'
                      WHEN need_ID IN ('8.8.4', '8.8.5') THEN 'manzherok'
                      WHEN need_ID IN ('2.22.4', '2.30.3', '2.30.5', '1.51.3', '1.54.4', '6.45.10', '6.44.1', '6.44.2',
                                      '10.19.10', '8.8.4', '8.8.7', '8.16.15', '8.16.1', '8.59.6', '8.59.5', '8.56.3',
                                      '3.4.4', '3.15.5', '3.26.4') THEN 'lifestyleGeneral'
                      WHEN need_ID IN ('2.22.4', '2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4', '2.27.1', '1.51.4',
                                      '1.51.3', '6.45.10', '6.45.11', '6.44.1', '6.44.2', '10.19.7', '10.19.10', '5.40.2',
                                      '8.8.4', '8.8.5', '8.56.3', '6.42.1', '3.4.4', '3.15.5', '3.26.4') THEN 'lifestyleGeneral'
                      ELSE ''
                    END as product_name
                  FROM arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products
                  left join epk_and_situation using(epk)
                ),
                
                -- Сначала убираем дубликаты по продуктам (берем продукт с максимальным скором)
                unique_products AS (
                  SELECT 
                    epk,
                    need_ID,
                    product_name,
                    life_situation,
                    need,
                    raw_score,
                    approval_level,
                    ROW_NUMBER() OVER (
                      PARTITION BY epk, product_name
                      ORDER BY raw_score DESC
                    ) as product_rank
                  FROM product_mapping
                  WHERE product_name IS NOT NULL and product_name != ''
                ),
                
                filtered_products AS (
                  SELECT 
                    epk,
                    need_ID,
                    product_name,
                    life_situation,
                    need,
                    raw_score,
                    approval_level
                  FROM unique_products
                  WHERE product_rank = 1  -- Берем только первую запись для каждого уникального продукт
                ),
                
                -- Затем убираем дубликаты по жизненным ситуациям (берем ситуацию с максимальным скором)
                unique_situations AS (
                  SELECT 
                    epk,
                    need_ID,
                    product_name,
                    life_situation,
                    need,
                    raw_score,
                    approval_level,
                    ROW_NUMBER() OVER (
                      PARTITION BY epk, life_situation
                      ORDER BY raw_score DESC
                    ) as situation_rank
                  FROM filtered_products
                ),
                
                filtered_situations AS (
                  SELECT 
                    epk,
                    need_ID,
                    product_name,
                    life_situation,
                    need,
                    raw_score,
                    approval_level
                  FROM unique_situations
                  WHERE situation_rank = 1  -- Берем только первую запись для каждой уникальной жизненной ситуации
                   and not (product_name = 'lifestyleGeneral' and approval_level < 4)
                   and not (product_name = 'vzr' and approval_level > 1)
                ),
                
                -- Теперь ранжируем уникальные комбинации по скору
                ranked_combinations AS (
                  SELECT
                    epk,
                    need_ID,
                    product_name,
                    life_situation,
                    need,
                    raw_score,
                    approval_level,
                    ROW_NUMBER() OVER (
                        PARTITION BY epk ORDER BY raw_score DESC) as rank_num
                  FROM filtered_situations
                ),
                
                mid_answer as (
                    SELECT
                      distinct epk,
                      -- Топ-1 продукт
                      MAX(CASE WHEN rank_num = 1 THEN product_name END) as top1_product,
                      MAX(CASE WHEN rank_num = 1 THEN need_ID END) as top1_need_ID,
                      MAX(CASE WHEN rank_num = 1 THEN raw_score END) as top1_score,
                      -- Топ-2 продукт
                      MAX(CASE WHEN rank_num = 2 THEN product_name END) as top2_product,
                      MAX(CASE WHEN rank_num = 2 THEN need_ID END) as top2_need_ID,
                      MAX(CASE WHEN rank_num = 2 THEN raw_score END) as top2_score,
                      -- Топ-3 продукт
                      MAX(CASE WHEN rank_num = 3 THEN product_name END) as top3_product,
                      MAX(CASE WHEN rank_num = 3 THEN need_ID END) as top3_need_ID,
                      MAX(CASE WHEN rank_num = 3 THEN raw_score END) as top3_score,
                      -- Топ-4 продукт
                      MAX(CASE WHEN rank_num = 4 THEN product_name END) as top4_product,
                      MAX(CASE WHEN rank_num = 4 THEN need_ID END) as top4_need_ID,
                      MAX(CASE WHEN rank_num = 4 THEN raw_score END) as top4_score,
                      -- Топ-5 продукт
                      MAX(CASE WHEN rank_num = 5 THEN product_name END) as top5_product,
                      MAX(CASE WHEN rank_num = 5 THEN need_ID END) as top5_need_ID,
                      MAX(CASE WHEN rank_num = 5 THEN raw_score END) as top5_score,
                      -- Топ-6 продукт
                      MAX(CASE WHEN rank_num = 6 THEN product_name END) as top6_product,
                      MAX(CASE WHEN rank_num = 6 THEN need_ID END) as top6_need_ID,
                      MAX(CASE WHEN rank_num = 6 THEN raw_score END) as top6_score,
                      -- Топ-7 продукт
                      MAX(CASE WHEN rank_num = 7 THEN product_name END) as top7_product,
                      MAX(CASE WHEN rank_num = 7 THEN need_ID END) as top7_need_ID,
                      MAX(CASE WHEN rank_num = 7 THEN raw_score END) as top7_score
                    FROM ranked_combinations
                    WHERE rank_num <= 7
                    GROUP BY epk
                ),
                
                answer as (
                    select distinct epk, 
                        case when top1_product is null or top1_product = '' then 'plug'
                        else top1_product end as top_1,
                        'ecosystem' as top1_type,
                        case when top1_score is null or top1_score = '' then 0
                        else top1_score end as top1_score,
                        '' as top1_offer_type, '' as top1_advantages, 0 as top1_is_recommended, 0 as top1_used_by_client,
                            case 
                                when top1_product = 'cooper' and top1_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets'
                                when top1_product = 'cooper' and top1_need_ID in ('1.51.3', '1.51.4') then 'sport' 
                                when top1_product = 'cooper' and top1_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty' 
                                when top1_product = 'cooper' and top1_need_ID in ('1.9.1', '1.9.2') then 'healthAndBeauty' 
                                when top1_product = 'cooper' and top1_need_ID in ('10.19.7', '10.19.10') then 'kids'
                                when top1_product = 'cooper' and top1_need_ID in ('8.8.4', '8.8.5') then 'trips'
                                when top1_product = 'cooper' and top1_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'entertainmentAndHobbies' 
                                when top1_product = 'cooper' and top1_need_ID in ('7.2.1', '7.2.2') then 'other' 
                                when top1_product = 'cooper' and top1_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other'
                                when top1_product = 'cooper' and top1_need_ID in ('7.2.1', '7.2.2') then 'realEstate' 
                                when top1_product = 'cooper' and top1_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'realEstate'
                                when top1_product = 'cooper' and top1_need_ID in ('7.2.1') then 'default' 
                                
                            
                                
                                
                                when top1_product = 'sberHealth' and top1_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets' 
                                when top1_product = 'sberHealth' and top1_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty' 
                                when top1_product = 'sberHealth' and top1_need_ID in ('10.17.6') then 'kids' 
                                when top1_product = 'sberHealth' and top1_need_ID in ('1.17.2', '1.17.6') then 'default' 
                                
                                
                                when top1_product = 'citydrive' and top1_need_ID in ('2.25.3', '2.25.4') then 'auto'
                                
                                
                                
                                when top1_product = 'sberPravo' and top1_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'auto' 
                                when top1_product = 'sberPravo' and top1_need_ID in ('1.6.1') then 'other'
                                when top1_product = 'sberPravo' and top1_need_ID in ('3.4.4', '3.15.5') then 'realEstate' 
                                when top1_product = 'sberPravo' and top1_need_ID in ('2.25.3', '2.25.4') then 'auto' 
                                
                                
                                
                                when top1_product = 'samokat' and top1_need_ID in ('1.51.3', '1.54.4') then 'sport' 
                                when top1_product = 'samokat' and top1_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty' 
                                when top1_product = 'samokat' and top1_need_ID in ('10.19.7', '10.19.10') then 'kids' 
                                when top1_product = 'samokat' and top1_need_ID in ('8.8.4', '8.8.5') then 'trips' 
                                when top1_product = 'samokat' and top1_need_ID in ('8.56.3') then 'entertainmentAndHobbies'
                                when top1_product = 'samokat' and top1_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'entertainmentAndHobbies' 
                                when top1_product = 'samokat' and top1_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other' 
                                when top1_product = 'samokat' and top1_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'realEstate' 
                                when top1_product = 'samokat' and top1_need_ID in ('7.2.1') then 'default' 
                                
                                
                                
                                when top1_product = 'okko' and top1_need_ID in ('1.51.3') then 'sport' 
                                when top1_product = 'okko' and top1_need_ID in ('8.8.4', '8.8.5') then 'trips' 
                                when top1_product = 'okko' and top1_need_ID in ('5.37.2', '5.38.4') then 'entertainmentAndHobbies' 
                                when top1_product = 'okko' and top1_need_ID in ('8.56.3') then 'entertainmentAndHobbies' 
                                
                                
                                
                                when top1_product = 'zvuk' and top1_need_ID in ('1.51.3') then 'sport' 
                                when top1_product = 'zvuk' and top1_need_ID in ('8.8.4', '8.8.5') then 'trips' 
                                
                                
                                
                                when top1_product = 'zls' and top1_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty' 
                                when top1_product = 'zls' and top1_need_ID in ('1.17.2', '1.17.6') then 'kids' 
                                when top1_product = 'zls' and top1_need_ID in ('3.4.4') then 'trips'  
                                when top1_product = 'zls' and top1_need_ID in ('4.31.1') then 'other'
                                when top1_product = 'zls' and top1_need_ID in ('3.26.4') then 'realEstate' 
                                when top1_product = 'zls' and top1_need_ID in ('3.4.4') then 'realEstate' 
                                when top1_product = 'zls' and top1_need_ID in ('3.15.5', '3.5.1') then 'realEstate' 
                                
                                
                                when top1_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'trips'
                                
                                
                                
                                when top1_product = 'mriya' and top1_need_ID in ('1.17.6', '1.17.2') then 'healthAndBeauty' 
                                when top1_product = 'mriya' and top1_need_ID in ('8.8.4', '8.8.5') then 'trips' 
                                
                                
                                
                                when top1_product = 'sberMobile' and top1_need_ID in ('8.8.4', '8.8.5') then 'trips' 
                                when top1_product = 'sberMobile' and top1_need_ID in ('3.26.4') then 'realEstate' 
                                
                                
                                
                                when top1_product = 'otello' and top1_need_ID in ('8.8.4', '8.8.5') then 'trips'
                                when top1_product = 'afisha' and approval_level >= 3 and top1_need_ID in ('8.56.3') then 'entertaimentAndHobbies' 
                                when top1_product = 'manzherok' and approval_level >= 4 and top1_need_ID in ('8.8.4', '8.8.5') then 'trips' 
                                
                                
                                
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('2.22.4') then 'auto' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('2.30.3', '2.30.5') then 'auto' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('1.51.3', '1.54.4') then 'sport' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('10.19.10') then 'kids'
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('8.8.4', '8.8.7') then 'trips'
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'entertaimentAndHobbies' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('8.59.5') then 'entertaimentAndHobbies'
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('8.59.3') then 'entertaimentAndHobbies' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'realEstate' 
                                
                                
                                
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('2.22.4') then 'auto'
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('2.30.3', '2.30.5') then 'auto' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'auto' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('2.27.1') then 'auto' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('1.51.4', '1.51.3') then 'sport'
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('10.19.7', '10.19.10') then 'kids'
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('5.40.2') then 'kids' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('8.8.4', '8.8.5') then 'trips' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'realEstate'
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('8.56.3') then 'entertaimentAndHobbies' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('6.42.1') then 'entertaimentAndHobbies'
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                                else '' end as top1_lifestyle_situation,
                                
                                
                            case 
                                when top1_product = 'cooper' and top1_need_ID in ('10.18.5', '10.18.6') then 'default'
                                when top1_product = 'cooper' and top1_need_ID in ('10.18.2') then 'health'
                                when top1_product = 'cooper' and top1_need_ID in ('1.51.3', '1.51.4') then 'default' 
                                when top1_product = 'cooper' and top1_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'beauty' 
                                when top1_product = 'cooper' and top1_need_ID in ('1.9.1', '1.9.2') then 'pharmacy' 
                                when top1_product = 'cooper' and top1_need_ID in ('10.19.7', '10.19.10') then 'kids0To6' 
                                when top1_product = 'cooper' and top1_need_ID in ('8.8.4', '8.8.5') then 'default' 
                                when top1_product = 'cooper' and top1_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'specialShops' 
                                when top1_product = 'cooper' and top1_need_ID in ('7.2.1', '7.2.2') then 'shopsNotFromList' 
                                when top1_product = 'cooper' and top1_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants' 
                                when top1_product = 'cooper' and top1_need_ID in ('7.2.1', '7.2.2') then 'supermarkets' 
                                when top1_product = 'cooper' and top1_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'shoppingForHouse' 
                                when top1_product = 'cooper' and top1_need_ID in ('7.2.1') then 'default' 
                            
                                
                                
                                
                                when top1_product = 'sberHealth' and top1_need_ID in ('1.9.1', '1.9.2') then 'default' 
                                when top1_product = 'sberHealth' and top1_need_ID in ('1.17.2', '1.17.6') then 'laboratory' 
                                when top1_product = 'sberHealth' and top1_need_ID in ('10.17.6') then 'kids0To14' 
                                when top1_product = 'sberHealth' and top1_need_ID in ('1.17.2', '1.17.6') then 'default' 
                                
                                
                                when top1_product = 'citydrive' and top1_need_ID in ('2.25.3', '2.25.4') then 'carsharing' 
                                
                                
                                when top1_product = 'sberPravo' and top1_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'fines' 
                                when top1_product = 'sberPravo' and top1_need_ID in ('1.6.1') then 'taxes'
                                when top1_product = 'sberPravo' and top1_need_ID in ('3.4.4', '3.15.5') then 'moreThan2Estates' 
                                when top1_product = 'sberPravo' and top1_need_ID in ('2.25.3', '2.25.4') then 'carsharing' 
                                
                                 
                                when top1_product = 'samokat' and top1_need_ID in ('1.51.3', '1.54.4') then 'default' 
                                when top1_product = 'samokat' and top1_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default' 
                                when top1_product = 'samokat' and top1_need_ID in ('10.19.7', '10.19.10') then 'dids0To6' 
                                when top1_product = 'samokat' and top1_need_ID in ('8.8.4', '8.8.5') then 'default' 
                                when top1_product = 'samokat' and top1_need_ID in ('8.56.3') then 'cinema' 
                                when top1_product = 'samokat' and top1_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'books' 
                                when top1_product = 'samokat' and top1_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants' 
                                when top1_product = 'samokat' and top1_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'orderingFood' 
                                when top1_product = 'samokat' and top1_need_ID in ('7.2.1') then 'default' 
                                
                                
                                when top1_product = 'okko' and top1_need_ID in ('1.51.3') then 'default' 
                                when top1_product = 'okko' and top1_need_ID in ('8.8.4', '8.8.5') then 'default' 
                                when top1_product = 'okko' and top1_need_ID in ('5.37.2', '5.38.4') then 'selfeEducation' 
                                when top1_product = 'okko' and top1_need_ID in ('8.56.3') then 'cinema' 
                                
                                
                                when top1_product = 'zvuk' and top1_need_ID in ('1.51.3') then 'default' 
                                when top1_product = 'zvuk' and top1_need_ID in ('8.8.4', '8.8.5') then 'default' 
                                
                                
                                when top1_product = 'zls' and top1_need_ID in ('1.17.2', '1.17.6') then 'healthInsurance' 
                                when top1_product = 'zls' and top1_need_ID in ('1.17.2', '1.17.6') then 'insuranceKids1To14'  
                                when top1_product = 'zls' and top1_need_ID in ('3.4.4') then 'realEstate' 
                                when top1_product = 'zls' and top1_need_ID in ('4.31.1') then 'financeInsurance' 
                                when top1_product = 'zls' and top1_need_ID in ('3.26.4') then 'houseInsurance' 
                                when top1_product = 'zls' and top1_need_ID in ('3.4.4') then 'realEstateInsurance' 
                                when top1_product = 'zls' and top1_need_ID in ('3.15.5', '3.5.1') then 'houseRepairInsurance' 
                                
                                
                                when top1_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'vzr1Level'
                                
                                
                                when top1_product = 'mriya' and top1_need_ID in ('1.17.6', '1.17.2') then 'default' 
                                when top1_product = 'mriya' and top1_need_ID in ('8.8.4', '8.8.5') then 'default' 
                                
                                
                                when top1_product = 'sberMobile' and top1_need_ID in ('8.8.4', '8.8.5') then 'default' 
                                when top1_product = 'sberMobile' and top1_need_ID in ('3.26.4') then 'newSbermobileSubscriber' 
                                
                                
                                when top1_product = 'otello' and top1_need_ID in ('8.8.4', '8.8.5') then 'hasBonuses' 
                                
                                
                                when top1_product = 'afisha' and approval_level >= 3 and top1_need_ID in ('8.56.3') then 'tickets' 
                                
                                when top1_product = 'manzherok' and approval_level >= 4 and top1_need_ID in ('8.8.4', '8.8.5') then 'default' 
                                
                                
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('2.22.4') then 'interestedInBuyingCar' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('2.30.3', '2.30.5') then 'carRepair' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('1.51.3', '1.54.4') then 'default' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('10.19.10') then 'kids0To6' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('8.8.4', '8.8.7') then 'default' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'specialShops' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('8.59.5') then 'sporingEvents' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('8.59.3') then 'concertsEvents' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 4 and top1_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'householdServices' 
                                
                                
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('2.22.4') then 'interestedInBuyingCar' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('2.30.3', '2.30.5') then 'carRepair' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'carOwners' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('2.27.1') then 'taxiTransfer' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('1.51.4', '1.51.3') then 'default' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'spaBeautySalons' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('10.19.7', '10.19.10') then 'kids0To6' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('5.40.2') then 'kids14To18' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('8.8.4', '8.8.5') then 'default' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'householdServices' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('8.56.3') then 'sportingEvents' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('8.56.3') then 'concertsEvents' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('6.42.1') then 'shoppingForClothes' 
                                when top1_product = 'lifestyleGeneral' and approval_level >= 6 and top1_need_ID in ('8.56.3') then 'concierge'
                                else '' end as top1_lifestyle_sub_situation,
                                
                        CASE 
                                WHEN top1_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_1, 0))
                                WHEN top1_product = 'okko' THEN round(coalesce(okko_without_premier_level_1, 0))
                                else 0
                                END as top1_benefit_rubles_lvl1,
                                
                                case 
                                WHEN top1_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_1))
                                WHEN top1_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_1))
                                else 0 
                                end as top1_benefit_bonuses_lvl1,
                
                                CASE 
                                WHEN top1_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_2))
                                WHEN top1_product = 'okko' THEN round(coalesce(okko_without_premier_level_2))
                                else 0
                                END as top1_benefit_rubles_lvl2,
                                
                                case 
                                WHEN top1_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_2))
                                WHEN top1_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_2))
                                else 0 
                                end as top1_benefit_bonuses_lvl2,
                
                                CASE 
                                WHEN top1_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_3))
                                WHEN top1_product = 'okko' THEN round(coalesce(okko_without_premier_level_3))
                                else 0
                                END as top1_benefit_rubles_lvl3,
                                
                                case 
                                WHEN top1_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_3))
                                WHEN top1_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_3))
                                else 0
                                end as top1_benefit_bonuses_lvl3,
                
                                CASE 
                                WHEN top1_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_4_profit, 0))
                                WHEN top1_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_4_profit, 0))
                                else 0
                                END as top1_benefit_rubles_lvl4,
                                
                                case 
                                WHEN top1_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_4, 0))
                                WHEN top1_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_4, 0))
                                else 0 
                                end as top1_benefit_bonuses_lvl4,
                
                                CASE 
                                WHEN top1_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top1_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top1_benefit_rubles_lvl5,
                                
                                case
                                WHEN top1_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top1_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top1_benefit_bonuses_lvl5,
                
                                CASE 
                                WHEN top1_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top1_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top1_benefit_rubles_lvl6,
                                
                                case
                                WHEN top1_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top1_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top1_benefit_bonuses_lvl6,
                        
                        
                        
                        
                        case when top2_product is null or top2_product = '' then 'plug'
                        else top2_product end as top_2,
                        'ecosystem' as top2_type,
                        case when top2_score is null or top2_score = '' then 0
                        else top2_score end as top2_score,
                        '' as top2_offer_type, '' as top2_advantages, 0 as top2_is_recommended, 0 as top2_used_by_client,
                            case
                    when top2_product = 'cooper' and top2_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets'
                    when top2_product = 'cooper' and top2_need_ID in ('1.51.3', '1.51.4') then 'sport' 
                    when top2_product = 'cooper' and top2_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty' 
                    when top2_product = 'cooper' and top2_need_ID in ('1.9.1', '1.9.2') then 'healthAndBeauty' 
                    when top2_product = 'cooper' and top2_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top2_product = 'cooper' and top2_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top2_product = 'cooper' and top2_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'entertainmentAndHobbies' 
                    when top2_product = 'cooper' and top2_need_ID in ('7.2.1', '7.2.2') then 'other' 
                    when top2_product = 'cooper' and top2_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other'
                    when top2_product = 'cooper' and top2_need_ID in ('7.2.1', '7.2.2') then 'realEstate' 
                    when top2_product = 'cooper' and top2_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'realEstate'
                    when top2_product = 'cooper' and top2_need_ID in ('7.2.1') then 'default'
                
                
                    when top2_product = 'sberHealth' and top2_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets'
                    when top2_product = 'sberHealth' and top2_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty'
                    when top2_product = 'sberHealth' and top2_need_ID in ('10.17.6') then 'kids'
                    when top2_product = 'sberHealth' and top2_need_ID in ('1.17.2', '1.17.6') then 'default'
                
                    when top2_product = 'citydrive' and top2_need_ID in ('2.25.3', '2.25.4') then 'auto'
                
                    when top2_product = 'sberPravo' and top2_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'auto'
                    when top2_product = 'sberPravo' and top2_need_ID in ('1.6.1') then 'other'
                    when top2_product = 'sberPravo' and top2_need_ID in ('3.4.4', '3.15.5') then 'realEstate'
                    when top2_product = 'sberPravo' and top2_need_ID in ('2.25.3', '2.25.4') then 'auto'
                
                    when top2_product = 'samokat' and top2_need_ID in ('1.51.3', '1.54.4') then 'sport'
                    when top2_product = 'samokat' and top2_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top2_product = 'samokat' and top2_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top2_product = 'samokat' and top2_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top2_product = 'samokat' and top2_need_ID in ('8.56.3') then 'entertainmentAndHobbies'
                    when top2_product = 'samokat' and top2_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'entertainmentAndHobbies'
                    when top2_product = 'samokat' and top2_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other'
                    when top2_product = 'samokat' and top2_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'realEstate'
                    when top2_product = 'samokat' and top2_need_ID in ('7.2.1') then 'default'
                
                    when top2_product = 'okko' and top2_need_ID in ('1.51.3') then 'sport'
                    when top2_product = 'okko' and top2_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top2_product = 'okko' and top2_need_ID in ('5.37.2', '5.38.4') then 'entertainmentAndHobbies'
                    when top2_product = 'okko' and top2_need_ID in ('8.56.3') then 'entertainmentAndHobbies'
                
                    when top2_product = 'zvuk' and top2_need_ID in ('1.51.3') then 'sport'
                    when top2_product = 'zvuk' and top2_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top2_product = 'zls' and top2_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty'
                    when top2_product = 'zls' and top2_need_ID in ('1.17.2', '1.17.6') then 'kids'
                    when top2_product = 'zls' and top2_need_ID in ('3.4.4') then 'trips'
                    when top2_product = 'zls' and top2_need_ID in ('4.31.1') then 'other'
                    when top2_product = 'zls' and top2_need_ID in ('3.26.4') then 'realEstate'
                    when top2_product = 'zls' and top2_need_ID in ('3.4.4') then 'realEstate'
                    when top2_product = 'zls' and top2_need_ID in ('3.15.5', '3.5.1') then 'realEstate'
                    
                    when top2_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'trips'
                
                    when top2_product = 'mriya' and top2_need_ID in ('1.17.6', '1.17.2') then 'healthAndBeauty'
                    when top2_product = 'mriya' and top2_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top2_product = 'sberMobile' and top2_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top2_product = 'sberMobile' and top2_need_ID in ('3.26.4') then 'realEstate'
                
                    when top2_product = 'otello' and top2_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top2_product = 'afisha' and approval_level >= 3 and top2_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top2_product = 'manzherok' and approval_level >= 4 and top2_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('2.22.4') then 'auto'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('2.30.3', '2.30.5') then 'auto'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('1.51.3', '1.54.4') then 'sport'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('10.19.10') then 'kids'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('8.8.4', '8.8.7') then 'trips'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'entertaimentAndHobbies'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('8.59.5') then 'entertaimentAndHobbies'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('8.59.3') then 'entertaimentAndHobbies'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'realEstate'
                
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('2.22.4') then 'auto'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('2.30.3', '2.30.5') then 'auto'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'auto'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('2.27.1') then 'auto'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('1.51.4', '1.51.3') then 'sport'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('5.40.2') then 'kids'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'realEstate'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('6.42.1') then 'entertaimentAndHobbies'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    else '' end as top2_lifestyle_situation,
                
                case
                    when top2_product = 'cooper' and top2_need_ID in ('10.18.5', '10.18.6') then 'default'
                    when top2_product = 'cooper' and top2_need_ID in ('10.18.2') then 'health'
                    when top2_product = 'cooper' and top2_need_ID in ('1.51.3', '1.51.4') then 'default' 
                    when top2_product = 'cooper' and top2_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'beauty' 
                    when top2_product = 'cooper' and top2_need_ID in ('1.9.1', '1.9.2') then 'pharmacy' 
                    when top2_product = 'cooper' and top2_need_ID in ('10.19.7', '10.19.10') then 'kids0To6' 
                    when top2_product = 'cooper' and top2_need_ID in ('8.8.4', '8.8.5') then 'default' 
                    when top2_product = 'cooper' and top2_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'specialShops' 
                    when top2_product = 'cooper' and top2_need_ID in ('7.2.1', '7.2.2') then 'shopsNotFromList' 
                    when top2_product = 'cooper' and top2_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants' 
                    when top2_product = 'cooper' and top2_need_ID in ('7.2.1', '7.2.2') then 'supermarkets' 
                    when top2_product = 'cooper' and top2_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'shoppingForHouse' 
                    when top2_product = 'cooper' and top2_need_ID in ('7.2.1') then 'default' 
                
                
                    when top2_product = 'sberHealth' and top2_need_ID in ('1.9.1', '1.9.2') then 'default'
                    when top2_product = 'sberHealth' and top2_need_ID in ('1.17.2', '1.17.6') then 'laboratory'
                    when top2_product = 'sberHealth' and top2_need_ID in ('10.17.6') then 'kids0To14'
                    when top2_product = 'sberHealth' and top2_need_ID in ('1.17.2', '1.17.6') then 'default'
                
                    when top2_product = 'citydrive' and top2_need_ID in ('2.25.3', '2.25.4') then 'carsharing'
                
                    when top2_product = 'sberPravo' and top2_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'fines'
                    when top2_product = 'sberPravo' and top2_need_ID in ('1.6.1') then 'taxes'
                    when top2_product = 'sberPravo' and top2_need_ID in ('3.4.4', '3.15.5') then 'moreThan2Estates'
                    when top2_product = 'sberPravo' and top2_need_ID in ('2.25.3', '2.25.4') then 'carsharing'
                
                    when top2_product = 'samokat' and top2_need_ID in ('1.51.3', '1.54.4') then 'default'
                    when top2_product = 'samokat' and top2_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default'
                    when top2_product = 'samokat' and top2_need_ID in ('10.19.7', '10.19.10') then 'dids0To6'
                    when top2_product = 'samokat' and top2_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top2_product = 'samokat' and top2_need_ID in ('8.56.3') then 'cinema'
                    when top2_product = 'samokat' and top2_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'books'
                    when top2_product = 'samokat' and top2_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants'
                    when top2_product = 'samokat' and top2_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'orderingFood'
                    when top2_product = 'samokat' and top2_need_ID in ('7.2.1') then 'default'
                
                    when top2_product = 'okko' and top2_need_ID in ('1.51.3') then 'default'
                    when top2_product = 'okko' and top2_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top2_product = 'okko' and top2_need_ID in ('5.37.2', '5.38.4') then 'selfeEducation'
                    when top2_product = 'okko' and top2_need_ID in ('8.56.3') then 'cinema'
                
                    when top2_product = 'zvuk' and top2_need_ID in ('1.51.3') then 'default'
                    when top2_product = 'zvuk' and top2_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top2_product = 'zls' and top2_need_ID in ('1.17.2', '1.17.6') then 'healthInsurance'
                    when top2_product = 'zls' and top2_need_ID in ('1.17.2', '1.17.6') then 'insuranceKids1To14'
                    when top2_product = 'zls' and top2_need_ID in ('3.4.4') then 'realEstate'
                    when top2_product = 'zls' and top2_need_ID in ('4.31.1') then 'financeInsurance'
                    when top2_product = 'zls' and top2_need_ID in ('3.26.4') then 'houseInsurance'
                    when top2_product = 'zls' and top2_need_ID in ('3.4.4') then 'realEstateInsurance'
                    when top2_product = 'zls' and top2_need_ID in ('3.15.5', '3.5.1') then 'houseRepairInsurance'
                    
                    when top2_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'vzr1Level'
                
                    when top2_product = 'mriya' and top2_need_ID in ('1.17.6', '1.17.2') then 'default'
                    when top2_product = 'mriya' and top2_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top2_product = 'sberMobile' and top2_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top2_product = 'sberMobile' and top2_need_ID in ('3.26.4') then 'newSbermobileSubscriber'
                
                    when top2_product = 'otello' and top2_need_ID in ('8.8.4', '8.8.5') then 'hasBonuses'
                
                    when top2_product = 'afisha' and approval_level >= 3 and top2_need_ID in ('8.56.3') then 'tickets'
                
                    when top2_product = 'manzherok' and approval_level >= 4 and top2_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('2.22.4') then 'interestedInBuyingCar'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('2.30.3', '2.30.5') then 'carRepair'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('1.51.3', '1.54.4') then 'default'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('10.19.10') then 'kids0To6'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('8.8.4', '8.8.7') then 'default'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'specialShops'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('8.59.5') then 'sporingEvents'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('8.59.3') then 'concertsEvents'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 4 and top2_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'householdServices'
                
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('2.22.4') then 'interestedInBuyingCar'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('2.30.3', '2.30.5') then 'carRepair'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'carOwners'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('2.27.1') then 'taxiTransfer'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('1.51.4', '1.51.3') then 'default'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'spaBeautySalons'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('10.19.7', '10.19.10') then 'kids0To6'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('5.40.2') then 'kids14To18'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'householdServices'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('8.56.3') then 'sportingEvents'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('8.56.3') then 'concertsEvents'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('6.42.1') then 'shoppingForClothes'
                    when top2_product = 'lifestyleGeneral' and approval_level >= 6 and top2_need_ID in ('8.56.3') then 'concierge'
                    else '' end as top2_lifestyle_sub_situation,
                
                
                        CASE 
                                WHEN top2_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_1, 0))
                                WHEN top2_product = 'okko' THEN round(coalesce(okko_without_premier_level_1, 0))
                                else 0
                                END as top2_benefit_rubles_lvl1,
                                
                                case 
                                WHEN top2_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_1, 0))
                                WHEN top2_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_1, 0))
                                else 0 
                                end as top2_benefit_bonuses_lvl1,
                
                                CASE 
                                WHEN top2_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_2, 0))
                                WHEN top2_product = 'okko' THEN round(coalesce(okko_without_premier_level_2, 0))
                                else 0
                                END as top2_benefit_rubles_lvl2,
                                
                                case 
                                WHEN top2_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_2, 0))
                                WHEN top2_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_2, 0))
                                else 0 
                                end as top2_benefit_bonuses_lvl2,
                
                                CASE 
                                WHEN top2_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_3, 0))
                                WHEN top2_product = 'okko' THEN round(coalesce(okko_without_premier_level_3, 0))
                                else 0
                                END as top2_benefit_rubles_lvl3,
                                
                                case 
                                WHEN top2_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_3, 0))
                                WHEN top2_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_3, 0))
                                else 0
                                end as top2_benefit_bonuses_lvl3,
                
                                CASE 
                                WHEN top2_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_4_profit, 0))
                                WHEN top2_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_4_profit, 0))
                                else 0
                                END as top2_benefit_rubles_lvl4,
                                
                                case 
                                WHEN top2_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_4, 0))
                                WHEN top2_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_4, 0))
                                else 0 
                                end as top2_benefit_bonuses_lvl4,
                
                                CASE 
                                WHEN top2_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top2_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top2_benefit_rubles_lvl5,
                                
                                case
                                WHEN top2_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top2_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top2_benefit_bonuses_lvl5,
                
                                CASE 
                                WHEN top2_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top2_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top2_benefit_rubles_lvl6,
                                
                                case
                                WHEN top2_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top2_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top2_benefit_bonuses_lvl6,
                
                        
                        
                        
                        
                        case when top3_product is null or top3_product = '' then 'plug'
                        else top3_product end as top3_product,
                        'ecosystem' as top3_type,
                        case when top3_score is null or top3_score = '' then 0
                        else top3_score end as top3_score,
                        '' as top3_offer_type, '' as top3_advantages, 0 as top3_is_recommended, 0 as top3_used_by_client,
                           case
                    when top3_product = 'cooper' and top3_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets'
                    when top3_product = 'cooper' and top3_need_ID in ('1.51.3', '1.51.4') then 'sport' 
                    when top3_product = 'cooper' and top3_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty' 
                    when top3_product = 'cooper' and top3_need_ID in ('1.9.1', '1.9.2') then 'healthAndBeauty' 
                    when top3_product = 'cooper' and top3_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top3_product = 'cooper' and top3_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top3_product = 'cooper' and top3_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'entertainmentAndHobbies' 
                    when top3_product = 'cooper' and top3_need_ID in ('7.2.1', '7.2.2') then 'other' 
                    when top3_product = 'cooper' and top3_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other'
                    when top3_product = 'cooper' and top3_need_ID in ('7.2.1', '7.2.2') then 'realEstate' 
                    when top3_product = 'cooper' and top3_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'realEstate'
                    when top3_product = 'cooper' and top3_need_ID in ('7.2.1') then 'default'
                
                
                    when top3_product = 'sberHealth' and top3_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets'
                    when top3_product = 'sberHealth' and top3_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty'
                    when top3_product = 'sberHealth' and top3_need_ID in ('10.17.6') then 'kids'
                    when top3_product = 'sberHealth' and top3_need_ID in ('1.17.2', '1.17.6') then 'default'
                
                    when top3_product = 'citydrive' and top3_need_ID in ('2.25.3', '2.25.4') then 'auto'
                
                    when top3_product = 'sberPravo' and top3_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'auto'
                    when top3_product = 'sberPravo' and top3_need_ID in ('1.6.1') then 'other'
                    when top3_product = 'sberPravo' and top3_need_ID in ('3.4.4', '3.15.5') then 'realEstate'
                    when top3_product = 'sberPravo' and top3_need_ID in ('2.25.3', '2.25.4') then 'auto'
                
                    when top3_product = 'samokat' and top3_need_ID in ('1.51.3', '1.54.4') then 'sport'
                    when top3_product = 'samokat' and top3_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top3_product = 'samokat' and top3_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top3_product = 'samokat' and top3_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top3_product = 'samokat' and top3_need_ID in ('8.56.3') then 'entertainmentAndHobbies'
                    when top3_product = 'samokat' and top3_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'entertainmentAndHobbies'
                    when top3_product = 'samokat' and top3_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other'
                    when top3_product = 'samokat' and top3_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'realEstate'
                    when top3_product = 'samokat' and top3_need_ID in ('7.2.1') then 'default'
                
                    when top3_product = 'okko' and top3_need_ID in ('1.51.3') then 'sport'
                    when top3_product = 'okko' and top3_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top3_product = 'okko' and top3_need_ID in ('5.37.2', '5.38.4') then 'entertainmentAndHobbies'
                    when top3_product = 'okko' and top3_need_ID in ('8.56.3') then 'entertainmentAndHobbies'
                
                    when top3_product = 'zvuk' and top3_need_ID in ('1.51.3') then 'sport'
                    when top3_product = 'zvuk' and top3_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top3_product = 'zls' and top3_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty'
                    when top3_product = 'zls' and top3_need_ID in ('1.17.2', '1.17.6') then 'kids'
                    when top3_product = 'zls' and top3_need_ID in ('3.4.4') then 'trips'
                    when top3_product = 'zls' and top3_need_ID in ('4.31.1') then 'other'
                    when top3_product = 'zls' and top3_need_ID in ('3.26.4') then 'realEstate'
                    when top3_product = 'zls' and top3_need_ID in ('3.4.4') then 'realEstate'
                    when top3_product = 'zls' and top3_need_ID in ('3.15.5', '3.5.1') then 'realEstate'
                    
                    when top3_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'trips'
                
                    when top3_product = 'mriya' and top3_need_ID in ('1.17.6', '1.17.2') then 'healthAndBeauty'
                    when top3_product = 'mriya' and top3_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top3_product = 'sberMobile' and top3_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top3_product = 'sberMobile' and top3_need_ID in ('3.26.4') then 'realEstate'
                
                    when top3_product = 'otello' and top3_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top3_product = 'afisha' and approval_level >= 3 and top3_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top3_product = 'manzherok' and approval_level >= 4 and top3_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('2.22.4') then 'auto'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('2.30.3', '2.30.5') then 'auto'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('1.51.3', '1.54.4') then 'sport'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('10.19.10') then 'kids'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('8.8.4', '8.8.7') then 'trips'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'entertaimentAndHobbies'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('8.59.5') then 'entertaimentAndHobbies'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('8.59.3') then 'entertaimentAndHobbies'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'realEstate'
                
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('2.22.4') then 'auto'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('2.30.3', '2.30.5') then 'auto'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'auto'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('2.27.1') then 'auto'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('1.51.4', '1.51.3') then 'sport'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('5.40.2') then 'kids'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'realEstate'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('6.42.1') then 'entertaimentAndHobbies'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    else '' end as top3_lifestyle_situation,
                
                case
                    when top3_product = 'cooper' and top3_need_ID in ('10.18.5', '10.18.6') then 'default'
                    when top3_product = 'cooper' and top3_need_ID in ('10.18.2') then 'health'
                    when top3_product = 'cooper' and top3_need_ID in ('1.51.3', '1.51.4') then 'default' 
                    when top3_product = 'cooper' and top3_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'beauty' 
                    when top3_product = 'cooper' and top3_need_ID in ('1.9.1', '1.9.2') then 'pharmacy' 
                    when top3_product = 'cooper' and top3_need_ID in ('10.19.7', '10.19.10') then 'kids0To6' 
                    when top3_product = 'cooper' and top3_need_ID in ('8.8.4', '8.8.5') then 'default' 
                    when top3_product = 'cooper' and top3_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'specialShops' 
                    when top3_product = 'cooper' and top3_need_ID in ('7.2.1', '7.2.2') then 'shopsNotFromList' 
                    when top3_product = 'cooper' and top3_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants' 
                    when top3_product = 'cooper' and top3_need_ID in ('7.2.1', '7.2.2') then 'supermarkets' 
                    when top3_product = 'cooper' and top3_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'shoppingForHouse' 
                    when top3_product = 'cooper' and top3_need_ID in ('7.2.1') then 'default' 
                
                
                    when top3_product = 'sberHealth' and top3_need_ID in ('1.9.1', '1.9.2') then 'default'
                    when top3_product = 'sberHealth' and top3_need_ID in ('1.17.2', '1.17.6') then 'laboratory'
                    when top3_product = 'sberHealth' and top3_need_ID in ('10.17.6') then 'kids0To14'
                    when top3_product = 'sberHealth' and top3_need_ID in ('1.17.2', '1.17.6') then 'default'
                
                    when top3_product = 'citydrive' and top3_need_ID in ('2.25.3', '2.25.4') then 'carsharing'
                
                    when top3_product = 'sberPravo' and top3_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'fines'
                    when top3_product = 'sberPravo' and top3_need_ID in ('1.6.1') then 'taxes'
                    when top3_product = 'sberPravo' and top3_need_ID in ('3.4.4', '3.15.5') then 'moreThan2Estates'
                    when top3_product = 'sberPravo' and top3_need_ID in ('2.25.3', '2.25.4') then 'carsharing'
                
                    when top3_product = 'samokat' and top3_need_ID in ('1.51.3', '1.54.4') then 'default'
                    when top3_product = 'samokat' and top3_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default'
                    when top3_product = 'samokat' and top3_need_ID in ('10.19.7', '10.19.10') then 'dids0To6'
                    when top3_product = 'samokat' and top3_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top3_product = 'samokat' and top3_need_ID in ('8.56.3') then 'cinema'
                    when top3_product = 'samokat' and top3_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'books'
                    when top3_product = 'samokat' and top3_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants'
                    when top3_product = 'samokat' and top3_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'orderingFood'
                    when top3_product = 'samokat' and top3_need_ID in ('7.2.1') then 'default'
                
                    when top3_product = 'okko' and top3_need_ID in ('1.51.3') then 'default'
                    when top3_product = 'okko' and top3_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top3_product = 'okko' and top3_need_ID in ('5.37.2', '5.38.4') then 'selfeEducation'
                    when top3_product = 'okko' and top3_need_ID in ('8.56.3') then 'cinema'
                
                    when top3_product = 'zvuk' and top3_need_ID in ('1.51.3') then 'default'
                    when top3_product = 'zvuk' and top3_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top3_product = 'zls' and top3_need_ID in ('1.17.2', '1.17.6') then 'healthInsurance'
                    when top3_product = 'zls' and top3_need_ID in ('1.17.2', '1.17.6') then 'insuranceKids1To14'
                    when top3_product = 'zls' and top3_need_ID in ('3.4.4') then 'realEstate'
                    when top3_product = 'zls' and top3_need_ID in ('4.31.1') then 'financeInsurance'
                    when top3_product = 'zls' and top3_need_ID in ('3.26.4') then 'houseInsurance'
                    when top3_product = 'zls' and top3_need_ID in ('3.4.4') then 'realEstateInsurance'
                    when top3_product = 'zls' and top3_need_ID in ('3.15.5', '3.5.1') then 'houseRepairInsurance'
                    
                    when top3_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'vzr1Level'
                
                    when top3_product = 'mriya' and top3_need_ID in ('1.17.6', '1.17.2') then 'default'
                    when top3_product = 'mriya' and top3_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top3_product = 'sberMobile' and top3_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top3_product = 'sberMobile' and top3_need_ID in ('3.26.4') then 'newSbermobileSubscriber'
                
                    when top3_product = 'otello' and top3_need_ID in ('8.8.4', '8.8.5') then 'hasBonuses'
                
                    when top3_product = 'afisha' and approval_level >= 3 and top3_need_ID in ('8.56.3') then 'tickets'
                
                    when top3_product = 'manzherok' and approval_level >= 4 and top3_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('2.22.4') then 'interestedInBuyingCar'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('2.30.3', '2.30.5') then 'carRepair'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('1.51.3', '1.54.4') then 'default'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('10.19.10') then 'kids0To6'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('8.8.4', '8.8.7') then 'default'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'specialShops'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('8.59.5') then 'sporingEvents'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('8.59.3') then 'concertsEvents'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 4 and top3_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'householdServices'
                
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('2.22.4') then 'interestedInBuyingCar'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('2.30.3', '2.30.5') then 'carRepair'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'carOwners'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('2.27.1') then 'taxiTransfer'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('1.51.4', '1.51.3') then 'default'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'spaBeautySalons'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('10.19.7', '10.19.10') then 'kids0To6'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('5.40.2') then 'kids14To18'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'householdServices'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('8.56.3') then 'sportingEvents'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('8.56.3') then 'concertsEvents'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('6.42.1') then 'shoppingForClothes'
                    when top3_product = 'lifestyleGeneral' and approval_level >= 6 and top3_need_ID in ('8.56.3') then 'concierge'
                    else '' end as top3_lifestyle_sub_situation,
                
                
                         CASE 
                                WHEN top3_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_1, 0))
                                WHEN top3_product = 'okko' THEN round(coalesce(okko_without_premier_level_1, 0))
                                else 0
                                END as top3_benefit_rubles_lvl1,
                                
                                case 
                                WHEN top3_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_1, 0))
                                WHEN top3_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_1, 0))
                                else 0 
                                end as top3_benefit_bonuses_lvl1,
                
                                CASE 
                                WHEN top3_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_2, 0))
                                WHEN top3_product = 'okko' THEN round(coalesce(okko_without_premier_level_2, 0))
                                else 0
                                END as top3_benefit_rubles_lvl2,
                                
                                case 
                                WHEN top3_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_2, 0))
                                WHEN top3_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_2, 0))
                                else 0 
                                end as top3_benefit_bonuses_lvl2,
                
                                CASE 
                                WHEN top3_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_3, 0))
                                WHEN top3_product = 'okko' THEN round(coalesce(okko_without_premier_level_3, 0))
                                else 0
                                END as top3_benefit_rubles_lvl3,
                                
                                case 
                                WHEN top3_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_3, 0))
                                WHEN top3_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_3, 0))
                                else 0
                                end as top3_benefit_bonuses_lvl3,
                
                                CASE 
                                WHEN top3_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_4_profit, 0))
                                WHEN top3_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_4_profit, 0))
                                else 0
                                END as top3_benefit_rubles_lvl4,
                                
                                case 
                                WHEN top3_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_4, 0))
                                WHEN top3_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_4, 0))
                                else 0 
                                end as top3_benefit_bonuses_lvl4,
                
                                CASE 
                                WHEN top3_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top3_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top3_benefit_rubles_lvl5,
                                
                                case
                                WHEN top3_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top3_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top3_benefit_bonuses_lvl5,
                
                                CASE 
                                WHEN top3_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top3_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top3_benefit_rubles_lvl6,
                                
                                case
                                WHEN top4_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top3_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top3_benefit_bonuses_lvl6,
                
                
                    case when top4_product is null or top4_product = '' then 'plug'
                        else top4_product end as top4_product,
                        'ecosystem' as top4_type,
                        case when top4_score is null or top4_score = '' then 0
                        else top4_score end as top4_score,
                    '' as top4_offer_type, '' as top4_advantages, 0 as top4_is_recommended, 0 as top4_used_by_client,
                           case
                    when top4_product = 'cooper' and top4_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets'
                    when top4_product = 'cooper' and top4_need_ID in ('1.51.3', '1.51.4') then 'sport' 
                    when top4_product = 'cooper' and top4_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty' 
                    when top4_product = 'cooper' and top4_need_ID in ('1.9.1', '1.9.2') then 'healthAndBeauty' 
                    when top4_product = 'cooper' and top4_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top4_product = 'cooper' and top4_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top4_product = 'cooper' and top4_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'entertainmentAndHobbies' 
                    when top4_product = 'cooper' and top4_need_ID in ('7.2.1', '7.2.2') then 'other' 
                    when top4_product = 'cooper' and top4_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other'
                    when top4_product = 'cooper' and top4_need_ID in ('7.2.1', '7.2.2') then 'realEstate' 
                    when top4_product = 'cooper' and top4_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'realEstate'
                    when top4_product = 'cooper' and top4_need_ID in ('7.2.1') then 'default'
                
                
                    when top4_product = 'sberHealth' and top4_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets'
                    when top4_product = 'sberHealth' and top4_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty'
                    when top4_product = 'sberHealth' and top4_need_ID in ('10.17.6') then 'kids'
                    when top4_product = 'sberHealth' and top4_need_ID in ('1.17.2', '1.17.6') then 'default'
                
                    when top4_product = 'citydrive' and top4_need_ID in ('2.25.3', '2.25.4') then 'auto'
                
                    when top4_product = 'sberPravo' and top4_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'auto'
                    when top4_product = 'sberPravo' and top4_need_ID in ('1.6.1') then 'other'
                    when top4_product = 'sberPravo' and top4_need_ID in ('3.4.4', '3.15.5') then 'realEstate'
                    when top4_product = 'sberPravo' and top4_need_ID in ('2.25.3', '2.25.4') then 'auto'
                
                    when top4_product = 'samokat' and top4_need_ID in ('1.51.3', '1.54.4') then 'sport'
                    when top4_product = 'samokat' and top4_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top4_product = 'samokat' and top4_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top4_product = 'samokat' and top4_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top4_product = 'samokat' and top4_need_ID in ('8.56.3') then 'entertainmentAndHobbies'
                    when top4_product = 'samokat' and top4_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'entertainmentAndHobbies'
                    when top4_product = 'samokat' and top4_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other'
                    when top4_product = 'samokat' and top4_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'realEstate'
                    when top4_product = 'samokat' and top4_need_ID in ('7.2.1') then 'default'
                
                    when top4_product = 'okko' and top4_need_ID in ('1.51.3') then 'sport'
                    when top4_product = 'okko' and top4_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top4_product = 'okko' and top4_need_ID in ('5.37.2', '5.38.4') then 'entertainmentAndHobbies'
                    when top4_product = 'okko' and top4_need_ID in ('8.56.3') then 'entertainmentAndHobbies'
                
                    when top4_product = 'zvuk' and top4_need_ID in ('1.51.3') then 'sport'
                    when top4_product = 'zvuk' and top4_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top4_product = 'zls' and top4_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty'
                    when top4_product = 'zls' and top4_need_ID in ('1.17.2', '1.17.6') then 'kids'
                    when top4_product = 'zls' and top4_need_ID in ('3.4.4') then 'trips'
                    when top4_product = 'zls' and top4_need_ID in ('4.31.1') then 'other'
                    when top4_product = 'zls' and top4_need_ID in ('3.26.4') then 'realEstate'
                    when top4_product = 'zls' and top4_need_ID in ('3.4.4') then 'realEstate'
                    when top4_product = 'zls' and top4_need_ID in ('3.15.5', '3.5.1') then 'realEstate'
                    
                    when top4_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'trips'
                
                    when top4_product = 'mriya' and top4_need_ID in ('1.17.6', '1.17.2') then 'healthAndBeauty'
                    when top4_product = 'mriya' and top4_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top4_product = 'sberMobile' and top4_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top4_product = 'sberMobile' and top4_need_ID in ('3.26.4') then 'realEstate'
                
                    when top4_product = 'otello' and top4_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top4_product = 'afisha' and approval_level >= 3 and top4_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top4_product = 'manzherok' and approval_level >= 4 and top4_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('2.22.4') then 'auto'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('2.30.3', '2.30.5') then 'auto'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('1.51.3', '1.54.4') then 'sport'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('10.19.10') then 'kids'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('8.8.4', '8.8.7') then 'trips'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'entertaimentAndHobbies'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('8.59.5') then 'entertaimentAndHobbies'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('8.59.3') then 'entertaimentAndHobbies'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'realEstate'
                
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('2.22.4') then 'auto'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('2.30.3', '2.30.5') then 'auto'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'auto'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('2.27.1') then 'auto'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('1.51.4', '1.51.3') then 'sport'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('5.40.2') then 'kids'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'realEstate'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('6.42.1') then 'entertaimentAndHobbies'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    else '' end as top4_lifestyle_situation,
                
                case
                    when top4_product = 'cooper' and top4_need_ID in ('10.18.5', '10.18.6') then 'default'
                    when top4_product = 'cooper' and top4_need_ID in ('10.18.2') then 'health'
                    when top4_product = 'cooper' and top4_need_ID in ('1.51.3', '1.51.4') then 'default' 
                    when top4_product = 'cooper' and top4_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'beauty' 
                    when top4_product = 'cooper' and top4_need_ID in ('1.9.1', '1.9.2') then 'pharmacy' 
                    when top4_product = 'cooper' and top4_need_ID in ('10.19.7', '10.19.10') then 'kids0To6' 
                    when top4_product = 'cooper' and top4_need_ID in ('8.8.4', '8.8.5') then 'default' 
                    when top4_product = 'cooper' and top4_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'specialShops' 
                    when top4_product = 'cooper' and top4_need_ID in ('7.2.1', '7.2.2') then 'shopsNotFromList' 
                    when top4_product = 'cooper' and top4_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants' 
                    when top4_product = 'cooper' and top4_need_ID in ('7.2.1', '7.2.2') then 'supermarkets' 
                    when top4_product = 'cooper' and top4_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'shoppingForHouse' 
                    when top4_product = 'cooper' and top4_need_ID in ('7.2.1') then 'default' 
                
                
                    when top4_product = 'sberHealth' and top4_need_ID in ('1.9.1', '1.9.2') then 'default'
                    when top4_product = 'sberHealth' and top4_need_ID in ('1.17.2', '1.17.6') then 'laboratory'
                    when top4_product = 'sberHealth' and top4_need_ID in ('10.17.6') then 'kids0To14'
                    when top4_product = 'sberHealth' and top4_need_ID in ('1.17.2', '1.17.6') then 'default'
                
                    when top4_product = 'citydrive' and top4_need_ID in ('2.25.3', '2.25.4') then 'carsharing'
                
                    when top4_product = 'sberPravo' and top4_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'fines'
                    when top4_product = 'sberPravo' and top4_need_ID in ('1.6.1') then 'taxes'
                    when top4_product = 'sberPravo' and top4_need_ID in ('3.4.4', '3.15.5') then 'moreThan2Estates'
                    when top4_product = 'sberPravo' and top4_need_ID in ('2.25.3', '2.25.4') then 'carsharing'
                
                    when top4_product = 'samokat' and top4_need_ID in ('1.51.3', '1.54.4') then 'default'
                    when top4_product = 'samokat' and top4_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default'
                    when top4_product = 'samokat' and top4_need_ID in ('10.19.7', '10.19.10') then 'dids0To6'
                    when top4_product = 'samokat' and top4_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top4_product = 'samokat' and top4_need_ID in ('8.56.3') then 'cinema'
                    when top4_product = 'samokat' and top4_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'books'
                    when top4_product = 'samokat' and top4_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants'
                    when top4_product = 'samokat' and top4_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'orderingFood'
                    when top4_product = 'samokat' and top4_need_ID in ('7.2.1') then 'default'
                
                    when top4_product = 'okko' and top4_need_ID in ('1.51.3') then 'default'
                    when top4_product = 'okko' and top4_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top4_product = 'okko' and top4_need_ID in ('5.37.2', '5.38.4') then 'selfeEducation'
                    when top4_product = 'okko' and top4_need_ID in ('8.56.3') then 'cinema'
                
                    when top4_product = 'zvuk' and top4_need_ID in ('1.51.3') then 'default'
                    when top4_product = 'zvuk' and top4_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top4_product = 'zls' and top4_need_ID in ('1.17.2', '1.17.6') then 'healthInsurance'
                    when top4_product = 'zls' and top4_need_ID in ('1.17.2', '1.17.6') then 'insuranceKids1To14'
                    when top4_product = 'zls' and top4_need_ID in ('3.4.4') then 'realEstate'
                    when top4_product = 'zls' and top4_need_ID in ('4.31.1') then 'financeInsurance'
                    when top4_product = 'zls' and top4_need_ID in ('3.26.4') then 'houseInsurance'
                    when top4_product = 'zls' and top4_need_ID in ('3.4.4') then 'realEstateInsurance'
                    when top4_product = 'zls' and top4_need_ID in ('3.15.5', '3.5.1') then 'houseRepairInsurance'
                    
                    when top4_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'vzr1Level'
                
                    when top4_product = 'mriya' and top4_need_ID in ('1.17.6', '1.17.2') then 'default'
                    when top4_product = 'mriya' and top4_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top4_product = 'sberMobile' and top4_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top4_product = 'sberMobile' and top4_need_ID in ('3.26.4') then 'newSbermobileSubscriber'
                
                    when top4_product = 'otello' and top4_need_ID in ('8.8.4', '8.8.5') then 'hasBonuses'
                
                    when top4_product = 'afisha' and approval_level >= 3 and top4_need_ID in ('8.56.3') then 'tickets'
                
                    when top4_product = 'manzherok' and approval_level >= 4 and top4_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('2.22.4') then 'interestedInBuyingCar'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('2.30.3', '2.30.5') then 'carRepair'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('1.51.3', '1.54.4') then 'default'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('10.19.10') then 'kids0To6'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('8.8.4', '8.8.7') then 'default'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'specialShops'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('8.59.5') then 'sporingEvents'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('8.59.3') then 'concertsEvents'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 4 and top4_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'householdServices'
                
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('2.22.4') then 'interestedInBuyingCar'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('2.30.3', '2.30.5') then 'carRepair'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'carOwners'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('2.27.1') then 'taxiTransfer'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('1.51.4', '1.51.3') then 'default'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'spaBeautySalons'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('10.19.7', '10.19.10') then 'kids0To6'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('5.40.2') then 'kids14To18'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'householdServices'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('8.56.3') then 'sportingEvents'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('8.56.3') then 'concertsEvents'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('6.42.1') then 'shoppingForClothes'
                    when top4_product = 'lifestyleGeneral' and approval_level >= 6 and top4_need_ID in ('8.56.3') then 'concierge'
                    else '' end as top4_lifestyle_sub_situation,
                
                
                        CASE 
                                WHEN top4_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_1, 0))
                                WHEN top4_product = 'okko' THEN round(coalesce(okko_without_premier_level_1, 0))
                                else 0
                                END as top4_benefit_rubles_lvl1,
                                
                                case 
                                WHEN top4_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_1, 0))
                                WHEN top4_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_1, 0))
                                else 0 
                                end as top4_benefit_bonuses_lvl1,
                
                                CASE 
                                WHEN top4_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_2, 0))
                                WHEN top4_product = 'okko' THEN round(coalesce(okko_without_premier_level_2, 0))
                                else 0
                                END as top4_benefit_rubles_lvl2,
                                
                                case 
                                WHEN top4_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_2, 0))
                                WHEN top4_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_2, 0))
                                else 0 
                                end as top4_benefit_bonuses_lvl2,
                
                                CASE 
                                WHEN top4_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_3, 0))
                                WHEN top4_product = 'okko' THEN round(coalesce(okko_without_premier_level_3, 0))
                                else 0
                                END as top4_benefit_rubles_lvl3,
                                
                                case 
                                WHEN top4_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_3, 0))
                                WHEN top4_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_3, 0))
                                else 0
                                end as top4_benefit_bonuses_lvl3,
                
                                CASE 
                                WHEN top4_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_4_profit, 0))
                                WHEN top4_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_4_profit, 0))
                                else 0
                                END as top4_benefit_rubles_lvl4,
                                
                                case 
                                WHEN top4_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_4, 0))
                                WHEN top4_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_4, 0))
                                else 0 
                                end as top4_benefit_bonuses_lvl4,
                
                                CASE 
                                WHEN top4_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top4_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top4_benefit_rubles_lvl5,
                                
                                case
                                WHEN top4_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top4_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top4_benefit_bonuses_lvl5,
                
                                CASE 
                                WHEN top4_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top4_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top4_benefit_rubles_lvl6,
                                
                                case
                                WHEN top4_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top4_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top4_benefit_bonuses_lvl6,
                
                
                    case when top5_product is null or top5_product = '' then 'plug'
                        else top5_product end as top5_product,
                        'ecosystem' as top5_type,
                        case when top5_score is null or top5_score = '' then 0
                        else top5_score end as top5_score,
                    '' as top5_offer_type, '' as top5_advantages, 0 as top5_is_recommended, 0 as top5_used_by_client,
                            case
                    when top5_product = 'cooper' and top5_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets'
                    when top5_product = 'cooper' and top5_need_ID in ('1.51.3', '1.51.4') then 'sport' 
                    when top5_product = 'cooper' and top5_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty' 
                    when top5_product = 'cooper' and top5_need_ID in ('1.9.1', '1.9.2') then 'healthAndBeauty' 
                    when top5_product = 'cooper' and top5_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top5_product = 'cooper' and top5_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top5_product = 'cooper' and top5_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'entertainmentAndHobbies' 
                    when top5_product = 'cooper' and top5_need_ID in ('7.2.1', '7.2.2') then 'other' 
                    when top5_product = 'cooper' and top5_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other'
                    when top5_product = 'cooper' and top5_need_ID in ('7.2.1', '7.2.2') then 'realEstate' 
                    when top5_product = 'cooper' and top5_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'realEstate'
                    when top5_product = 'cooper' and top5_need_ID in ('7.2.1') then 'default'
                
                
                    when top5_product = 'sberHealth' and top5_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets'
                    when top5_product = 'sberHealth' and top5_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty'
                    when top5_product = 'sberHealth' and top5_need_ID in ('10.17.6') then 'kids'
                    when top5_product = 'sberHealth' and top5_need_ID in ('1.17.2', '1.17.6') then 'default'
                
                    when top5_product = 'citydrive' and top5_need_ID in ('2.25.3', '2.25.4') then 'auto'
                
                    when top5_product = 'sberPravo' and top5_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'auto'
                    when top5_product = 'sberPravo' and top5_need_ID in ('1.6.1') then 'other'
                    when top5_product = 'sberPravo' and top5_need_ID in ('3.4.4', '3.15.5') then 'realEstate'
                    when top5_product = 'sberPravo' and top5_need_ID in ('2.25.3', '2.25.4') then 'auto'
                
                    when top5_product = 'samokat' and top5_need_ID in ('1.51.3', '1.54.4') then 'sport'
                    when top5_product = 'samokat' and top5_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top5_product = 'samokat' and top5_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top5_product = 'samokat' and top5_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top5_product = 'samokat' and top5_need_ID in ('8.56.3') then 'entertainmentAndHobbies'
                    when top5_product = 'samokat' and top5_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'entertainmentAndHobbies'
                    when top5_product = 'samokat' and top5_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other'
                    when top5_product = 'samokat' and top5_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'realEstate'
                    when top5_product = 'samokat' and top5_need_ID in ('7.2.1') then 'default'
                
                    when top5_product = 'okko' and top5_need_ID in ('1.51.3') then 'sport'
                    when top5_product = 'okko' and top5_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top5_product = 'okko' and top5_need_ID in ('5.37.2', '5.38.4') then 'entertainmentAndHobbies'
                    when top5_product = 'okko' and top5_need_ID in ('8.56.3') then 'entertainmentAndHobbies'
                
                    when top5_product = 'zvuk' and top5_need_ID in ('1.51.3') then 'sport'
                    when top5_product = 'zvuk' and top5_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top5_product = 'zls' and top5_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty'
                    when top5_product = 'zls' and top5_need_ID in ('1.17.2', '1.17.6') then 'kids'
                    when top5_product = 'zls' and top5_need_ID in ('3.4.4') then 'trips'
                    when top5_product = 'zls' and top5_need_ID in ('4.31.1') then 'other'
                    when top5_product = 'zls' and top5_need_ID in ('3.26.4') then 'realEstate'
                    when top5_product = 'zls' and top5_need_ID in ('3.4.4') then 'realEstate'
                    when top5_product = 'zls' and top5_need_ID in ('3.15.5', '3.5.1') then 'realEstate'
                    
                    when top5_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'trips'
                
                    when top5_product = 'mriya' and top5_need_ID in ('1.17.6', '1.17.2') then 'healthAndBeauty'
                    when top5_product = 'mriya' and top5_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top5_product = 'sberMobile' and top5_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top5_product = 'sberMobile' and top5_need_ID in ('3.26.4') then 'realEstate'
                
                    when top5_product = 'otello' and top5_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top5_product = 'afisha' and approval_level >= 3 and top5_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top5_product = 'manzherok' and approval_level >= 4 and top5_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('2.22.4') then 'auto'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('2.30.3', '2.30.5') then 'auto'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('1.51.3', '1.54.4') then 'sport'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('10.19.10') then 'kids'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('8.8.4', '8.8.7') then 'trips'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'entertaimentAndHobbies'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('8.59.5') then 'entertaimentAndHobbies'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('8.59.3') then 'entertaimentAndHobbies'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'realEstate'
                
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('2.22.4') then 'auto'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('2.30.3', '2.30.5') then 'auto'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'auto'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('2.27.1') then 'auto'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('1.51.4', '1.51.3') then 'sport'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('5.40.2') then 'kids'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'realEstate'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('6.42.1') then 'entertaimentAndHobbies'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    else '' end as top5_lifestyle_situation,
                
                case
                    when top5_product = 'cooper' and top5_need_ID in ('10.18.5', '10.18.6') then 'default'
                    when top5_product = 'cooper' and top5_need_ID in ('10.18.2') then 'health'
                    when top5_product = 'cooper' and top5_need_ID in ('1.51.3', '1.51.4') then 'default' 
                    when top5_product = 'cooper' and top5_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'beauty' 
                    when top5_product = 'cooper' and top5_need_ID in ('1.9.1', '1.9.2') then 'pharmacy' 
                    when top5_product = 'cooper' and top5_need_ID in ('10.19.7', '10.19.10') then 'kids0To6' 
                    when top5_product = 'cooper' and top5_need_ID in ('8.8.4', '8.8.5') then 'default' 
                    when top5_product = 'cooper' and top5_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'specialShops' 
                    when top5_product = 'cooper' and top5_need_ID in ('7.2.1', '7.2.2') then 'shopsNotFromList' 
                    when top5_product = 'cooper' and top5_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants' 
                    when top5_product = 'cooper' and top5_need_ID in ('7.2.1', '7.2.2') then 'supermarkets' 
                    when top5_product = 'cooper' and top5_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'shoppingForHouse' 
                    when top5_product = 'cooper' and top5_need_ID in ('7.2.1') then 'default' 
                
                
                    when top5_product = 'sberHealth' and top5_need_ID in ('1.9.1', '1.9.2') then 'default'
                    when top5_product = 'sberHealth' and top5_need_ID in ('1.17.2', '1.17.6') then 'laboratory'
                    when top5_product = 'sberHealth' and top5_need_ID in ('10.17.6') then 'kids0To14'
                    when top5_product = 'sberHealth' and top5_need_ID in ('1.17.2', '1.17.6') then 'default'
                
                    when top5_product = 'citydrive' and top5_need_ID in ('2.25.3', '2.25.4') then 'carsharing'
                
                    when top5_product = 'sberPravo' and top5_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'fines'
                    when top5_product = 'sberPravo' and top5_need_ID in ('1.6.1') then 'taxes'
                    when top5_product = 'sberPravo' and top5_need_ID in ('3.4.4', '3.15.5') then 'moreThan2Estates'
                    when top5_product = 'sberPravo' and top5_need_ID in ('2.25.3', '2.25.4') then 'carsharing'
                
                    when top5_product = 'samokat' and top5_need_ID in ('1.51.3', '1.54.4') then 'default'
                    when top5_product = 'samokat' and top5_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default'
                    when top5_product = 'samokat' and top5_need_ID in ('10.19.7', '10.19.10') then 'dids0To6'
                    when top5_product = 'samokat' and top5_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top5_product = 'samokat' and top5_need_ID in ('8.56.3') then 'cinema'
                    when top5_product = 'samokat' and top5_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'books'
                    when top5_product = 'samokat' and top5_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants'
                    when top5_product = 'samokat' and top5_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'orderingFood'
                    when top5_product = 'samokat' and top5_need_ID in ('7.2.1') then 'default'
                
                    when top5_product = 'okko' and top5_need_ID in ('1.51.3') then 'default'
                    when top5_product = 'okko' and top5_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top5_product = 'okko' and top5_need_ID in ('5.37.2', '5.38.4') then 'selfeEducation'
                    when top5_product = 'okko' and top5_need_ID in ('8.56.3') then 'cinema'
                
                    when top5_product = 'zvuk' and top5_need_ID in ('1.51.3') then 'default'
                    when top5_product = 'zvuk' and top5_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top5_product = 'zls' and top5_need_ID in ('1.17.2', '1.17.6') then 'healthInsurance'
                    when top5_product = 'zls' and top5_need_ID in ('1.17.2', '1.17.6') then 'insuranceKids1To14'
                    when top5_product = 'zls' and top5_need_ID in ('3.4.4') then 'realEstate'
                    when top5_product = 'zls' and top5_need_ID in ('4.31.1') then 'financeInsurance'
                    when top5_product = 'zls' and top5_need_ID in ('3.26.4') then 'houseInsurance'
                    when top5_product = 'zls' and top5_need_ID in ('3.4.4') then 'realEstateInsurance'
                    when top5_product = 'zls' and top5_need_ID in ('3.15.5', '3.5.1') then 'houseRepairInsurance'
                    
                    when top5_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'vzr1Level'
                
                    when top5_product = 'mriya' and top5_need_ID in ('1.17.6', '1.17.2') then 'default'
                    when top5_product = 'mriya' and top5_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top5_product = 'sberMobile' and top5_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top5_product = 'sberMobile' and top5_need_ID in ('3.26.4') then 'newSbermobileSubscriber'
                
                    when top5_product = 'otello' and top5_need_ID in ('8.8.4', '8.8.5') then 'hasBonuses'
                
                    when top5_product = 'afisha' and approval_level >= 3 and top5_need_ID in ('8.56.3') then 'tickets'
                
                    when top5_product = 'manzherok' and approval_level >= 4 and top5_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('2.22.4') then 'interestedInBuyingCar'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('2.30.3', '2.30.5') then 'carRepair'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('1.51.3', '1.54.4') then 'default'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('10.19.10') then 'kids0To6'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('8.8.4', '8.8.7') then 'default'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'specialShops'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('8.59.5') then 'sporingEvents'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('8.59.3') then 'concertsEvents'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 4 and top5_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'householdServices'
                
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('2.22.4') then 'interestedInBuyingCar'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('2.30.3', '2.30.5') then 'carRepair'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'carOwners'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('2.27.1') then 'taxiTransfer'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('1.51.4', '1.51.3') then 'default'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'spaBeautySalons'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('10.19.7', '10.19.10') then 'kids0To6'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('5.40.2') then 'kids14To18'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'householdServices'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('8.56.3') then 'sportingEvents'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('8.56.3') then 'concertsEvents'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('6.42.1') then 'shoppingForClothes'
                    when top5_product = 'lifestyleGeneral' and approval_level >= 6 and top5_need_ID in ('8.56.3') then 'concierge'
                    else '' end as top5_lifestyle_sub_situation,
                
                
                         CASE 
                                WHEN top5_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_1, 0))
                                WHEN top5_product = 'okko' THEN round(coalesce(okko_without_premier_level_1, 0))
                                else 0
                                END as top5_benefit_rubles_lvl1,
                                
                                case 
                                WHEN top5_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_1, 0))
                                WHEN top5_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_1, 0))
                                else 0 
                                end as top5_benefit_bonuses_lvl1,
                
                                CASE 
                                WHEN top5_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_2, 0))
                                WHEN top5_product = 'okko' THEN round(coalesce(okko_without_premier_level_2, 0))
                                else 0
                                END as top5_benefit_rubles_lvl2,
                                
                                case 
                                WHEN top5_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_2, 0))
                                WHEN top5_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_2, 0))
                                else 0 
                                end as top5_benefit_bonuses_lvl2,
                
                                CASE 
                                WHEN top5_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_3, 0))
                                WHEN top5_product = 'okko' THEN round(coalesce(okko_without_premier_level_3, 0))
                                else 0
                                END as top5_benefit_rubles_lvl3,
                                
                                case 
                                WHEN top5_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_3, 0))
                                WHEN top5_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_3, 0))
                                else 0
                                end as top5_benefit_bonuses_lvl3,
                
                                CASE 
                                WHEN top5_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_4_profit, 0))
                                WHEN top5_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_4_profit, 0))
                                else 0
                                END as top5_benefit_rubles_lvl4,
                                
                                case 
                                WHEN top5_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_4, 0))
                                WHEN top5_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_4, 0))
                                else 0 
                                end as top5_benefit_bonuses_lvl4,
                
                                CASE 
                                WHEN top5_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top5_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top5_benefit_rubles_lvl5,
                                
                                case
                                WHEN top5_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top5_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top5_benefit_bonuses_lvl5,
                
                                CASE 
                                WHEN top5_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top5_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top5_benefit_rubles_lvl6,
                                
                                case
                                WHEN top5_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top5_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top5_benefit_bonuses_lvl6,
                
                
                case when top6_product is null or top6_product = '' then 'plug'
                        else top6_product end as top6_product,
                        'ecosystem' as top6_type,
                        case when top6_score is null or top6_score = '' then 0
                        else top6_score end as top6_score,
                '' as top6_offer_type, '' as top6_advantages, 0 as top6_is_recommended, 0 as top6_used_by_client,
                            case
                    when top6_product = 'cooper' and top6_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets'
                    when top6_product = 'cooper' and top6_need_ID in ('1.51.3', '1.51.4') then 'sport' 
                    when top6_product = 'cooper' and top6_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty' 
                    when top6_product = 'cooper' and top6_need_ID in ('1.9.1', '1.9.2') then 'healthAndBeauty' 
                    when top6_product = 'cooper' and top6_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top6_product = 'cooper' and top6_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top6_product = 'cooper' and top6_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'entertainmentAndHobbies' 
                    when top6_product = 'cooper' and top6_need_ID in ('7.2.1', '7.2.2') then 'other' 
                    when top6_product = 'cooper' and top6_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other'
                    when top6_product = 'cooper' and top6_need_ID in ('7.2.1', '7.2.2') then 'realEstate' 
                    when top6_product = 'cooper' and top6_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'realEstate'
                    when top6_product = 'cooper' and top6_need_ID in ('7.2.1') then 'default'
                
                
                    when top6_product = 'sberHealth' and top6_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets'
                    when top6_product = 'sberHealth' and top6_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty'
                    when top6_product = 'sberHealth' and top6_need_ID in ('10.17.6') then 'kids'
                    when top6_product = 'sberHealth' and top6_need_ID in ('1.17.2', '1.17.6') then 'default'
                
                    when top6_product = 'citydrive' and top6_need_ID in ('2.25.3', '2.25.4') then 'auto'
                
                    when top6_product = 'sberPravo' and top6_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'auto'
                    when top6_product = 'sberPravo' and top6_need_ID in ('1.6.1') then 'other'
                    when top6_product = 'sberPravo' and top6_need_ID in ('3.4.4', '3.15.5') then 'realEstate'
                    when top6_product = 'sberPravo' and top6_need_ID in ('2.25.3', '2.25.4') then 'auto'
                
                    when top6_product = 'samokat' and top6_need_ID in ('1.51.3', '1.54.4') then 'sport'
                    when top6_product = 'samokat' and top6_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top6_product = 'samokat' and top6_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top6_product = 'samokat' and top6_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top6_product = 'samokat' and top6_need_ID in ('8.56.3') then 'entertainmentAndHobbies'
                    when top6_product = 'samokat' and top6_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'entertainmentAndHobbies'
                    when top6_product = 'samokat' and top6_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other'
                    when top6_product = 'samokat' and top6_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'realEstate'
                    when top6_product = 'samokat' and top6_need_ID in ('7.2.1') then 'default'
                
                    when top6_product = 'okko' and top6_need_ID in ('1.51.3') then 'sport'
                    when top6_product = 'okko' and top6_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top6_product = 'okko' and top6_need_ID in ('5.37.2', '5.38.4') then 'entertainmentAndHobbies'
                    when top6_product = 'okko' and top6_need_ID in ('8.56.3') then 'entertainmentAndHobbies'
                
                    when top6_product = 'zvuk' and top6_need_ID in ('1.51.3') then 'sport'
                    when top6_product = 'zvuk' and top6_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top6_product = 'zls' and top6_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty'
                    when top6_product = 'zls' and top6_need_ID in ('1.17.2', '1.17.6') then 'kids'
                    when top6_product = 'zls' and top6_need_ID in ('3.4.4') then 'trips'
                    when top6_product = 'zls' and top6_need_ID in ('4.31.1') then 'other'
                    when top6_product = 'zls' and top6_need_ID in ('3.26.4') then 'realEstate'
                    when top6_product = 'zls' and top6_need_ID in ('3.4.4') then 'realEstate'
                    when top6_product = 'zls' and top6_need_ID in ('3.15.5', '3.5.1') then 'realEstate'
                    
                    when top6_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'trips'
                
                    when top6_product = 'mriya' and top6_need_ID in ('1.17.6', '1.17.2') then 'healthAndBeauty'
                    when top6_product = 'mriya' and top6_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top6_product = 'sberMobile' and top6_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top6_product = 'sberMobile' and top6_need_ID in ('3.26.4') then 'realEstate'
                
                    when top6_product = 'otello' and top6_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top6_product = 'afisha' and approval_level >= 3 and top6_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top6_product = 'manzherok' and approval_level >= 4 and top6_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('2.22.4') then 'auto'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('2.30.3', '2.30.5') then 'auto'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('1.51.3', '1.54.4') then 'sport'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('10.19.10') then 'kids'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('8.8.4', '8.8.7') then 'trips'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'entertaimentAndHobbies'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('8.59.5') then 'entertaimentAndHobbies'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('8.59.3') then 'entertaimentAndHobbies'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'realEstate'
                
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('2.22.4') then 'auto'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('2.30.3', '2.30.5') then 'auto'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'auto'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('2.27.1') then 'auto'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('1.51.4', '1.51.3') then 'sport'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('5.40.2') then 'kids'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'realEstate'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('6.42.1') then 'entertaimentAndHobbies'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    else '' end as top6_lifestyle_situation,
                
                case
                    when top6_product = 'cooper' and top6_need_ID in ('10.18.5', '10.18.6') then 'default'
                    when top6_product = 'cooper' and top6_need_ID in ('10.18.2') then 'health'
                    when top6_product = 'cooper' and top6_need_ID in ('1.51.3', '1.51.4') then 'default' 
                    when top6_product = 'cooper' and top6_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'beauty' 
                    when top6_product = 'cooper' and top6_need_ID in ('1.9.1', '1.9.2') then 'pharmacy' 
                    when top6_product = 'cooper' and top6_need_ID in ('10.19.7', '10.19.10') then 'kids0To6' 
                    when top6_product = 'cooper' and top6_need_ID in ('8.8.4', '8.8.5') then 'default' 
                    when top6_product = 'cooper' and top6_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'specialShops' 
                    when top6_product = 'cooper' and top6_need_ID in ('7.2.1', '7.2.2') then 'shopsNotFromList' 
                    when top6_product = 'cooper' and top6_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants' 
                    when top6_product = 'cooper' and top6_need_ID in ('7.2.1', '7.2.2') then 'supermarkets' 
                    when top6_product = 'cooper' and top6_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'shoppingForHouse' 
                    when top6_product = 'cooper' and top6_need_ID in ('7.2.1') then 'default' 
                
                
                    when top6_product = 'sberHealth' and top6_need_ID in ('1.9.1', '1.9.2') then 'default'
                    when top6_product = 'sberHealth' and top6_need_ID in ('1.17.2', '1.17.6') then 'laboratory'
                    when top6_product = 'sberHealth' and top6_need_ID in ('10.17.6') then 'kids0To14'
                    when top6_product = 'sberHealth' and top6_need_ID in ('1.17.2', '1.17.6') then 'default'
                
                    when top6_product = 'citydrive' and top6_need_ID in ('2.25.3', '2.25.4') then 'carsharing'
                
                    when top6_product = 'sberPravo' and top6_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'fines'
                    when top6_product = 'sberPravo' and top6_need_ID in ('1.6.1') then 'taxes'
                    when top6_product = 'sberPravo' and top6_need_ID in ('3.4.4', '3.15.5') then 'moreThan2Estates'
                    when top6_product = 'sberPravo' and top6_need_ID in ('2.25.3', '2.25.4') then 'carsharing'
                
                    when top6_product = 'samokat' and top6_need_ID in ('1.51.3', '1.54.4') then 'default'
                    when top6_product = 'samokat' and top6_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default'
                    when top6_product = 'samokat' and top6_need_ID in ('10.19.7', '10.19.10') then 'dids0To6'
                    when top6_product = 'samokat' and top6_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top6_product = 'samokat' and top6_need_ID in ('8.56.3') then 'cinema'
                    when top6_product = 'samokat' and top6_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'books'
                    when top6_product = 'samokat' and top6_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants'
                    when top6_product = 'samokat' and top6_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'orderingFood'
                    when top6_product = 'samokat' and top6_need_ID in ('7.2.1') then 'default'
                
                    when top6_product = 'okko' and top6_need_ID in ('1.51.3') then 'default'
                    when top6_product = 'okko' and top6_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top6_product = 'okko' and top6_need_ID in ('5.37.2', '5.38.4') then 'selfeEducation'
                    when top6_product = 'okko' and top6_need_ID in ('8.56.3') then 'cinema'
                
                    when top6_product = 'zvuk' and top6_need_ID in ('1.51.3') then 'default'
                    when top6_product = 'zvuk' and top6_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top6_product = 'zls' and top6_need_ID in ('1.17.2', '1.17.6') then 'healthInsurance'
                    when top6_product = 'zls' and top6_need_ID in ('1.17.2', '1.17.6') then 'insuranceKids1To14'
                    when top6_product = 'zls' and top6_need_ID in ('3.4.4') then 'realEstate'
                    when top6_product = 'zls' and top6_need_ID in ('4.31.1') then 'financeInsurance'
                    when top6_product = 'zls' and top6_need_ID in ('3.26.4') then 'houseInsurance'
                    when top6_product = 'zls' and top6_need_ID in ('3.4.4') then 'realEstateInsurance'
                    when top6_product = 'zls' and top6_need_ID in ('3.15.5', '3.5.1') then 'houseRepairInsurance'
                    
                    when top6_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'vzr1Level'
                
                    when top6_product = 'mriya' and top6_need_ID in ('1.17.6', '1.17.2') then 'default'
                    when top6_product = 'mriya' and top6_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top6_product = 'sberMobile' and top6_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top6_product = 'sberMobile' and top6_need_ID in ('3.26.4') then 'newSbermobileSubscriber'
                
                    when top6_product = 'otello' and top6_need_ID in ('8.8.4', '8.8.5') then 'hasBonuses'
                
                    when top6_product = 'afisha' and approval_level >= 3 and top6_need_ID in ('8.56.3') then 'tickets'
                
                    when top6_product = 'manzherok' and approval_level >= 4 and top6_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('2.22.4') then 'interestedInBuyingCar'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('2.30.3', '2.30.5') then 'carRepair'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('1.51.3', '1.54.4') then 'default'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('10.19.10') then 'kids0To6'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('8.8.4', '8.8.7') then 'default'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'specialShops'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('8.59.5') then 'sporingEvents'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('8.59.3') then 'concertsEvents'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 4 and top6_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'householdServices'
                
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('2.22.4') then 'interestedInBuyingCar'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('2.30.3', '2.30.5') then 'carRepair'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'carOwners'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('2.27.1') then 'taxiTransfer'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('1.51.4', '1.51.3') then 'default'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'spaBeautySalons'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('10.19.7', '10.19.10') then 'kids0To6'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('5.40.2') then 'kids14To18'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'householdServices'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('8.56.3') then 'sportingEvents'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('8.56.3') then 'concertsEvents'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('6.42.1') then 'shoppingForClothes'
                    when top6_product = 'lifestyleGeneral' and approval_level >= 6 and top6_need_ID in ('8.56.3') then 'concierge'
                    else '' end as top6_lifestyle_sub_situation,
                
                                CASE 
                                WHEN top6_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_1, 0))
                                WHEN top6_product = 'okko' THEN round(coalesce(okko_without_premier_level_1, 0))
                                else 0
                                END as top6_benefit_rubles_lvl1,
                                
                                case 
                                WHEN top6_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_1, 0))
                                WHEN top6_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_1, 0))
                                else 0 
                                end as top6_benefit_bonuses_lvl1,
                
                                CASE 
                                WHEN top6_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_2, 0))
                                WHEN top6_product = 'okko' THEN round(coalesce(okko_without_premier_level_2, 0))
                                else 0
                                END as top6_benefit_rubles_lvl2,
                                
                                case 
                                WHEN top6_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_2, 0))
                                WHEN top6_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_2, 0))
                                else 0 
                                end as top6_benefit_bonuses_lvl2,
                
                                CASE 
                                WHEN top6_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_3, 0))
                                WHEN top6_product = 'okko' THEN round(coalesce(okko_without_premier_level_3, 0))
                                else 0
                                END as top6_benefit_rubles_lvl3,
                                
                                case 
                                WHEN top6_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_3, 0))
                                WHEN top6_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_3, 0))
                                else 0
                                end as top6_benefit_bonuses_lvl3,
                
                                CASE 
                                WHEN top6_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_4_profit, 0))
                                WHEN top6_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_4_profit, 0))
                                else 0
                                END as top6_benefit_rubles_lvl4,
                                
                                case 
                                WHEN top6_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_4, 0))
                                WHEN top6_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_4, 0))
                                else 0 
                                end as top6_benefit_bonuses_lvl4,
                
                                CASE 
                                WHEN top6_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top6_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top6_benefit_rubles_lvl5,
                                
                                case
                                WHEN top6_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top6_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top6_benefit_bonuses_lvl5,
                
                                CASE 
                                WHEN top6_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top6_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top6_benefit_rubles_lvl6,
                                
                                case
                                WHEN top6_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top6_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top6_benefit_bonuses_lvl6,
                
                
                case when top7_product is null or top7_product = '' then 'plug'
                        else top7_product end as top7_product,
                        'ecosystem' as top7_type,
                        case when top7_score is null or top7_score = '' then 0
                        else top7_score end as top7_score, 
                '' as top7_offer_type, '' as top7_advantages, 0 as top7_is_recommended, 0 as top7_used_by_client,
                           case
                    when top7_product = 'cooper' and top7_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets'
                    when top7_product = 'cooper' and top7_need_ID in ('1.51.3', '1.51.4') then 'sport' 
                    when top7_product = 'cooper' and top7_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty' 
                    when top7_product = 'cooper' and top7_need_ID in ('1.9.1', '1.9.2') then 'healthAndBeauty' 
                    when top7_product = 'cooper' and top7_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top7_product = 'cooper' and top7_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top7_product = 'cooper' and top7_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'entertainmentAndHobbies' 
                    when top7_product = 'cooper' and top7_need_ID in ('7.2.1', '7.2.2') then 'other' 
                    when top7_product = 'cooper' and top7_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other'
                    when top7_product = 'cooper' and top7_need_ID in ('7.2.1', '7.2.2') then 'realEstate' 
                    when top7_product = 'cooper' and top7_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'realEstate'
                    when top7_product = 'cooper' and top7_need_ID in ('7.2.1') then 'default'
                
                
                    when top7_product = 'sberHealth' and top7_need_ID in ('10.18.5', '10.18.6', '10.18.2') then 'pets'
                    when top7_product = 'sberHealth' and top7_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty'
                    when top7_product = 'sberHealth' and top7_need_ID in ('10.17.6') then 'kids'
                    when top7_product = 'sberHealth' and top7_need_ID in ('1.17.2', '1.17.6') then 'default'
                
                    when top7_product = 'citydrive' and top7_need_ID in ('2.25.3', '2.25.4') then 'auto'
                
                    when top7_product = 'sberPravo' and top7_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'auto'
                    when top7_product = 'sberPravo' and top7_need_ID in ('1.6.1') then 'other'
                    when top7_product = 'sberPravo' and top7_need_ID in ('3.4.4', '3.15.5') then 'realEstate'
                    when top7_product = 'sberPravo' and top7_need_ID in ('2.25.3', '2.25.4') then 'auto'
                
                    when top7_product = 'samokat' and top7_need_ID in ('1.51.3', '1.54.4') then 'sport'
                    when top7_product = 'samokat' and top7_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top7_product = 'samokat' and top7_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top7_product = 'samokat' and top7_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top7_product = 'samokat' and top7_need_ID in ('8.56.3') then 'entertainmentAndHobbies'
                    when top7_product = 'samokat' and top7_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'entertainmentAndHobbies'
                    when top7_product = 'samokat' and top7_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'other'
                    when top7_product = 'samokat' and top7_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'realEstate'
                    when top7_product = 'samokat' and top7_need_ID in ('7.2.1') then 'default'
                
                    when top7_product = 'okko' and top7_need_ID in ('1.51.3') then 'sport'
                    when top7_product = 'okko' and top7_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top7_product = 'okko' and top7_need_ID in ('5.37.2', '5.38.4') then 'entertainmentAndHobbies'
                    when top7_product = 'okko' and top7_need_ID in ('8.56.3') then 'entertainmentAndHobbies'
                
                    when top7_product = 'zvuk' and top7_need_ID in ('1.51.3') then 'sport'
                    when top7_product = 'zvuk' and top7_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top7_product = 'zls' and top7_need_ID in ('1.17.2', '1.17.6') then 'healthAndBeauty'
                    when top7_product = 'zls' and top7_need_ID in ('1.17.2', '1.17.6') then 'kids'
                    when top7_product = 'zls' and top7_need_ID in ('3.4.4') then 'trips'
                    when top7_product = 'zls' and top7_need_ID in ('4.31.1') then 'other'
                    when top7_product = 'zls' and top7_need_ID in ('3.26.4') then 'realEstate'
                    when top7_product = 'zls' and top7_need_ID in ('3.4.4') then 'realEstate'
                    when top7_product = 'zls' and top7_need_ID in ('3.15.5', '3.5.1') then 'realEstate'
                    
                    when top7_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'trips'
                
                    when top7_product = 'mriya' and top7_need_ID in ('1.17.6', '1.17.2') then 'healthAndBeauty'
                    when top7_product = 'mriya' and top7_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top7_product = 'sberMobile' and top7_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top7_product = 'sberMobile' and top7_need_ID in ('3.26.4') then 'realEstate'
                
                    when top7_product = 'otello' and top7_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top7_product = 'afisha' and approval_level >= 3 and top7_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top7_product = 'manzherok' and approval_level >= 4 and top7_need_ID in ('8.8.4', '8.8.5') then 'trips'
                
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('2.22.4') then 'auto'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('2.30.3', '2.30.5') then 'auto'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('1.51.3', '1.54.4') then 'sport'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('10.19.10') then 'kids'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('8.8.4', '8.8.7') then 'trips'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'entertaimentAndHobbies'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('8.59.5') then 'entertaimentAndHobbies'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('8.59.3') then 'entertaimentAndHobbies'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'realEstate'
                
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('2.22.4') then 'auto'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('2.30.3', '2.30.5') then 'auto'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'auto'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('2.27.1') then 'auto'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('1.51.4', '1.51.3') then 'sport'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'healthAndBeauty'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('10.19.7', '10.19.10') then 'kids'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('5.40.2') then 'kids'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('8.8.4', '8.8.5') then 'trips'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'realEstate'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('6.42.1') then 'entertaimentAndHobbies'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('8.56.3') then 'entertaimentAndHobbies'
                    else '' end as top7_lifestyle_situation,
                
                case
                    when top7_product = 'cooper' and top7_need_ID in ('10.18.5', '10.18.6') then 'default'
                    when top7_product = 'cooper' and top7_need_ID in ('10.18.2') then 'health'
                    when top7_product = 'cooper' and top7_need_ID in ('1.51.3', '1.51.4') then 'default' 
                    when top7_product = 'cooper' and top7_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'beauty' 
                    when top7_product = 'cooper' and top7_need_ID in ('1.9.1', '1.9.2') then 'pharmacy' 
                    when top7_product = 'cooper' and top7_need_ID in ('10.19.7', '10.19.10') then 'kids0To6' 
                    when top7_product = 'cooper' and top7_need_ID in ('8.8.4', '8.8.5') then 'default' 
                    when top7_product = 'cooper' and top7_need_ID in ('8.16.15', '8.16.1', '8.59.6', '8.59.5') then 'specialShops' 
                    when top7_product = 'cooper' and top7_need_ID in ('7.2.1', '7.2.2') then 'shopsNotFromList' 
                    when top7_product = 'cooper' and top7_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants' 
                    when top7_product = 'cooper' and top7_need_ID in ('7.2.1', '7.2.2') then 'supermarkets' 
                    when top7_product = 'cooper' and top7_need_ID in ('3.4.4', '3.5.1', '3.15.5') then 'shoppingForHouse' 
                    when top7_product = 'cooper' and top7_need_ID in ('7.2.1') then 'default' 
                
                
                    when top7_product = 'sberHealth' and top7_need_ID in ('1.9.1', '1.9.2') then 'default'
                    when top7_product = 'sberHealth' and top7_need_ID in ('1.17.2', '1.17.6') then 'laboratory'
                    when top7_product = 'sberHealth' and top7_need_ID in ('10.17.6') then 'kids0To14'
                    when top7_product = 'sberHealth' and top7_need_ID in ('1.17.2', '1.17.6') then 'default'
                
                    when top7_product = 'citydrive' and top7_need_ID in ('2.25.3', '2.25.4') then 'carsharing'
                
                    when top7_product = 'sberPravo' and top7_need_ID in ('2.30.2', '2.30.3', '2.1.5', '2.25.4', '2.30.5') then 'fines'
                    when top7_product = 'sberPravo' and top7_need_ID in ('1.6.1') then 'taxes'
                    when top7_product = 'sberPravo' and top7_need_ID in ('3.4.4', '3.15.5') then 'moreThan2Estates'
                    when top7_product = 'sberPravo' and top7_need_ID in ('2.25.3', '2.25.4') then 'carsharing'
                
                    when top7_product = 'samokat' and top7_need_ID in ('1.51.3', '1.54.4') then 'default'
                    when top7_product = 'samokat' and top7_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default'
                    when top7_product = 'samokat' and top7_need_ID in ('10.19.7', '10.19.10') then 'dids0To6'
                    when top7_product = 'samokat' and top7_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top7_product = 'samokat' and top7_need_ID in ('8.56.3') then 'cinema'
                    when top7_product = 'samokat' and top7_need_ID in ('8.62.1', '5.37.2', '5.38.4') then 'books'
                    when top7_product = 'samokat' and top7_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'cafeAndRestaurants'
                    when top7_product = 'samokat' and top7_need_ID in ('7.12.1', '7.12.4', '7.50.1', '7.50.3') then 'orderingFood'
                    when top7_product = 'samokat' and top7_need_ID in ('7.2.1') then 'default'
                
                    when top7_product = 'okko' and top7_need_ID in ('1.51.3') then 'default'
                    when top7_product = 'okko' and top7_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top7_product = 'okko' and top7_need_ID in ('5.37.2', '5.38.4') then 'selfeEducation'
                    when top7_product = 'okko' and top7_need_ID in ('8.56.3') then 'cinema'
                
                    when top7_product = 'zvuk' and top7_need_ID in ('1.51.3') then 'default'
                    when top7_product = 'zvuk' and top7_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top7_product = 'zls' and top7_need_ID in ('1.17.2', '1.17.6') then 'healthInsurance'
                    when top7_product = 'zls' and top7_need_ID in ('1.17.2', '1.17.6') then 'insuranceKids1To14'
                    when top7_product = 'zls' and top7_need_ID in ('3.4.4') then 'realEstate'
                    when top7_product = 'zls' and top7_need_ID in ('4.31.1') then 'financeInsurance'
                    when top7_product = 'zls' and top7_need_ID in ('3.26.4') then 'houseInsurance'
                    when top7_product = 'zls' and top7_need_ID in ('3.4.4') then 'realEstateInsurance'
                    when top7_product = 'zls' and top7_need_ID in ('3.15.5', '3.5.1') then 'houseRepairInsurance'
                    
                    when top7_product = 'vzr' and approval_level = 1 and top1_need_ID in ('8.8.4', '8.8.7') then 'vzr1Level'
                
                    when top7_product = 'mriya' and top7_need_ID in ('1.17.6', '1.17.2') then 'default'
                    when top7_product = 'mriya' and top7_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top7_product = 'sberMobile' and top7_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top7_product = 'sberMobile' and top7_need_ID in ('3.26.4') then 'newSbermobileSubscriber'
                
                    when top7_product = 'otello' and top7_need_ID in ('8.8.4', '8.8.5') then 'hasBonuses'
                
                    when top7_product = 'afisha' and approval_level >= 3 and top7_need_ID in ('8.56.3') then 'tickets'
                
                    when top7_product = 'manzherok' and approval_level >= 4 and top7_need_ID in ('8.8.4', '8.8.5') then 'default'
                
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('2.22.4') then 'interestedInBuyingCar'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('2.30.3', '2.30.5') then 'carRepair'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('1.51.3', '1.54.4') then 'default'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('6.45.10', '6.44.1', '6.44.2') then 'default'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('10.19.10') then 'kids0To6'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('8.8.4', '8.8.7') then 'default'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('8.16.15', '8.16.1', '8.59.6') then 'specialShops'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('8.59.5') then 'sporingEvents'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('8.59.3') then 'concertsEvents'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 4 and top7_need_ID in ('3.4.4.', '3.15.5', '3.26.4') then 'householdServices'
                
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('2.22.4') then 'interestedInBuyingCar'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('2.30.3', '2.30.5') then 'carRepair'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('2.30.3', '2.30.5', '2.30.2', '2.1.5', '2.25.4') then 'carOwners'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('2.27.1') then 'taxiTransfer'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('1.51.4', '1.51.3') then 'default'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('6.45.10', '6.45.11', '6.44.1', '6.44.2') then 'spaBeautySalons'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('10.19.7', '10.19.10') then 'kids0To6'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('5.40.2') then 'kids14To18'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('8.8.4', '8.8.5') then 'default'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('3.4.4', '3.15.5', '3.26.4') then 'householdServices'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('8.56.3') then 'sportingEvents'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('8.56.3') then 'concertsEvents'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('6.42.1') then 'shoppingForClothes'
                    when top7_product = 'lifestyleGeneral' and approval_level >= 6 and top7_need_ID in ('8.56.3') then 'concierge'
                    else '' end as top7_lifestyle_sub_situation,
                
                
                                CASE 
                                WHEN top7_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_1, 0))
                                WHEN top7_product = 'okko' THEN round(coalesce(okko_without_premier_level_1, 0))
                                else 0
                                END as top7_benefit_rubles_lvl1,
                                
                                case 
                                WHEN top7_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_1, 0))
                                WHEN top7_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_1, 0))
                                else 0 
                                end as top7_benefit_bonuses_lvl1,
                
                                CASE 
                                WHEN top7_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_2, 0))
                                WHEN top7_product = 'okko' THEN round(coalesce(okko_without_premier_level_2, 0))
                                else 0
                                END as top7_benefit_rubles_lvl2,
                                
                                case 
                                WHEN top7_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_2, 0))
                                WHEN top7_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_2, 0))
                                else 0 
                                end as top7_benefit_bonuses_lvl2,
                
                                CASE 
                                WHEN top7_product = 'zvuk' THEN round(coalesce(sound_without_premier_level_3, 0))
                                WHEN top7_product = 'okko' THEN round(coalesce(okko_without_premier_level_3, 0))
                                else 0
                                END as top7_benefit_rubles_lvl3,
                                
                                case 
                                WHEN top7_product = 'cooper' THEN round(coalesce(kuper_without_premier_level_3, 0))
                                WHEN top7_product = 'samokat' THEN round(coalesce(samokat_without_premier_level_3, 0))
                                else 0
                                end as top7_benefit_bonuses_lvl3,
                
                                CASE 
                                WHEN top7_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_4_profit, 0))
                                WHEN top7_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_4_profit, 0))
                                else 0
                                END as top7_benefit_rubles_lvl4,
                                
                                case 
                                WHEN top7_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_4, 0))
                                WHEN top7_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_4, 0))
                                else 0 
                                end as top7_benefit_bonuses_lvl4,
                
                                CASE 
                                WHEN top7_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top7_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top7_benefit_rubles_lvl5,
                                
                                case
                                WHEN top7_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top7_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top7_benefit_bonuses_lvl5,
                
                                CASE 
                                WHEN top7_product = 'zvuk' THEN round(coalesce(sound_without_sber_first_level_5_profit, 0))
                                WHEN top7_product = 'okko' THEN round(coalesce(okko_without_sber_first_level_5_profit, 0))
                                else 0
                                END as top7_benefit_rubles_lvl6,
                                
                                case
                                WHEN top7_product = 'cooper' THEN round(coalesce(kuper_without_sber_first_level_5, 0))
                                WHEN top7_product = 'samokat' THEN round(coalesce(samokat_without_sber_first_level_5, 0))
                                else 0 
                                end as top7_benefit_bonuses_lvl6
                
                
               
                from arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products a
                left join mid_answer using(epk)
                left join {potential_benefit} b on a.epk = b.epk_id
                where report_dt = '{date_now_free}' and death_flag = 0
                )
                
                select distinct * from arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products
                left join answer using(epk)
                ''').write.saveAsTable('arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products_and_romashka', mode='overwrite')

In [28]:
spark.sql('''
select count(epk), count(distinct epk) from arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products_and_romashka
''').show()

+----------+-------------------+
|count(epk)|count(DISTINCT epk)|
+----------+-------------------+
|  14098384|           14098384|
+----------+-------------------+



# Определение жизненной ситуации

In [29]:
model_scores = spark.read.parquet('hdfs://hdfsgw/arnsdpmlexp__bpm_model_romashka-CUSTOM_ROZN_COREML_EXT_SCORES-DM_ROMASHKA_TRNSF_MODEL_SCORES_DESCRIPTION/data/custom/rozn/coreml_ext_scores/pa/dm_romashka_trnsf_model_scores_description')
model_scores.createOrReplaceTempView('need_id')

In [30]:
spark.sql(f"""
WITH
params AS (
  SELECT
    date('{date_now_free}') AS dt0,
    last_day(date('{date_now_free}') - interval 1 month) AS m1_end,
    last_day(date('{date_now_free}') - interval 2 month) AS m2_end,
    last_day(date('{date_now_free}') - interval 3 month) AS m3_end
),
 
base_clients AS (
  SELECT DISTINCT epk
  FROM arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products_and_romashka
),
 
pos_src AS (
  SELECT
    a.epk,
    b.amt AS amt,
    b.mcc_code AS mcc,
    b.day_part AS dt,
    lower(b.merchant_name) AS mn,
    'pos' AS src
  FROM base_clients a
  LEFT JOIN {pos} b
    ON a.epk = b.epk_id
  CROSS JOIN params p
  WHERE b.mcc_code IS NOT NULL
    AND b.ecom_fl = 0
    AND b.is_transaction = 1
    AND b.day_part BETWEEN p.m3_end AND p.dt0
),
 
aq_src AS (
  SELECT
    a.epk,
    b.transaction_ccy_amt AS amt,
    b.merchant_category_code AS mcc,
    b.part_1_day AS dt,
    lower(b.merchant_name) AS mn,
    'aquaring' AS src
  FROM base_clients a
  LEFT JOIN {aquaring} b
    ON a.epk = b.epk_id
  CROSS JOIN params p
  WHERE b.status_order_code = 'PAID'
    AND b.merchant_category_code IS NOT NULL
    AND b.part_1_day BETWEEN p.m3_end AND p.dt0
),
 
tx_all AS (
  SELECT * FROM pos_src
  UNION ALL
  SELECT * FROM aq_src
),
 
rules AS (
  SELECT
    t.*,
 
    -- MCC-флаги
    (
      t.mcc IN ('4582','4112','4411','4722','4011','4511','3042','7011','7032','7033',
                '4723','5309','8999','7277','9399','5999','6300')
      OR t.mcc BETWEEN '3501' AND '3827'
      OR t.mcc BETWEEN '3000' AND '3350'
    ) AS is_trips_mcc,
 
    (t.mcc IN ('5655','5940','5941','7941','7997','7911','7277','7999','8299',
               '5331','5661','5699','7011','7298','5814','7996','9399','8011',
               '4814','7297','5247','7230','5999','5812','3567','5499','7032','7914')
    ) AS is_sport_mcc,
 
    (t.mcc IN ('3991','3990','7922','7832','7999','5815','5399','5814','7011',
               '5812','7991','7929','9399','8299','8999','7911','7933','7994',
               '5945','5462','5441','5947','5971','8220')
    ) AS is_ent_mcc,
 
    (t.mcc IN ('8021','9071','8031','8049','8050','8062','7277','4119','5047',
               '5296','5975','5976','8011','8041','8071','8099','8351','8676','5122',
               '5292','5295','5912','7299')
    ) AS is_health_mcc,
    
    (t.mcc IN ('5511', '7538', '5541', '6300', '5533', '5599', '4784', '9399',
                '6012', '4789', '7523')
    ) AS is_auto_mcc,
 
    (t.mcc IN ('0742', '5995', '0742', '5912', '8011', '8999', '8062', '5411', 
                '7299', '8099', '5992', '5331', '5945', '0742', '5814', '4900', '9402')
    ) AS is_pets_mcc,
 
    (t.mcc IN ('5812', '5499', '5814', '5411')
    ) AS is_cafesAndRestaurants_mcc,
 
    (t.mcc IN ('3990', '5411', '5812')
    ) AS is_purchases_mcc,

 
    (t.mn like '%aviaperevozchik%' or t.mn like '%aeroport%' or t.mn like '%terminal_aeroport%' 
            or t.mn like '%vokzal%' or t.mn like '%kruiz%' or t.mn like '%parom%' or t.mn like '%perevozki%'
            or t.mn like '%avia%' or t.mn like '%otel%' or t.mn like '%camping%' or t.mn like '%kemping%' 
            or t.mn like '%lager%' or t.mn like '%tyr%' or t.mn like '%tur%' or t.mn like '%duty%' 
            or t.mn like '%visa%' or t.mn like '%viza%') AS is_trips_kw,
    (t.mn like '%sport%' or t.mn like '%club%' or t.mn like '%klub%' or t.mn like '%pool%' 
            or t.mn like '%basseyn%' or t.mn like '%dance%' or t.mn like '%tansy%' or t.mn like '%massage%' 
            or t.mn like '%massazh%' or t.mn like '%spa%' or t.mn like '%fitnes%' or t.mn like '%beaut%' 
            or t.mn like '%krasot%') AS is_sport_kw,
    (t.mn like '%ticket%' or t.mn like '%bilet%' or t.mn like '%cinema%' or t.mn like '%kino%' 
            or t.mn like '%teatr%' or t.mn like '%theater%' or t.mn like '%filarmoniya%' 
            or t.mn like '%park%' or t.mn like '%razvlechen%' or t.mn like '%bouling%' 
            or t.mn like '%bowling%' or t.mn like '%game%' or t.mn like '%igra%' or t.mn like '%muzei%' 
            or t.mn like '%museums%' or t.mn like '%vystavki%' or t.mn like '%exhibition%') AS is_ent_kw,
    (t.mn like '%dentist%' or t.mn like '%dantist%' or t.mn like '%doctor%' or t.mn like '%medicine%' 
            or t.mn like '%meditsina%' or t.mn like '%ortoped%' or t.mn like '%orthopedist%' 
            or t.mn like '%sanatoriy%' or t.mn like '%sanatorium%' or t.mn like '%hospital%' 
            or t.mn like '%bolnitsa%' or t.mn like '%atler%' or t.mn like '%apteka%' or t.mn like '%pharmacy%'
            or t.mn like '%analiz%' or t.mn like '%test%') AS is_health_kw,
    (t.mn like '%avto%' or t.mn like '%diler%' or t.mn like '%major%' or t.mn like '%rolf%' or t.mn like '%avilon%' 
            or t.mn like '%borishof%' or t.mn like '%service%' or t.mn like '%techcenter%' or t.mn like '%diagnosti%' 
            or t.mn like '%shinomont%' or t.mn like '%kuzov%' or t.mn like '%pokras%' or t.mn like '%parking%' 
            or t.mn like '%avtodor%' or t.mn like '%strahov%' or t.mn like '%insurance%' or t.mn like '%osago%' 
            or t.mn like '%kasko%' or t.mn like '%ingosstrakh%' or t.mn like '%alfastrah%' or t.mn like '%sogaz%' 
            or t.mn like '%vsk%' or t.mn like '%renessans%' or t.mn like '%azs%' or t.mn like '%gas%' or t.mn like '%fuel%'
            or t.mn like '%petrol%' or t.mn like '%gasoline%' or t.mn like '%diesel%' or t.mn like '%refuel%' 
            or t.mn like '%gazpromneft%' or t.mn like '%lukoil%' or t.mn like '%rosneft%') AS is_auto_kw,
    (t.mn like '%zoo%' or t.mn like '%vet%' or t.mn like '%soo%' or t.mn like '%pet%') as is_pets_kw,
    (t.mn like '%cofe%' or t.mn like '%cafe%' or t.mn like '%kantata%' or t.mn like '%drinkit%' 
            or t.mn like '%cofemani%' or t.mn like '%surf%' or t.mn like '%cofix%' or t.mn like '%stars%'
            or t.mn like '%coffee%' or t.mn like '%skuratov%' or t.mn like '%vostochn%') as is_cafesAndRestaurants_kw,
    (t.mn like '%yandex%' or t.mn like '%lavka%' or t.mn like '%lavca%' or t.mn like '%lafka%' 
            or t.mn like '%lafca%') AS is_purchases_kw
 
  FROM tx_all t
),
 
tx_cat AS (
  SELECT
    epk, src, amt, dt,
    CASE
      WHEN is_trips_mcc  AND is_trips_kw  THEN 'trips'
      WHEN is_sport_mcc  AND is_sport_kw  THEN 'sport'
      WHEN is_ent_mcc    AND is_ent_kw    THEN 'entertainmentAndHobbies'
      WHEN is_health_mcc AND is_health_kw THEN 'health'
      WHEN is_auto_mcc AND is_auto_kw THEN 'auto'
      WHEN is_pets_mcc AND is_pets_kw THEN 'pets'
      WHEN is_cafesAndRestaurants_mcc AND is_cafesAndRestaurants_kw THEN 'cafesAndRestaurants'
      WHEN is_purchases_mcc AND is_purchases_kw THEN 'purchases'
      ELSE 'other'
    END AS life_sphere
  FROM rules
  WHERE
    (is_trips_mcc  AND is_trips_kw)
 OR (is_sport_mcc  AND is_sport_kw)
 OR (is_ent_mcc    AND is_ent_kw)
 OR (is_health_mcc AND is_health_kw)
 OR (is_auto_mcc AND is_auto_kw)
 OR (is_pets_mcc AND is_pets_kw)
 OR (is_cafesAndRestaurants_mcc AND is_cafesAndRestaurants_kw)
 OR (is_purchases_mcc AND is_purchases_kw)
),
 
tx_bucketed AS (
  SELECT
    t.*,
    CASE
      WHEN t.dt >  p.m1_end AND t.dt <= p.dt0 THEN 1
      WHEN t.dt >  p.m2_end AND t.dt <= p.m1_end THEN 2
      WHEN t.dt >  p.m3_end AND t.dt <= p.m2_end THEN 3
      ELSE NULL
    END AS m_bucket
  FROM tx_cat t
  CROSS JOIN params p
),
 
agg_by_src AS (
  SELECT
    epk,
    src,
 
    -- counts
    SUM(CASE WHEN life_sphere='sport' THEN 1 ELSE 0 END) AS sport_tx_cnt,
    SUM(CASE WHEN life_sphere='health' THEN 1 ELSE 0 END) AS health_tx_cnt,
    SUM(CASE WHEN life_sphere='trips' THEN 1 ELSE 0 END) AS trips_tx_cnt,
    SUM(CASE WHEN life_sphere='entertainmentAndHobbies' THEN 1 ELSE 0 END) AS ent_tx_cnt,
    SUM(CASE WHEN life_sphere='auto' THEN 1 ELSE 0 END) AS auto_tx_cnt,
    SUM(CASE WHEN life_sphere='pets' THEN 1 ELSE 0 END) AS pets_tx_cnt,
    SUM(CASE WHEN life_sphere='cafesAndRestaurants' THEN 1 ELSE 0 END) AS cafe_tx_cnt,
    SUM(CASE WHEN life_sphere='purchases' THEN 1 ELSE 0 END) AS purchases_tx_cnt,
 
    -- amounts
    SUM(CASE WHEN life_sphere='sport' THEN amt ELSE 0 END) AS sport_amt,
    SUM(CASE WHEN life_sphere='health' THEN amt ELSE 0 END) AS health_amt,
    SUM(CASE WHEN life_sphere='trips' THEN amt ELSE 0 END) AS trips_amt,
    SUM(CASE WHEN life_sphere='entertainmentAndHobbies' THEN amt ELSE 0 END) AS ent_amt,
    SUM(CASE WHEN life_sphere='auto' THEN amt ELSE 0 END) AS auto_amt,
    SUM(CASE WHEN life_sphere='pets' THEN amt ELSE 0 END) AS pets_amt,
    SUM(CASE WHEN life_sphere='cafesAndRestaurants' THEN amt ELSE 0 END) AS cafe_amt,
    SUM(CASE WHEN life_sphere='purchases' THEN amt ELSE 0 END) AS purchases_amt,
 
    -- totals (для долей)
    COUNT(1) AS total_tx_cnt,
    SUM(amt) AS total_amt
 
  FROM tx_cat
  GROUP BY epk, src
),
 
agg_all AS (
  SELECT
    epk,
 
    -- counts sums
    SUM(sport_tx_cnt)  AS sport_tx_cnt,
    SUM(health_tx_cnt) AS health_tx_cnt,
    SUM(trips_tx_cnt)  AS trips_tx_cnt,
    SUM(ent_tx_cnt)    AS ent_tx_cnt,
    SUM(auto_tx_cnt)   AS auto_tx_cnt,
    SUM(pets_tx_cnt)   AS pets_tx_cnt,
    SUM(cafe_tx_cnt)   AS cafe_tx_cnt,
    SUM(purchases_tx_cnt)   AS purchases_tx_cnt,
 
    -- amounts sums
    SUM(sport_amt)  AS sport_amt,
    SUM(health_amt) AS health_amt,
    SUM(trips_amt)  AS trips_amt,
    SUM(ent_amt)    AS ent_amt,
    SUM(auto_amt)    AS auto_amt,
    SUM(pets_amt)    AS pets_amt,
    SUM(cafe_amt)    AS cafe_amt,
    SUM(purchases_amt)    AS purchases_amt,
 
    -- totals sums
    SUM(total_tx_cnt) AS total_tx_cnt,
    SUM(total_amt)    AS total_amt
  FROM agg_by_src
  GROUP BY epk
),
 
flags AS (
  SELECT
    epk,
 
    MAX(CASE WHEN life_sphere='trips' AND amt>0 AND m_bucket=3 THEN 1 ELSE 0 END) AS flag_buy_trips_3month_ago,
    MAX(CASE WHEN life_sphere='trips' AND amt>0 AND m_bucket=2 THEN 1 ELSE 0 END) AS flag_buy_trips_2month_ago,
    MAX(CASE WHEN life_sphere='trips' AND amt>0 AND m_bucket=1 THEN 1 ELSE 0 END) AS flag_buy_trips_1month_ago,
 
    MAX(CASE WHEN life_sphere='sport' AND amt>0 AND m_bucket=3 THEN 1 ELSE 0 END) AS flag_buy_sport_3month_ago,
    MAX(CASE WHEN life_sphere='sport' AND amt>0 AND m_bucket=2 THEN 1 ELSE 0 END) AS flag_buy_sport_2month_ago,
    MAX(CASE WHEN life_sphere='sport' AND amt>0 AND m_bucket=1 THEN 1 ELSE 0 END) AS flag_buy_sport_1month_ago,
 
    MAX(CASE WHEN life_sphere='health' AND amt>0 AND m_bucket=3 THEN 1 ELSE 0 END) AS flag_buy_health_3month_ago,
    MAX(CASE WHEN life_sphere='health' AND amt>0 AND m_bucket=2 THEN 1 ELSE 0 END) AS flag_buy_health_2month_ago,
    MAX(CASE WHEN life_sphere='health' AND amt>0 AND m_bucket=1 THEN 1 ELSE 0 END) AS flag_buy_health_1month_ago,
 
    MAX(CASE WHEN life_sphere='entertainmentAndHobbies' AND amt>0 AND m_bucket=3 THEN 1 ELSE 0 END) AS flag_buy_ent_3month_ago,
    MAX(CASE WHEN life_sphere='entertainmentAndHobbies' AND amt>0 AND m_bucket=2 THEN 1 ELSE 0 END) AS flag_buy_ent_2month_ago,
    MAX(CASE WHEN life_sphere='entertainmentAndHobbies' AND amt>0 AND m_bucket=1 THEN 1 ELSE 0 END) AS flag_buy_ent_1month_ago,
    
    MAX(CASE WHEN life_sphere='auto' AND amt>0 AND m_bucket=3 THEN 1 ELSE 0 END) AS flag_buy_auto_3month_ago,
    MAX(CASE WHEN life_sphere='auto' AND amt>0 AND m_bucket=2 THEN 1 ELSE 0 END) AS flag_buy_auto_2month_ago,
    MAX(CASE WHEN life_sphere='auto' AND amt>0 AND m_bucket=1 THEN 1 ELSE 0 END) AS flag_buy_auto_1month_ago,
 
    MAX(CASE WHEN life_sphere='pets' AND amt>0 AND m_bucket=3 THEN 1 ELSE 0 END) AS flag_buy_pets_3month_ago,
    MAX(CASE WHEN life_sphere='pets' AND amt>0 AND m_bucket=2 THEN 1 ELSE 0 END) AS flag_buy_pets_2month_ago,
    MAX(CASE WHEN life_sphere='pets' AND amt>0 AND m_bucket=1 THEN 1 ELSE 0 END) AS flag_buy_pets_1month_ago,
 
    MAX(CASE WHEN life_sphere='cafesAndRestaurants' AND amt>0 AND m_bucket=3 THEN 1 ELSE 0 END) AS flag_buy_cafe_3month_ago,
    MAX(CASE WHEN life_sphere='cafesAndRestaurants' AND amt>0 AND m_bucket=2 THEN 1 ELSE 0 END) AS flag_buy_cafe_2month_ago,
    MAX(CASE WHEN life_sphere='cafesAndRestaurants' AND amt>0 AND m_bucket=1 THEN 1 ELSE 0 END) AS flag_buy_cafe_1month_ago,
 
    MAX(CASE WHEN life_sphere='purchases' AND amt>0 AND m_bucket=3 THEN 1 ELSE 0 END) AS flag_buy_purchases_3month_ago,
    MAX(CASE WHEN life_sphere='purchases' AND amt>0 AND m_bucket=2 THEN 1 ELSE 0 END) AS flag_buy_purchases_2month_ago,
    MAX(CASE WHEN life_sphere='purchases' AND amt>0 AND m_bucket=1 THEN 1 ELSE 0 END) AS flag_buy_purchases_1month_ago
 
  FROM tx_bucketed
  GROUP BY epk
),
 
score_names AS (
  SELECT
    a.epk,
 
    COALESCE(a.sport_tx_cnt  / NULLIF(a.total_tx_cnt, 0), 0) AS sport_tx_cnt_share,
    COALESCE(a.sport_amt     / NULLIF(a.total_amt, 0), 0)    AS sport_amt_share,
    CASE
      WHEN (f.flag_buy_sport_3month_ago=1 AND f.flag_buy_sport_2month_ago=1)
        OR (f.flag_buy_sport_3month_ago=1 AND f.flag_buy_sport_1month_ago=1)
        OR (f.flag_buy_sport_2month_ago=1 AND f.flag_buy_sport_1month_ago=1)
      THEN 1 ELSE 0 END AS sport_regular_flag,
 
    COALESCE(a.health_tx_cnt / NULLIF(a.total_tx_cnt, 0), 0) AS health_tx_cnt_share,
    COALESCE(a.health_amt    / NULLIF(a.total_amt, 0), 0)    AS health_amt_share,
    CASE
      WHEN (f.flag_buy_health_3month_ago=1 AND f.flag_buy_health_2month_ago=1)
        OR (f.flag_buy_health_3month_ago=1 AND f.flag_buy_health_1month_ago=1)
        OR (f.flag_buy_health_2month_ago=1 AND f.flag_buy_health_1month_ago=1)
      THEN 1 ELSE 0 END AS health_regular_flag,
 
    COALESCE(a.trips_tx_cnt  / NULLIF(a.total_tx_cnt, 0), 0) AS trips_tx_cnt_share,
    COALESCE(a.trips_amt     / NULLIF(a.total_amt, 0), 0)    AS trips_amt_share,
    CASE
      WHEN (f.flag_buy_trips_3month_ago=1 AND f.flag_buy_trips_2month_ago=1)
        OR (f.flag_buy_trips_3month_ago=1 AND f.flag_buy_trips_1month_ago=1)
        OR (f.flag_buy_trips_2month_ago=1 AND f.flag_buy_trips_1month_ago=1)
      THEN 1 ELSE 0 END AS trips_regular_flag,
 
    COALESCE(a.ent_tx_cnt    / NULLIF(a.total_tx_cnt, 0), 0) AS ent_tx_cnt_share,
    COALESCE(a.ent_amt       / NULLIF(a.total_amt, 0), 0)    AS ent_amt_share,
    CASE
      WHEN (f.flag_buy_ent_3month_ago=1 AND f.flag_buy_ent_2month_ago=1)
        OR (f.flag_buy_ent_3month_ago=1 AND f.flag_buy_ent_1month_ago=1)
        OR (f.flag_buy_ent_2month_ago=1 AND f.flag_buy_ent_1month_ago=1)
      THEN 1 ELSE 0 END AS ent_regular_flag,
    
        COALESCE(a.auto_tx_cnt    / NULLIF(a.total_tx_cnt, 0), 0) AS auto_tx_cnt_share,
    COALESCE(a.auto_amt       / NULLIF(a.total_amt, 0), 0)    AS auto_amt_share,
    CASE
      WHEN (f.flag_buy_auto_3month_ago=1 AND f.flag_buy_auto_2month_ago=1)
        OR (f.flag_buy_auto_3month_ago=1 AND f.flag_buy_auto_1month_ago=1)
        OR (f.flag_buy_auto_2month_ago=1 AND f.flag_buy_auto_1month_ago=1)
      THEN 1 ELSE 0 END AS auto_regular_flag,
 
    COALESCE(a.pets_tx_cnt    / NULLIF(a.total_tx_cnt, 0), 0) AS pets_tx_cnt_share,
    COALESCE(a.pets_amt       / NULLIF(a.total_amt, 0), 0)    AS pets_amt_share,
    CASE
      WHEN (f.flag_buy_pets_3month_ago=1 AND f.flag_buy_pets_2month_ago=1)
        OR (f.flag_buy_pets_3month_ago=1 AND f.flag_buy_pets_1month_ago=1)
        OR (f.flag_buy_pets_2month_ago=1 AND f.flag_buy_pets_1month_ago=1)
      THEN 1 ELSE 0 END AS pets_regular_flag,
 
    COALESCE(a.cafe_tx_cnt    / NULLIF(a.total_tx_cnt, 0), 0) AS cafe_tx_cnt_share,
    COALESCE(a.cafe_amt       / NULLIF(a.total_amt, 0), 0)    AS cafe_amt_share,
    CASE
      WHEN (f.flag_buy_cafe_3month_ago=1 AND f.flag_buy_cafe_2month_ago=1)
        OR (f.flag_buy_cafe_3month_ago=1 AND f.flag_buy_cafe_1month_ago=1)
        OR (f.flag_buy_cafe_2month_ago=1 AND f.flag_buy_cafe_1month_ago=1)
      THEN 1 ELSE 0 END AS cafe_regular_flag,
 
    COALESCE(a.purchases_tx_cnt    / NULLIF(a.total_tx_cnt, 0), 0) AS purchases_tx_cnt_share,
    COALESCE(a.purchases_amt       / NULLIF(a.total_amt, 0), 0)    AS purchases_amt_share,
    CASE
      WHEN (f.flag_buy_purchases_3month_ago=1 AND f.flag_buy_purchases_2month_ago=1)
        OR (f.flag_buy_purchases_3month_ago=1 AND f.flag_buy_purchases_1month_ago=1)
        OR (f.flag_buy_purchases_2month_ago=1 AND f.flag_buy_purchases_1month_ago=1)
      THEN 1 ELSE 0 END AS purchases_regular_flag

 
  FROM agg_all a
  LEFT JOIN flags f
    ON a.epk = f.epk
),

transaction_eligibility as (
    select
        epk,
        case 
            when (
                (coalesce(sport_tx_cnt, 0) >= x
                    or coalesce(sport_amt, 0) >= x)
                or (coalesce(health_tx_cnt, 0) >= x
                    and coalesce(health_amt, 0) >= x)
                or (coalesce(trips_tx_cnt, 0) >= x
                    or coalesce(trips_amt, 0) >= x)
                or (coalesce(ent_tx_cnt, 0) >= x
                    or coalesce(ent_amt, 0) >= x)
            ) then 1
            else 0
        end as has_minimum_transactions
    from agg_all 
),
 
scores AS (
  SELECT
    epk,
    ent_amt_share   * 0.5 + ent_tx_cnt_share   * 0.3 + ent_regular_flag   * 0.2 AS entertainmentAndHobbies_score,
    trips_amt_share * 0.5 + trips_tx_cnt_share * 0.3 + trips_regular_flag * 0.2 AS trips_score,
    health_amt_share* 0.5 + health_tx_cnt_share* 0.3 + health_regular_flag* 0.2 AS health_score,
    sport_amt_share * 0.5 + sport_tx_cnt_share * 0.3 + sport_regular_flag * 0.2 AS sport_score,
    auto_amt_share  * 0.5 + auto_tx_cnt_share  * 0.3 + auto_regular_flag  * 0.2 as auto_score,
    pets_amt_share  * 0.5 + pets_tx_cnt_share  * 0.3 + pets_regular_flag  * 0.2 as pets_score,
    cafe_amt_share  * 0.5 + cafe_tx_cnt_share  * 0.3 + cafe_regular_flag  * 0.2 as cafe_score,
    purchases_amt_share  * 0.5 + purchases_tx_cnt_share  * 0.3 + purchases_regular_flag  * 0.2 as purchases_score
  FROM score_names
),
 
unpivoted AS (
  SELECT
    epk,
    stack(8,
      'спорт', sport_score,
      'здоровье', health_score,
      'путешествие', trips_score,
      'развлечение', entertainmentAndHobbies_score,
      'авто', auto_score,
      'животные', pets_score,
      'кафе', cafe_score,
      'самокат', purchases_score
    ) AS (category, original_score)
  FROM scores
),
 
top2 AS (
  SELECT
    epk, category, original_score,
    row_number() over (partition by epk order by original_score desc) AS rn
  FROM unpivoted
),
 
top2_filtered AS (
  SELECT epk, category, original_score
  FROM top2
  WHERE rn <= 2
),
 
client_tags AS (
  SELECT
    t.epk,
    tg.nsi_id AS tag_id,
    CASE
      WHEN tg.nsi_id in ('10024','10026','12860','33093','10036','10035','33174','10034','15875','10033',
                        '10045','12854','10039','33095','33094','10052','10032','10025','10041','10046',
                        '10037','10038','10044','10050','10047','10040','10048','10042','10031','10062',
                        '12891','10051','15932','15878','10054','15874','10049','15876','12874','15882',
                        '29910','15879','32226','10056','32246','33106','15877','32216','12856','32243',
                        '33105','14457','15933','38379','33108','15884','33107','38380','40875','15934',
                        '14459','10067','10095','10564','21929','21928','33936','21936','21930','21937','21934')
        THEN 'развлечение'
      WHEN tg.nsi_id in ('10440','11826','10441','27183','33926','10881','11825','27180','33933','10282',
                        '22502','15962','38899','33929','33184','38898')
        THEN 'здоровье'
      WHEN tg.nsi_id in ('33538','10139','10967','22486','20127','20120','20092','20098','10140','15852',
                        '15854','15865','15861','15856','15853','15859','15851','15869','15857')
        THEN 'спорт'
      WHEN tg.nsi_id in ('10227','33524','10609','10231','10597','10582','10588','10584','10583','10581',
                        '10576','10432','22631','10594','12935','10585','10577','10613','20886','10579',
                        '10578','12920','10580','22630','10229','12911','22094','12914','12929','12906',
                        '22089','12917','12904','22091','10228')
        THEN 'путешествие'
      WHEN tg.nsi_id in ('10283', '10292', '10610', '10611', '10612', '10615', '10616', '10617', '10886',
                        '14472', '15846', '15894', '22230', '22231', '22232', '22233', '22234', '22235',
                        '33182', '33505', '33511', '33516', '33517', '34204', '34205', '39047', '39048',
                        '39049', '39594', '39595', '39596')
        THEN 'авто'
      WHEN tg.nsi_id in ('33551', '39157', '39159')
        THEN 'животные'
      ELSE 'other'
    END AS category_name
  FROM top2_filtered t
  LEFT JOIN {tags} tg
    ON t.epk = tg.epk_id 
    where tg.is_deleted = 0 and tg.row_actual_to_dt = '9999-12-31'
),
 
top2_with_tags AS (
  SELECT
    t.epk,
    t.category,
    t.original_score,
    LEAST(
      COUNT(DISTINCT CASE WHEN ct.category_name = t.category THEN ct.tag_id END) * 0.05,
      t.original_score * 0.3
    ) AS tag_bonus
  FROM top2_filtered t
  LEFT JOIN client_tags ct
    ON t.epk = ct.epk
  GROUP BY t.epk, t.category, t.original_score
),
 
need_id_and_situation AS (
  SELECT need_ID, life_situation, model_id, need
  FROM need_id
  WHERE model_name like '%d30'
    AND need_ID in ('8.8.4','8.8.5','8.8.7','1.51.3','7.2.1', '7.2.2', '7.50.1', '7.50.3', '10.18.5', '10.18.6', '10.18.2')
),
 
epk_and_situation AS (
  SELECT
    epk_id AS epk,
    need_ID,
    life_situation,
    need,
    raw_score
  FROM need_id_and_situation
  INNER JOIN {romashka_hist} USING(model_id)
  CROSS JOIN params p
  WHERE scoring_dt BETWEEN p.m2_end AND p.dt0
    AND raw_score >= 70
),
 
romashka AS (
  SELECT
    bc.epk,
    coalesce(max(es.raw_score), 0) AS propensity_score
  FROM base_clients bc
  LEFT JOIN epk_and_situation es
    ON bc.epk = es.epk
  group by bc.epk
),
 
top2_with_propensity AS (
  SELECT
    t.epk,
    t.category,
    t.original_score,
    t.tag_bonus,
    COALESCE(r.propensity_score, 0) AS propensity_score
  FROM top2_with_tags t
  LEFT JOIN romashka r
    ON t.epk = r.epk
),
 
scored AS (
  SELECT
    epk,
    category,
    original_score,
    tag_bonus,
    propensity_score,
 
    ABS(first_value(original_score) OVER w - last_value(original_score) OVER w) * 100
      / NULLIF(first_value(original_score) OVER w, 0) AS diff_base,
 
    ABS(first_value(original_score + tag_bonus) OVER w - last_value(original_score + tag_bonus) OVER w) * 100
      / NULLIF(first_value(original_score + tag_bonus) OVER w, 0) AS diff_with_tags,
 
    original_score
      + CASE
          WHEN ABS(first_value(original_score) OVER w - last_value(original_score) OVER w) * 100
               / NULLIF(first_value(original_score) OVER w, 0) < 10
          THEN tag_bonus ELSE 0
        END
      + CASE
          WHEN ABS(first_value(original_score) OVER w - last_value(original_score) OVER w) * 100
               / NULLIF(first_value(original_score) OVER w, 0) < 10
           AND ABS(first_value(original_score + tag_bonus) OVER w - last_value(original_score + tag_bonus) OVER w) * 100
               / NULLIF(first_value(original_score + tag_bonus) OVER w, 0) < 10
          THEN propensity_score * 0.1 ELSE 0
        END AS final_score
  FROM top2_with_propensity
  WINDOW w AS (
    PARTITION BY epk
    ORDER BY original_score DESC
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
  )
),
 
ranked_final AS (
  SELECT
    epk,
    category,
    final_score,
    diff_base,
    diff_with_tags,
    row_number() OVER (PARTITION BY epk ORDER BY final_score DESC) AS rn
  FROM scored
),
 
answer AS (
  SELECT
    epk,
     case when has_minimum_transactions = 1 and category = 'спорт' then 'sport'
        when has_minimum_transactions = 1 and category = 'здоровье' then 'health'
        when has_minimum_transactions = 1 and category = 'путешествие' then 'trips'
        when has_minimum_transactions = 1 and category = 'развлечение' then 'entertainmentAndHobbies'
        WHEN category = 'авто' THEN 'auto'
        when category = 'животные' THEN 'pets'
        when category = 'кафе' then 'cafesAndRestaurants'
        when category = 'самокат' then 'purchases'
    else 'general' end as lifestyle_situation
  FROM ranked_final
  left join transaction_eligibility using(epk)
  WHERE rn = 1
)
 
SELECT
  b.*, coalesce(a.lifestyle_situation, 'general') as lifestyle_situation
FROM arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products_and_romashka b
LEFT JOIN answer a on b.epk = a.epk
""").write.saveAsTable("arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products_and_romashka_and_js_and", mode="overwrite")

# Определение под ЖС

In [31]:
model_scores = spark.read.parquet('hdfs://hdfsgw/arnsdpmlexp__bpm_model_romashka-CUSTOM_ROZN_COREML_EXT_SCORES-DM_ROMASHKA_TRNSF_MODEL_SCORES_DESCRIPTION/data/custom/rozn/coreml_ext_scores/pa/dm_romashka_trnsf_model_scores_description')
model_scores.createOrReplaceTempView('need_id')

In [32]:
spark.conf.set("spark.sql.shuffle.partitions", "400")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
 
TMP_DB = "arnsdpsbx_team_ss"
 
ALL_MCC = (
    "'7032','5651','7542','7538','7535','7534',"
    "'7298','7230','5812','8099','8050','8011','5995','5814',"
    "'5977','8041','5976','7399','5211','5331','7011','7996','8299','9399','4814','7297','5247','5999',"
    "'7911','7277','7999','7997','7941','3567','5499',"
    "'7922','7991','3990','7832','4899',"
    "'4119','5047','5296','5975','5976','8021','8031','8049','8050','8062','8071','8099','8351','8676',"
    "'5122','5292','5295','7299','5912'"
)
 
ALL_NSI = (
    "'10245','10252','10253','11846','11845','11847','11844','11848','11843','11849',"
    "'11842','11850','11841','10251','11840','11837','11839','11836','10250','11838',"
    "'11835','11834','10249','41621','10228','22631','10139','10967','22486','20127',"
    "'20120','20092','20098','33538','15852','15854','15865','15861','15856','15853',"
    "'15859','15851','15869','15857','20544','15880','10067','10095','10564','21929',"
    "'21928','33936','21936','21930','21937','21934','10024','10026','12860','33093',"
    "'10036','10035','33174','10034','15875','10033','10045','12854','10039','33095',"
    "'33094','10052','10032','10025','10041','10046','10037','10038','10044','10050',"
    "'10047','10040','10048','10042','10031','10062','12891','10051','15932','15878',"
    "'10054','15874','10049','15876','12874','15882','29910','15879','32226','10056',"
    "'32246','33106','15877','32216','12856','32243','33105','14457','15933','38379',"
    "'33108','15884','33107','38380','40875','15934','14459'"
)
 
dt0 = date_now_free 
 
spark.sql(f"""
SELECT
  a.epk,
  b.amt,
  b.mcc_code,
  lower(b.merchant_name) AS mn
FROM arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products_and_romashka_and_js_and a
INNER JOIN {pos} b ON a.epk = b.epk_id
WHERE b.mcc_code IS NOT NULL
  AND b.mcc_code IN ({ALL_MCC})
  AND b.ecom_fl = 0
  AND b.is_transaction = 1
  AND b.day_part BETWEEN last_day(date('{dt0}') - interval 3 month) AND date('{dt0}')
 
UNION ALL
 
SELECT
  a.epk,
  b.transaction_ccy_amt AS amt,
  b.merchant_category_code AS mcc_code,
  lower(b.merchant_name) AS mn
FROM arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products_and_romashka_and_js_and a
INNER JOIN {aquaring} b ON a.epk = b.epk_id
WHERE b.status_order_code = 'PAID'
  AND b.merchant_category_code IS NOT NULL
  AND b.merchant_category_code IN ({ALL_MCC})
  AND b.part_1_day BETWEEN last_day(date('{dt0}') - interval 3 month) AND date('{dt0}')
""").write.mode("overwrite").saveAsTable(f"{TMP_DB}.tmp_tx_all")
 
print("Шаг 1: tx_all готов")
 
spark.sql(f"""
SELECT
  epk,
  amt,
  CASE
    WHEN mcc_code = '7032' AND mn LIKE '%ski%' THEN 'sport'
 
    WHEN mcc_code = '5651' AND (mn LIKE '%family%' OR mn LIKE '%semia%') THEN 'family'
 
    WHEN mcc_code IN ('7542','7538','7535','7534')
      AND (mn LIKE '%avto%' OR mn LIKE '%moyka%' OR mn LIKE '%cto%'
        OR mn LIKE '%sto%' OR mn LIKE '%pokras%' OR mn LIKE '%shino%') THEN 'auto'
 
    WHEN mcc_code IN ('7298','7230','5812','8099','8050','8011','5995','5814','5977','8041','5976','7399','5211','5331',
                      '7011','7996','8299','9399','4814','7297','5247','5999','7911','7277','7999','7997')
      AND (mn LIKE '%beauty%' OR mn LIKE '%massage%' OR mn LIKE '%dance%'
        OR mn LIKE '%spa%' OR mn LIKE '%krasot%' OR mn LIKE '%pool%' OR mn LIKE '%basse%') THEN 'beauty'
 
    WHEN mcc_code IN ('7997','8299') AND mn LIKE '%fitmost%' THEN 'hasFitmost'
 
    WHEN mcc_code IN ('7997','5812','7941','8011','3567','5499','7298','7032','5814')
      AND (mn LIKE '%fitnes%' OR mn LIKE '%sport%' OR mn LIKE '%club%') THEN 'fitness'
 
    WHEN mcc_code IN ('7922','7991','3990','9399','5814','5812')
      AND (mn LIKE '%theat%' OR mn LIKE '%teatr%' OR mn LIKE '%bufet%') THEN 'theaters'
 
    WHEN mcc_code IN ('7832','5814','3991','7922','7011','5812','4899')
      AND (mn LIKE '%kino%' OR mn LIKE '%cinema%' OR mn LIKE '%okko%' OR mn LIKE '%ticket%') THEN 'cinema'
 
    WHEN mcc_code IN ('4119','5047','5296','5975','5976','8011','8021','8031','8049','8050','8062','8071','8099','8351','8676')
      AND (mn LIKE '%medical%' OR mn LIKE '%center%' OR mn LIKE '%medcenter%') THEN 'medical'
 
    WHEN mcc_code IN ('4119','5047','5296','5975','5976','8011','8021','8031','8049','8050','8062','8071','8099','8351','8676','5122','5292','5295')
      AND (mn LIKE '%medical%' OR mn LIKE '%center%' OR mn LIKE '%medcenter%' OR mn LIKE '%aptek%') THEN 'kids'
 
    WHEN mcc_code IN ('8062','8071','8011','7299','5912','8099')
      AND (mn LIKE '%laborat%' OR mn LIKE '%analyz%') THEN 'laboratory'
 
  END AS sub_life_situation
FROM {TMP_DB}.tmp_tx_all
WHERE mcc_code IN ({ALL_MCC})
""").where("sub_life_situation IS NOT NULL") \
  .write.mode("overwrite").saveAsTable(f"{TMP_DB}.tmp_tx_classified")
 
print("Шаг 2: tx_classified готов")

Шаг 1: tx_all готов


Шаг 2: tx_classified готов


In [33]:
spark.sql(f"""
WITH tx_with_life AS (
  SELECT
    t.epk,
    t.amt,
    t.sub_life_situation,
    CASE t.sub_life_situation
      WHEN 'sport'      THEN 'trips'
      WHEN 'family'     THEN 'trips'
      WHEN 'auto'       THEN 'trips'
      WHEN 'beauty'     THEN 'sport'
      WHEN 'hasFitmost' THEN 'sport'
      WHEN 'fitness'    THEN 'sport'
      WHEN 'theaters'   THEN 'entertainmentAndHobbies'
      WHEN 'cinema'     THEN 'entertainmentAndHobbies'
      WHEN 'medical'    THEN 'health'
      WHEN 'kids'       THEN 'health'
      WHEN 'laboratory' THEN 'health'
      ELSE 'general'
    END AS life_situation_theory
  FROM {TMP_DB}.tmp_tx_classified t
)
SELECT
  t.epk,
  t.sub_life_situation,
  COUNT(*) AS transaction_count,
  SUM(t.amt) AS total_amount,
  SUM(CASE WHEN t.life_situation_theory = 'general' THEN 1 ELSE 0 END) AS other_count,
  SUM(CASE WHEN t.life_situation_theory = 'general' THEN t.amt ELSE 0 END) AS other_amount
FROM tx_with_life t
INNER JOIN arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products_and_romashka_and_js_and b
  ON t.epk = b.epk AND t.life_situation_theory = b.lifestyle_situation
GROUP BY t.epk, t.sub_life_situation
""").write.mode("overwrite").saveAsTable(f"{TMP_DB}.tmp_filtered_agg")
 
print("Шаг 3: filtered_aggregated готов")
 
spark.sql(f"""
WITH sub_scores AS (
  SELECT
    epk,
    sub_life_situation,
    transaction_count,
    total_amount,
    other_count,
    other_amount,
    CASE WHEN other_count > 0
      THEN CAST(transaction_count AS DOUBLE) / other_count
      ELSE 0.0
    END AS transaction_count_share,
    CASE WHEN other_amount > 0
      THEN total_amount / other_amount
      ELSE 0.0
    END AS total_amount_share,
    (CASE WHEN other_amount > 0 THEN total_amount / other_amount ELSE 0.0 END) * 0.5 +
    (CASE WHEN other_count > 0 THEN CAST(transaction_count AS DOUBLE) / other_count ELSE 0.0 END) * 0.3 AS base_score
  FROM {TMP_DB}.tmp_filtered_agg
  WHERE sub_life_situation != 'general'
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY epk ORDER BY base_score DESC) AS rn,
    LEAD(base_score) OVER (PARTITION BY epk ORDER BY base_score DESC) AS second_score
  FROM sub_scores
)
SELECT
  epk,
  sub_life_situation,
  base_score,
  COALESCE(second_score, 0) AS second_score,
  CASE
    WHEN base_score > 0 THEN (base_score - COALESCE(second_score, 0)) * 100 / base_score
    ELSE 100.0
  END AS diff_percent
FROM ranked
WHERE rn = 1
""").write.mode("overwrite").saveAsTable(f"{TMP_DB}.tmp_top1")
 
print("Шаг 4: top1 готов")

Шаг 3: filtered_aggregated готов


Шаг 4: top1 готов


In [34]:
spark.sql(f"""
SELECT
  epk_id AS epk,
  COUNT(DISTINCT
    CASE
      WHEN nsi_id IN ('10245','10252','10253','11846','11845','11847','11844','11848','11843','11849',
                      '11842','11850','11841','10251','11840','11837','11839','11836','10250','11838',
                      '11835','11834','10249','41621') THEN 'family'
      WHEN nsi_id IN ('10228','22631') THEN 'auto'
      WHEN nsi_id IN ('10139','10967','22486','20127','20120','20092','20098','33538','15852','15854',
                      '15865','15861','15856','15853','15859','15851','15869','15857') THEN 'fitness'
      WHEN nsi_id IN ('20544','15880','10067','10095','10564','21929','21928','33936','21936','21930','21937','21934') THEN 'theaters'
      WHEN nsi_id IN ('10024','10026','12860','33093','10036','10035','33174','10034','15875','10033','10045','12854',
                      '10039','33095','33094','10052','10032','10025','10041','10046','10037','10038','10044','10050','10047',
                      '10040','10048','10042','10031','10062','12891','10051','15932','15878','10054','15874','10049','15876',
                      '12874','15882','29910','15879','32226','10056','32246','33106','15877','32216','12856','32243','33105',
                      '14457','15933','38379','33108','15884','33107','38380','40875','15934','14459') THEN 'cinema'
    END
  ) AS matching_tag_count,
  MAX(
    CASE
      WHEN nsi_id IN ('10245','10252','10253','11846','11845','11847','11844','11848','11843','11849',
                      '11842','11850','11841','10251','11840','11837','11839','11836','10250','11838',
                      '11835','11834','10249','41621') THEN 'family'
      WHEN nsi_id IN ('10228','22631') THEN 'auto'
      WHEN nsi_id IN ('10139','10967','22486','20127','20120','20092','20098','33538','15852','15854',
                      '15865','15861','15856','15853','15859','15851','15869','15857') THEN 'fitness'
      WHEN nsi_id IN ('20544','15880','10067','10095','10564','21929','21928','33936','21936','21930','21937','21934') THEN 'theaters'
      WHEN nsi_id IN ('10024','10026','12860','33093','10036','10035','33174','10034','15875','10033','10045','12854',
                      '10039','33095','33094','10052','10032','10025','10041','10046','10037','10038','10044','10050','10047',
                      '10040','10048','10042','10031','10062','12891','10051','15932','15878','10054','15874','10049','15876',
                      '12874','15882','29910','15879','32226','10056','32246','33106','15877','32216','12856','32243','33105',
                      '14457','15933','38379','33108','15884','33107','38380','40875','15934','14459') THEN 'cinema'
    END
  ) AS tag_sub_life
FROM {tags}
WHERE row_actual_to_dt = '9999-12-31'
  AND is_deleted = 0
  AND nsi_id IN ({ALL_NSI})
GROUP BY epk_id
""").write.mode("overwrite").saveAsTable(f"{TMP_DB}.tmp_client_tag_agg")
 
print("Шаг 5: client_tag_agg готов")
 
spark.sql(f"""
SELECT
  t.epk,
  t.sub_life_situation,
  t.base_score,
  LEAST(
    cta.matching_tag_count * 0.05,
    t.base_score * 0.3
  ) AS tag_bonus
FROM {TMP_DB}.tmp_top1 t
INNER JOIN {TMP_DB}.tmp_client_tag_agg cta
  ON t.epk = cta.epk
  AND t.diff_percent < 10
  AND cta.tag_sub_life = t.sub_life_situation
""").write.mode("overwrite").saveAsTable(f"{TMP_DB}.tmp_tag_bonus")
 
print("Шаг 6: tag_bonus готов")

Шаг 5: client_tag_agg готов


Шаг 6: tag_bonus готов


In [35]:
spark.conf.set("spark.sql.shuffle.partitions", "400")

In [36]:
spark.sql(f'''
select epk_id as epk, max(raw_score) as propensity_score
from need_id
inner join {romashka_hist} using(model_id)
where scoring_dt BETWEEN last_day(date('{dt0}') - interval 1 month) AND date('{dt0}')
    AND raw_score >= 70
    AND need_ID IN ('8.8.4','8.8.7','8.56.3','1.51.3')
    AND model_name LIKE '%d30'
group by epk_id
''').repartition(500).write.mode('overwrite').saveAsTable(f"{TMP_DB}.tmp_romashka_scores")

In [37]:
spark.sql(f'''
select b.epk, coalesce(rs.propensity_score, 0) as propensity_score
from arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products_and_romashka_and_js_and b
left join {TMP_DB}.tmp_romashka_scores rs on b.epk = rs.epk
group by b.epk, rs.propensity_score
''').write.mode('overwrite').saveAsTable(f"{TMP_DB}.tmp_romashka")

In [38]:
spark.sql(f"""
WITH final_score AS (
  SELECT
    t.epk,
    t.sub_life_situation,
    t.base_score,
    t.diff_percent,
    COALESCE(tb.tag_bonus, 0) AS tag_bonus,
    COALESCE(r.propensity_score, 0) AS propensity_score,
    t.base_score +
      CASE WHEN t.diff_percent < 10 THEN COALESCE(tb.tag_bonus, 0) ELSE 0 END +
      CASE WHEN t.diff_percent < 10 THEN COALESCE(r.propensity_score, 0) * 0.1 ELSE 0 END AS final_score
  FROM {TMP_DB}.tmp_top1 t
  LEFT JOIN {TMP_DB}.tmp_tag_bonus tb
    ON t.epk = tb.epk AND t.sub_life_situation = tb.sub_life_situation
  LEFT JOIN {TMP_DB}.tmp_romashka r ON t.epk = r.epk
),
ranked_final AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY epk ORDER BY final_score DESC) AS rn
  FROM final_score
),
ans AS (
  SELECT
    epk,
    COALESCE(sub_life_situation, 'general') AS lifestyle_sub_situation
  FROM ranked_final
  WHERE rn = 1
)
SELECT
  base.*,
  CASE
    WHEN base.lifestyle_situation IN ('auto','pets','cafesAndRestaurants','purchases') THEN ''
    ELSE COALESCE(ans.lifestyle_sub_situation, 'general')
  END AS lifestyle_sub_situation
FROM arnsdpsbx_team_ss.bpm_agent_py_levels_and_benefit_and_finance_lifetyle_products_and_romashka_and_js_and base
LEFT JOIN ans USING(epk)
""").write.mode("overwrite").saveAsTable(f"{TMP_DB}.bpm_agent_py_final")
 
print("Шаг 8: ФИНАЛ — bpm_agent_py_final_paid записан")

Шаг 8: ФИНАЛ — bpm_agent_py_final_paid записан


In [39]:
for t in ["tmp_tx_all", "tmp_tx_classified", "tmp_filtered_agg",
          "tmp_top1", "tmp_client_tag_agg", "tmp_tag_bonus", "tmp_romashka_scores", "tmp_romashka"]:
    spark.sql(f"DROP TABLE IF EXISTS {TMP_DB}.{t}")
 
print("Временные таблицы удалены")

Временные таблицы удалены


In [40]:
spark.sql(f'''
select count(epk), count(distinct epk) from {TMP_DB}.bpm_agent_py_final
''').show()

+----------+-------------------+
|count(epk)|count(DISTINCT epk)|
+----------+-------------------+
|  14098384|           14098384|
+----------+-------------------+



# Финальная проверка

In [45]:
spark.sql(f'''
select distinct * from {TMP_DB}.bpm_agent_py_final
where payment_status = 'FREE'
UNION ALL
select distinct * from {TMP_DB}.bpm_agent_py_final
where payment_status = 'PAID' 
    and epk not in (select epk from {TMP_DB}.bpm_agent_py_final where payment_status = 'FREE')
''').write.saveAsTable('arnsdpsbx_team_ss.bpm_final_py_all')

In [46]:
spark.sql(f'''
select count(epk), count(distinct epk) from arnsdpsbx_team_ss.bpm_agent_py_final
''').show()

+----------+-------------------+
|count(epk)|count(DISTINCT epk)|
+----------+-------------------+
|  14098384|           14098384|
+----------+-------------------+



In [47]:
spark.sql(f'''
select payment_status, count(epk), count(distinct epk) from arnsdpsbx_team_ss.bpm_agent_py_final
group by payment_status
''').show()

+--------------+----------+-------------------+
|payment_status|count(epk)|count(DISTINCT epk)|
+--------------+----------+-------------------+
|          FREE|   5580874|            5580874|
|          PAID|   8517510|            8517510|
+--------------+----------+-------------------+

